# Bagheria 2026 — thread Genere

Gap di genere fra i giovani a Bagheria, confrontato con Palermo, Sicilia e Italia.

**Prerequisiti**: `pipeline.fetch`, `pipeline.build` e `notebooks/analisi.ipynb` già eseguiti.
Le definizioni delle misure (tasso di occupazione, disoccupazione, fuori da lavoro e studio)
non vengono ridefinite qui: si riusano quelle del notebook condiviso, leggendo
`data/processed/analisi_condizione_15_24.csv`. Serve a garantire che i numeri di questo thread
e quelli degli altri due si sommino nella stessa proposal.

**Fasce d'età**: 15-24 sul lavoro, 9-24 sull'istruzione. Non sono scelte, sono le uniche
classi giovanili disponibili a livello comunale — la cella di verifica qui sotto lo mostra.

# **Caricamento**

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 30)

RADICE = Path.cwd()
if RADICE.name == "notebooks":
    RADICE = RADICE.parent
PROCESSED = RADICE / "data" / "processed"


def leggi(nome: str) -> pd.DataFrame:
    """Codici come stringhe: `condizione` contiene 1, 12, 99 e pandas li leggerebbe come interi."""
    tabella = pd.read_csv(PROCESSED / nome, dtype=str)
    for colonna in ("valore", "eta_anni", "popolazione", "occupati", "tasso_occupazione",
                    "tasso_disoccupazione", "fuori_da_lavoro_e_studio"):
        if colonna in tabella.columns:
            tabella[colonna] = pd.to_numeric(tabella[colonna])
    if "anno" in tabella.columns:
        tabella["anno"] = tabella["anno"].astype(int)
    return tabella


atteso = PROCESSED / "analisi_condizione_15_24.csv"
if not atteso.exists():
    raise SystemExit(f"manca {atteso.name}: esegui prima notebooks/analisi.ipynb")

istr_lav = leggi("censpop_istr_lav_long.csv")
ottomila = leggi("ottomilacensus_long.csv")
indicatori = leggi("indicatori.csv")
codici = leggi("codici.csv")
territori = leggi("territori.csv")
condizione = leggi("analisi_condizione_15_24.csv")
popolazione = leggi("censpop_popolazione_long.csv")

BAGHERIA, PALERMO, SICILIA, ITALIA = "082006", "082053", "ITG1", "IT"
CONFRONTO = [BAGHERIA, PALERMO, SICILIA, ITALIA]
NOMI = territori.set_index("territorio")["nome_territorio"].to_dict()
ORDINE = ["Bagheria", "Palermo", "Sicilia", "Italia"]

etichette = codici.set_index(["dimensione", "codice"])["etichetta"].to_dict()
print("misure disponibili dal notebook condiviso:", [c for c in condizione.columns if "tasso" in c or "fuori" in c])

misure disponibili dal notebook condiviso: ['tasso_occupazione', 'tasso_disoccupazione', 'fuori_da_lavoro_e_studio']


# **Verifica di fattibilità del thread**
Il piano iniziale del thread prevedeva la decomposizione del gap occupazionale per titolo di
studio. Prima di scrivere l'analisi, si controlla che l'incrocio esista.

In [2]:
lavoro = istr_lav[istr_lav["tavola"].eq("lavoro")]
istruzione = istr_lav[istr_lav["tavola"].eq("istruzione")]

print("Tavola LAVORO — titoli di studio presenti:", sorted(lavoro["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — titoli di studio presenti:", sorted(istruzione["titolo_studio"].unique()))
print("Tavola ISTRUZIONE — condizioni professionali presenti:", sorted(istruzione["condizione"].unique()))
print()
print("Classi d'età per tavola:")
print("  lavoro:    ", sorted(lavoro["eta"].unique()))
print("  istruzione:", sorted(istruzione["eta"].unique()))
print()
print("Anni con dato sulla classe 15-24 (lavoro):", sorted(lavoro[lavoro["eta"].eq("Y15-24")]["anno"].unique().tolist()))
print("Anni con dato sulla classe 9-24 (istruzione):", sorted(istruzione[istruzione["eta"].eq("Y9-24")]["anno"].unique().tolist()))

Tavola LAVORO — titoli di studio presenti: ['ALL']
Tavola ISTRUZIONE — titoli di studio presenti: ['ALL', 'BL', 'IL', 'LBNA', 'LSE', 'ML', 'ML_RDD', 'NED', 'PSE', 'RDD', 'USE_IF']
Tavola ISTRUZIONE — condizioni professionali presenti: ['99']

Classi d'età per tavola:
  lavoro:     ['Y15-24', 'Y25-49', 'Y50-64', 'Y_GE15', 'Y_GE65']
  istruzione: ['Y25-49', 'Y50-64', 'Y9-24', 'Y_GE65', 'Y_GE9']

Anni con dato sulla classe 15-24 (lavoro): [2018, 2019, 2021, 2022, 2023, 2024]
Anni con dato sulla classe 9-24 (istruzione): [2018, 2019, 2020, 2021, 2022, 2023, 2024]


**📌 Risultato chiave** — L'incrocio *titolo di studio × condizione professionale* non esiste a livello comunale: nella tavola lavoro il titolo è solo `ALL`, in quella istruzione la condizione è solo `99`. La decomposizione del gap per titolo di studio non è difficile, è impossibile con questi dati — il piano del thread cambia di conseguenza.

> **La decomposizione per titolo di studio non è possibile.** Nella tavola sul lavoro il titolo
> di studio esiste solo come `ALL`: il censimento permanente non pubblica a livello comunale
> l'incrocio condizione professionale × titolo di studio. Nella tavola sull'istruzione vale il
> simmetrico: i titoli di studio ci sono tutti, ma la condizione professionale è solo `99`, il totale.
>
> Le due tavole si toccano solo sui totali, quindi *"tra le diplomate, quante lavorano"* non è una
> domanda a cui questi dati rispondono. Non è aggirabile con un'aggregazione diversa.
>
> **Cosa si fa invece.** Si misurano i due gap separatamente — occupazione (15-24) e istruzione
> (9-24) — e si guarda se vanno nella stessa direzione. È una domanda diversa e più debole della
> decomposizione, ma è onesta: dice se le ragazze di Bagheria sono svantaggiate *anche* nello
> studio o *solo* nel lavoro.
>
> Nota sulle fasce: lavoro 15-24, istruzione 9-24. Non sono sovrapponibili e non vanno mai
> mostrate come se fossero la stessa popolazione.

# **Gap di genere sull'occupazione, 15-24**
Le tre misure definite nel notebook condiviso, lette per genere. Il 2020 manca alla fonte.

In [3]:
giovani = condizione[condizione["territorio"].isin(CONFRONTO)].copy()

# Serie di Bagheria: le tre misure per maschi e femmine.
bagheria = (giovani[giovani["territorio"].eq(BAGHERIA) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="anno", columns="genere",
                 values=["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]))
bagheria.round(1)

fuori_da_lavoro_e_studio       tasso_disoccupazione       tasso_occupazione      
genere                        F     M                    F     M                 F     M
anno                                                                                    
2018                       38.0  37.4                 79.0  61.7               4.7  11.5
2019                       32.7  32.8                 76.8  61.9               5.2  11.8
2021                       29.9  29.7                 59.2  47.0               6.2  13.8
2022                       27.8  26.8                 54.3  41.8               7.7  15.4
2023                       31.1  30.4                 56.2  45.2               8.0  15.2
2024                       26.7  27.1                 45.6  35.1               8.2  16.5

In [4]:
# Il gap in punti percentuali (maschi meno femmine) sulle tre misure, per i quattro territori.
def gap_di_genere(misura: str) -> pd.DataFrame:
    largo = (giovani[giovani["genere"].isin(["M", "F"])]
             .pivot_table(index=["nome_territorio", "anno"], columns="genere", values=misura))
    return (largo["M"] - largo["F"]).rename(misura)


gap = pd.concat([gap_di_genere(m) for m in
                 ["tasso_occupazione", "tasso_disoccupazione", "fuori_da_lavoro_e_studio"]], axis=1)
gap = gap.round(1).reset_index()
gap["nome_territorio"] = pd.Categorical(gap["nome_territorio"], ORDINE, ordered=True)
gap = gap.sort_values(["nome_territorio", "anno"])
gap.to_csv(PROCESSED / "genere_gap_occupazione.csv", index=False)

gap.pivot(index="anno", columns="nome_territorio", values="tasso_occupazione")

nome_territorio,Bagheria,Palermo,Sicilia,Italia
anno,,,,
2018,6.8,5.7,7.3,8.2
2019,6.6,5.2,7.4,8.5
2021,7.6,6.4,8.8,9.8
2022,7.7,6.7,9.3,9.7
2023,7.2,6.3,9.4,9.5
2024,8.3,6.9,9.9,9.6


**📌 Risultato chiave** — A Bagheria fra 2018 e 2024 il tasso di occupazione femminile 15-24 sale da 4.7% a 8.2% e quello maschile da 11.5% a 16.5%: entrambi crescono, ma il gap in punti non si chiude (6.9 → 8.3 pp).

> Un gap positivo sul tasso di occupazione significa che gli uomini lavorano di più. Sul
> `fuori_da_lavoro_e_studio` un gap positivo significa il contrario di quel che sembra: sono
> *gli uomini* a essere più spesso fuori da lavoro e studio. Le due misure vanno lette insieme,
> perché una fascia dove molti studiano ancora nasconde metà del fenomeno.

# **Quanto è preciso il gap? Intervalli di confidenza**
Prima di confrontare i gap fra territori serve l'ordine di grandezza dell'errore di ogni
numero: Bagheria è un comune, e i suoi conteggi sono piccoli. CI di Wilson sui tassi,
CI di Newcombe sulla differenza M-F.

Per gap occupazionale (o divario di genere nell'occupazione) si intende la differenza 
tra il tasso di occupazione maschile e quello femminile all'interno 
di una determinata popolazione o fascia d'età.

Avvertenza di metodo: occupati e popolazione, numeratore e denominatore del tasso, sono
conteggi interi in ogni anno (sezione «Le casalinghe sono un conteggio o una stima?»):
ogni residente risulta occupato o no. L'intervallo binomiale non è quindi l'errore di una
stima campionaria: misura quanto il tasso di una popolazione di quella taglia si
muoverebbe per caso a parità di condizioni. È la convenzione del notebook, e sulla
variazione di un anno regge il confronto con quella osservata fra i comuni siciliani di
taglia simile (deviazione 0.72 punti binomiale contro 0.71 osservata, sezione «Il KPI si
può misurare?»); sugli anni aggregati no, e lì si usa il metro osservato. L'errore di
classificazione del censimento, che ISTAT non pubblica, resta fuori dall'intervallo.

In [5]:
from statsmodels.stats.proportion import proportion_confint


def wilson(successi, totale):
    return proportion_confint(successi, totale, alpha=0.05, method="wilson")


def gap_con_ci(occ_m, pop_m, occ_f, pop_f):
    """Gap M-F fra proporzioni con CI 95% di Newcombe, costruito sui limiti di Wilson."""
    p_m, p_f = occ_m / pop_m, occ_f / pop_f
    l_m, u_m = wilson(occ_m, pop_m)
    l_f, u_f = wilson(occ_f, pop_f)
    g = p_m - p_f
    return (g,
            g - ((p_m - l_m) ** 2 + (u_f - p_f) ** 2) ** 0.5,
            g + ((u_m - p_m) ** 2 + (p_f - l_f) ** 2) ** 0.5)


# Conteggi non arrotondati dal notebook condiviso: per questo qualche decimale può
# differire dalla tabella dei gap sopra, che parte dai tassi già arrotondati.
conteggi_occ = (giovani[giovani["genere"].isin(["M", "F"])]
                .pivot_table(index=["territorio", "nome_territorio", "anno"],
                             columns="genere", values=["occupati", "popolazione"]))
conteggi_occ.columns = [f"{misura}_{gen}" for misura, gen in conteggi_occ.columns]

righe = []
for (territorio, nome, anno), r in conteggi_occ.iterrows():
    g, lo, hi = gap_con_ci(r["occupati_M"], r["popolazione_M"], r["occupati_F"], r["popolazione_F"])
    righe.append({"territorio": territorio, "nome_territorio": nome, "anno": anno,
                  "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
                  "tasso_M": 100 * r["occupati_M"] / r["popolazione_M"],
                  "gap": 100 * g, "gap_lo": 100 * lo, "gap_hi": 100 * hi})
ci_gap = pd.DataFrame(righe)
ci_gap["rapporto_M_F"] = ci_gap["tasso_M"] / ci_gap["tasso_F"]
ci_gap = ci_gap.round({"tasso_F": 1, "tasso_M": 1, "gap": 1, "gap_lo": 1, "gap_hi": 1, "rapporto_M_F": 2})
ci_gap.to_csv(PROCESSED / "genere_gap_occupazione_ci.csv", index=False)

print("Bagheria — gap occupazionale M-F (punti) con CI 95%:")
print(ci_gap[ci_gap["territorio"].eq(BAGHERIA)][["anno", "tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]
      .to_string(index=False))
# CI di Wilson anche sul LIVELLO femminile: il claim "tasso più basso del panel" non è
# testato dai CI sul gap; il test formale sta nella sezione LPM (effetti principali).
livelli_2024 = []
for (territorio, nome, anno), r in conteggi_occ.iterrows():
    if anno != 2024:
        continue
    basso, alto = wilson(r["occupati_F"], r["popolazione_F"])
    livelli_2024.append({"nome_territorio": nome,
                         "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
                         "CI 95% basso": 100 * basso, "CI 95% alto": 100 * alto})
print("\nTasso di occupazione femminile 2024 con CI 95% di Wilson:")
print(pd.DataFrame(livelli_2024).set_index("nome_territorio").reindex(ORDINE).round(1).to_string())

print("\nAnno 2024, i quattro territori:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]

Bagheria — gap occupazionale M-F (punti) con CI 95%:
 anno  tasso_F  tasso_M  gap  gap_lo  gap_hi
 2018      4.7     11.5  6.9     5.5     8.2
 2019      5.2     11.8  6.7     5.3     8.1
 2021      6.2     13.8  7.6     6.1     9.1
 2022      7.7     15.4  7.7     6.1     9.3
 2023      8.0     15.2  7.2     5.5     8.8
 2024      8.2     16.5  8.3     6.6    10.0

Tasso di occupazione femminile 2024 con CI 95% di Wilson:
                 tasso_F  CI 95% basso  CI 95% alto
nome_territorio                                    
Bagheria             8.2           7.2          9.2
Palermo              9.6           9.3          9.9
Sicilia             10.4          10.3         10.5
Italia              17.3          17.2         17.3

Anno 2024, i quattro territori:


,tasso_F,tasso_M,gap,gap_lo,gap_hi
nome_territorio,,,,,
Bagheria,8.2,16.5,8.3,6.6,10.0
Palermo,9.6,16.5,6.9,6.4,7.5
Sicilia,10.4,20.3,9.9,9.7,10.1
Italia,17.3,26.9,9.7,9.6,9.7


**📌 Risultato chiave** — Gap occupazionale 2024 di Bagheria: **8.3 pp** [CI 95% 6.6-10.0]. Il gap è solido — nessun CI tocca lo zero, in nessun anno e in nessun territorio — ma la precisione comunale è di ±1.35-1.7 pp: le oscillazioni annue (7.7 → 7.2 → 8.3) sono rumore, non trend.

> Il gap esiste ed è solido: in nessun anno e in nessun territorio il CI 95% tocca lo zero.
> Ma la precisione comunale è di ±1.35-1.7 punti: le oscillazioni anno su anno di Bagheria
> (7.7 → 7.2 → 8.3 fra 2022 e 2024) stanno tutte dentro gli intervalli e non vanno
> raccontate come peggioramenti o recuperi annuali. Il confronto sensato è fra territori
> e su più anni, ed è quello che fanno le sezioni successive.

# **Punti percentuali o rapporto? Entrambi**
Il gap in punti risponde a "quanti punti separano i tassi"; il rapporto M/F a "quante
volte è più probabile che un ragazzo lavori rispetto a una coetanea". Con tassi base
molto diversi fra territori le due scale possono ordinare i territori in modo opposto:
riportarne una sola sarebbe una scelta di comodo.

In [6]:
# Il gap in punti e il rapporto fra i tassi, fianco a fianco.
print("Rapporto M/F sul tasso di occupazione 15-24, serie:")
print(ci_gap.pivot(index="anno", columns="nome_territorio", values="rapporto_M_F")[ORDINE].to_string())
print("\nAnno 2024, le due scale:")
ci_gap[ci_gap["anno"].eq(2024)].set_index("nome_territorio").reindex(ORDINE)[
    ["tasso_F", "tasso_M", "gap", "rapporto_M_F"]]

Rapporto M/F sul tasso di occupazione 15-24, serie:
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2018                 2.47     1.90     1.97    1.58
2019                 2.29     1.75     1.93    1.59
2021                 2.23     1.84     2.04    1.65
2022                 2.00     1.77     2.00    1.59
2023                 1.89     1.68     1.94    1.56
2024                 2.01     1.72     1.95    1.56

Anno 2024, le due scale:


,tasso_F,tasso_M,gap,rapporto_M_F
nome_territorio,,,,
Bagheria,8.2,16.5,8.3,2.01
Palermo,9.6,16.5,6.9,1.72
Sicilia,10.4,20.3,9.9,1.95
Italia,17.3,26.9,9.7,1.56


**📌 Risultato chiave** — Le due scale ordinano i territori in modo diverso. **In punti** Bagheria (8.3) sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); **in rapporto M/F** (2.01) è la peggiore del panel (Palermo 1.72, Sicilia 1.95, Italia 1.56). Il tratto locale non è l'ampiezza del divario: è il **livello** del tasso femminile, 8.2%, il più basso dei quattro territori — CI 95% di Wilson [7.2-9.2], interamente sotto il 9.6 di Palermo; il test formale dei livelli sta nella sezione LPM (tutti i confronti p ≤ 0.009).

> Le due scale raccontano storie diverse, ed è il motivo per cui vanno dichiarate entrambe:
>
> - **in punti**, il gap 2024 di Bagheria (8.3) è sopra Palermo (6.9) ma sotto Sicilia (9.9)
>   e Italia (9.7);
> - **in rapporto**, Bagheria è la peggiore delle quattro: un ragazzo ha il doppio della
>   probabilità di lavorare di una coetanea (2.0, contro l'1.6 nazionale) — con una cautela: verso la Sicilia lo scarto
>   (2.01 contro 1.95) è dentro il rumore campionario e nel 2023 l'ordine era
>   invertito (1.89 contro 1.94); il primato nel rapporto è robusto verso Palermo
>   e Italia, quello nel livello verso tutti.
>
> Quando i tassi sono bassi per entrambi i generi, gli stessi punti percentuali pesano
> molto di più. Le scale divergono anche nel tempo: dal 2018 il gap in punti sale
> (6.9 → 8.3) mentre il rapporto scende (2.47 → 2.01), perché entrambi i tassi crescono.
> Ogni claim della proposal deve dire quale scala sta usando.

# **Il gap di Bagheria è un'anomalia locale? Modello lineare di probabilità**
La domanda di policy del thread — gap locale o regionale — testata formalmente invece che
a occhio. GLM binomiale con link identità sui conteggi aggregati: è il modello lineare di
probabilità, quindi i coefficienti si leggono direttamente in punti percentuali.
L'interazione genere × territorio stima la differenza fra il gap di Bagheria e quello di
ciascun benchmark; gli effetti principali del territorio (con F come riferimento) stimano
la differenza fra i **tassi femminili** — gap e livello escono dallo stesso modello. Confronti pre-specificati, per non pescare a strascico: anno di
riferimento 2024, e pooled 2022-2024 come robustezza.

Caveat sul pooled: tre annualità contano più volte le stesse persone, quindi i suoi CI
sono un po' ottimisti. Il pooled serve a stabilizzare il punto, non a moltiplicare i dati.

In [7]:
import warnings

import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import DomainWarning, PerfectSeparationWarning

# Il modello è saturo (una media per cella genere × territorio): il link identità fuori
# dal dominio [0,1] e la "separazione perfetta" sono attesi qui, non problemi di stima.
warnings.filterwarnings("ignore", category=DomainWarning)
warnings.filterwarnings("ignore", category=PerfectSeparationWarning)
warnings.filterwarnings("ignore", message="divide by zero", category=RuntimeWarning)

BENCHMARK = ["Palermo", "Sicilia", "Italia"]


def lpm_gap_e_livelli(anni: list, periodo: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """LPM sui conteggi aggregati: interazioni = eccesso di gap, effetti principali = livelli F.

    Con F e Bagheria come riferimenti, l'effetto principale del territorio è la differenza
    fra i tassi FEMMINILI (benchmark − Bagheria): è il test formale del claim sul livello.
    """
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"], as_index=False)[["occupati", "popolazione"]].sum())
    d["p"] = d["occupati"] / d["popolazione"]
    modello = smf.glm("p ~ C(nome_territorio, Treatment('Bagheria')) * C(genere, Treatment('F'))",
                      data=d, family=sm.families.Binomial(link=sm.families.links.Identity()),
                      var_weights=d["popolazione"]).fit()
    intervalli = modello.conf_int()
    gap, livelli = [], []
    for nome in BENCHMARK:
        principale = f"C(nome_territorio, Treatment('Bagheria'))[T.{nome}]"
        interazione = f"{principale}:C(genere, Treatment('F'))[T.M]"
        basso, alto = intervalli.loc[interazione]
        # L'interazione stima gap_benchmark - gap_Bagheria: segno invertito per leggere
        # l'eccesso locale (positivo = il gap di Bagheria è più largo).
        gap.append({"periodo": periodo, "confronto": f"vs {nome}",
                    "eccesso gap Bagheria (pp)": -100 * modello.params[interazione],
                    "CI 95% basso": -100 * alto, "CI 95% alto": -100 * basso,
                    "p": modello.pvalues[interazione]})
        basso, alto = intervalli.loc[principale]
        livelli.append({"periodo": periodo, "confronto": f"vs {nome}",
                        "livello F: benchmark − Bagheria (pp)": 100 * modello.params[principale],
                        "CI 95% basso": 100 * basso, "CI 95% alto": 100 * alto,
                        "p": modello.pvalues[principale]})
    return pd.DataFrame(gap), pd.DataFrame(livelli)


coppie = [lpm_gap_e_livelli([2024], "2024"),
          lpm_gap_e_livelli([2022, 2023, 2024], "pooled 2022-2024")]
eccessi = pd.concat([g for g, _ in coppie], ignore_index=True)
livelli_f = pd.concat([l for _, l in coppie], ignore_index=True)

print("Livello femminile: di quanto il tasso F di ciascun benchmark supera quello di Bagheria.")
print("Positivo e significativo su ogni confronto e periodo: il primato negativo del livello è testato.")
print(livelli_f.round({"livello F: benchmark − Bagheria (pp)": 2, "CI 95% basso": 2,
                       "CI 95% alto": 2, "p": 4}).to_string(index=False))
print("\nEccesso di gap (interazioni): il gap di Bagheria meno quello di ciascun benchmark.")
eccessi.round({"eccesso gap Bagheria (pp)": 1, "CI 95% basso": 1, "CI 95% alto": 1, "p": 3})

Livello femminile: di quanto il tasso F di ciascun benchmark supera quello di Bagheria.
Positivo e significativo su ogni confronto e periodo: il primato negativo del livello è testato.
         periodo  confronto  livello F: benchmark − Bagheria (pp)  CI 95% basso  CI 95% alto      p
            2024 vs Palermo                                  1.40          0.35         2.45 0.0090
            2024 vs Sicilia                                  2.22          1.21         3.23 0.0000
            2024  vs Italia                                  9.09          8.08        10.09 0.0000
pooled 2022-2024 vs Palermo                                  1.18          0.58         1.78 0.0001
pooled 2022-2024 vs Sicilia                                  1.93          1.35         2.50 0.0000
pooled 2022-2024  vs Italia                                  8.89          8.32         9.46 0.0000

Eccesso di gap (interazioni): il gap di Bagheria meno quello di ciascun benchmark.


,periodo,confronto,eccesso gap Bagheria (pp),CI 95% basso,CI 95% alto,p
0,2024,vs Palermo,1.3,-0.4,3.1,0.130
1,2024,vs Sicilia,-1.6,-3.2,0.1,0.066
2,2024,vs Italia,-1.4,-3.0,0.3,0.107
3,pooled 2022-2024,vs Palermo,1.1,0.1,2.1,0.031
4,pooled 2022-2024,vs Sicilia,-1.8,-2.7,-0.8,0.000
5,pooled 2022-2024,vs Italia,-1.9,-2.9,-1.0,0.000


**📌 Risultato chiave** — Pooled 2022-2024, il gap di Bagheria è **+1.1 pp più largo di Palermo** (p=0.03) e **1.8-1.9 pp più stretto di Sicilia e Italia** (p<0.001); sul solo 2024 nessuna differenza è significativa. In punti percentuali il gap di Bagheria **non è un'anomalia locale**. Ma il "vantaggio" sulla Sicilia nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile migliore. Gli **effetti principali dello stesso modello testano i livelli femminili**: Bagheria sotto Palermo di 1.2 pp [CI 0.6-1.8, p=0.0001], sotto la Sicilia di 1.9 [1.3-2.5] e sotto l'Italia di 8.9 [8.3-9.5] (pooled 2022-2024; anche sul solo 2024 tutti p ≤ 0.009). Il gap non distingue Bagheria; **il livello sì**.

> Sul solo 2024 nessuna differenza fra gap è significativa al 5%: un anno singolo di un
> comune non ha la precisione per distinguerli. Sul pooled 2022-2024 il quadro si separa:
>
> - il gap di Bagheria è **più largo di quello di Palermo** (+1.1 punti, p=0.03);
> - ed è **più stretto di quello di Sicilia e Italia** (-1.8 e -1.9 punti, p<0.001).
>
> Quindi, in punti percentuali, il gap di genere di Bagheria non è un'anomalia locale: sta
> dentro il paesaggio regionale. Ma il "vantaggio" verso la Sicilia non è una buona
> notizia: nasce dal tasso maschile più basso (16.5 contro 20.3), non da un tasso femminile
> migliore. Messo insieme alla sezione sulle scale, il tratto distintivo di Bagheria non è
> l'ampiezza del divario: è il **livello** — le ragazze hanno il tasso di occupazione più
> basso del panel (8.2%) e lo svantaggio relativo più alto (M/F = 2.0). È questo il
> bersaglio per la proposal, non la chiusura di un gap "anomalo" che i dati non mostrano. La tabella dei livelli
> qui sopra lo formalizza: tutti e tre i confronti sono positivi e significativi, in
> entrambi i periodi.

# **Il gap si sta allargando? Trend 2018-2024**
OLS sul gap in punti, anno centrato sul 2021, interazione col territorio per confrontare
le pendenze. Sei punti temporali per territorio e ogni punto è a sua volta una stima:
il modello dà direzione e ordine di grandezza, non inferenza fine.

In [8]:
serie_gap = ci_gap[["nome_territorio", "anno", "gap"]].assign(anno_c=lambda d: d["anno"] - 2021)
trend = smf.ols("gap ~ anno_c * C(nome_territorio, Treatment('Bagheria'))", data=serie_gap).fit()

pendenze = (pd.DataFrame({"pendenza (pp/anno)": trend.params, "p": trend.pvalues})
            .join(trend.conf_int().set_axis(["CI 95% basso", "CI 95% alto"], axis=1)))
pendenze = pendenze[pendenze.index.str.startswith("anno_c")]
pendenze.index = (pendenze.index
    .str.replace("anno_c:C(nome_territorio, Treatment('Bagheria'))[T.",
                 "differenza vs Bagheria: ", regex=False)
    .str.replace("]", "", regex=False)
    .str.replace("anno_c", "Bagheria", regex=False))
# La stessa pendenza sulla sola serie di Bagheria: il p del modello congiunto usa una varianza
# residua comune ai quattro territori, dove pesano le serie lisce di Sicilia e Italia.
solo_b = smf.ols("gap ~ anno_c", data=serie_gap[serie_gap["nome_territorio"].eq("Bagheria")]).fit()
ci_b = solo_b.conf_int().loc["anno_c"]
print(f"Solo Bagheria, sei annate: pendenza {solo_b.params['anno_c']:.3f} pp/anno, "
      f"CI 95% {ci_b[0]:.3f} / {ci_b[1]:.3f}, p = {solo_b.pvalues['anno_c']:.3f}")
pendenze.round(3)[["pendenza (pp/anno)", "CI 95% basso", "CI 95% alto", "p"]]

Solo Bagheria, sei annate: pendenza 0.205 pp/anno, CI 95% -0.001 / 0.411, p = 0.051


,pendenza (pp/anno),CI 95% basso,CI 95% alto,p
Bagheria,0.205,0.060,0.350,0.008
differenza vs Bagheria: Italia,0.058,-0.147,0.262,0.558
differenza vs Bagheria: Palermo,0.030,-0.174,0.235,0.757
differenza vs Bagheria: Sicilia,0.252,0.047,0.456,0.019


**📌 Risultato chiave** — La pendenza stimata del gap in punti di Bagheria è **+0.21 pp/anno**, ma l'allargamento **non è acquisito**. Il p=0.008 viene dal modello congiunto, che usa una varianza residua comune ai quattro territori; sulla sola serie di Bagheria (sei annate, stampata sopra la tabella) l'intervallo tocca lo zero e il p sale a circa 0.05, e ogni annata porta un proprio errore campionario (±1.35-1.7 punti, sezione sugli intervalli). È una direzione, non un risultato. Il passo è indistinguibile da Palermo e Italia; la Sicilia allarga più in fretta (+0.25 pp/anno in più). Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): le due scale divergono perché entrambi i tassi salgono.

> La pendenza di Bagheria è positiva, ~0.2 punti l'anno, ma con sei annate e questi
> errori campionari non basta a dire che il gap si stia allargando. Coerente con la
> sezione sulle scale: il divario in punti tende a salire mentre quello relativo scende,
> perché entrambi i tassi crescono e quello maschile cresce di più in valore assoluto.


# **Dentro il "fuori da lavoro e studio": la popolazione per stato**
Il tasso di disoccupazione ha per denominatore le sole forze di lavoro, che per le
ragazze di Bagheria sono qualche centinaio di persone: è la misura più fragile del
thread e da sola non regge conclusioni. Al suo posto: la popolazione 15-24 ripartita
in quattro stati esaustivi — occupati, in cerca, studenti, altri inattivi — tutti su
denominatore-popolazione. La partizione è verificata contro i totali prima dell'uso;
per costruzione "in cerca + altri inattivi" coincide con la misura condivisa
`fuori_da_lavoro_e_studio`.

In [9]:
lavoro_15_24 = istr_lav[
    istr_lav["tavola"].eq("lavoro")
    & istr_lav["territorio"].isin(CONFRONTO)
    & istr_lav["eta"].eq("Y15-24")
    & istr_lav["cittadinanza"].eq("TOTAL")
    & istr_lav["titolo_studio"].eq("ALL")
    & istr_lav["genere"].isin(["M", "F", "T"])]
largo = lavoro_15_24.pivot_table(index=["territorio", "anno", "genere"],
                                 columns="condizione", values="valore", aggfunc="sum")

# Verifica della partizione. Tolleranza e non uguaglianza: i conteggi comunali non sono
# interi, il censimento permanente è una stima registro + campione.
DETTAGLIO = ["1", "12", "5", "4", "24", "7"]  # occupato, in cerca, studente, casalinga/o, pensione, altro
assert (largo[DETTAGLIO].sum(axis=1) - largo["99"]).abs().max() < 0.001, "la partizione non ricostruisce il totale"
assert (largo[["1", "12"]].sum(axis=1) - largo["22"]).abs().max() < 0.001, "occupati + in cerca != forze di lavoro"
per_genere = largo["99"].unstack("genere")
assert (per_genere["M"] + per_genere["F"] - per_genere["T"]).abs().max() < 0.001, "M + F != T"

stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "altri inattivi": largo[["4", "24", "7"]].sum(axis=1),
})
quote = (100 * stati.div(largo["99"], axis=0)).reset_index()
quote["nome_territorio"] = quote["territorio"].map(NOMI)

lungo = (quote.melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
                    var_name="stato", value_name="quota")
         .assign(quota=lambda d: d["quota"].round(1)))
lungo.to_csv(PROCESSED / "genere_composizione_stato.csv", index=False)

forze_f = largo.loc[(BAGHERIA, slice(None), "F"), "22"].droplevel([0, 2]).round(0).astype(int)
print("Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):")
print(forze_f.to_dict())
print("\nComposizione 2024 (% della popolazione 15-24):")
(lungo[lungo["anno"].eq(2024) & lungo["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns=["stato", "genere"], values="quota")
    .reindex(ORDINE)[["occupati", "in cerca", "studenti", "altri inattivi"]])

Forze di lavoro femminili 15-24 a Bagheria (denominatore del tasso di disoccupazione):
{2018: 676, 2019: 663, 2021: 432, 2022: 479, 2023: 528, 2024: 434}

Composizione 2024 (% della popolazione 15-24):


stato           occupati       in cerca      studenti       altri inattivi      
genere                 F     M        F    M        F     M              F     M
nome_territorio                                                                 
Bagheria             8.2  16.5      6.9  8.9     65.1  56.4           19.9  18.2
Palermo              9.6  16.5      7.2  9.3     66.8  59.8           16.4  14.3
Sicilia             10.4  20.3      6.6  8.3     67.8  57.0           15.2  14.4
Italia              17.3  26.9      5.9  6.4     67.8  57.4            9.0   9.3

**📌 Risultato chiave** — Il minor tasso di occupazione femminile a 15-24 è assorbito dallo **studio**, non dall'inattività: studenti 65.1% F contro 56.4% M, in cerca 6.9% contro 8.9%, altri inattivi 19.9% contro 18.2%. E sotto il gap di genere ce n'è uno territoriale che colpisce **entrambi** i generi: "altri inattivi" al 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale.

> Tre letture:
>
> 1. Le ragazze **non** sono più spesso "fuori da tutto" dei ragazzi: altri inattivi
>    19.9% contro 18.2%, in cerca 6.9% contro 8.9%. Il tasso di occupazione più basso è
>    assorbito quasi per intero da più studio (65.1% contro 56.4%). A 15-24 il gap
>    occupazionale fotografa soprattutto ragazze ancora nel sistema formativo, non
>    inattività femminile.
> 2. Ciò che separa Bagheria, Palermo e la Sicilia dall'Italia è la quota di "altri
>    inattivi" per **entrambi** i generi: ~14-20% contro il ~9% nazionale.
> 3. Il denominatore del tasso di disoccupazione femminile di Bagheria è di 430-680
>    persone: i suoi sbalzi annuali sono in gran parte rumore, e fra 2019 e 2021 la misura
>    della condizione «in cerca» cambia (rottura di misura, `docs/sources.md` §7). Le conclusioni del thread
>    usano occupazione e composizione, non quel tasso.
>
> Il punto 1 sposta la domanda di policy in avanti: se a 15-24 le ragazze studiano di più
> e il vantaggio non si converte poi in occupazione, il nodo sta all'uscita dal percorso
> formativo — che i dati comunali non permettono di osservare oltre i 24 anni (la classe
> successiva, 25-49, sfora il target giovani e non è scomponibile).

### Riconciliazione con il thread educazione: stesso 19%, due scomposizioni
Il thread educazione (tavola `edu_youth_states_2018_2024.csv`, prodotta da
`pipeline/edu/`) pubblica gli stessi quattro stati sul totale
di genere: i suoi "inattivi non studenti" (19,0% nel 2024) sono l'aggregato T di
casalinghe + altra condizione + pensionati di questa sezione. Le due pipeline sono
indipendenti (fetch, pulizia e codice diversi), quindi la coincidenza è una verifica
incrociata, non un copia-incolla. Nella proposal il "19% invisibile" si racconta una
volta sola: livello e confronto territoriale dal thread educazione, composizione di
genere da qui.

In [10]:
edu_stati = leggi("edu_youth_states_2018_2024.csv")
edu_b24 = edu_stati[edu_stati["territorio"].eq(BAGHERIA) & edu_stati["anno"].eq(2024)].iloc[0]

lav_b24 = istr_lav[
    istr_lav["tavola"].eq("lavoro") & istr_lav["territorio"].eq(BAGHERIA)
    & istr_lav["anno"].eq(2024) & istr_lav["eta"].eq("Y15-24")
    & istr_lav["cittadinanza"].eq("TOTAL") & istr_lav["titolo_studio"].eq("ALL")]


def conteggio(genere, condizioni):
    sel = lav_b24[lav_b24["genere"].eq(genere) & lav_b24["condizione"].isin(condizioni)]
    return sel["valore"].sum()


INATTIVI = ["4", "7", "24"]  # casalinghe, altra condizione, pensionati/percettori
confronto_19 = pd.DataFrame([
    {"componente": nome,
     "F": conteggio("F", cond), "M": conteggio("M", cond),
     "F+M": conteggio("F", cond) + conteggio("M", cond),
     "T (edu)": float(edu_b24[col])}
    for nome, cond, col in [
        ("occupati", ["1"], "occupati"), ("in cerca", ["12"], "in_cerca"),
        ("studenti", ["5"], "studenti"),
        ("inattivi non studenti", INATTIVI, "inattivi_non_studenti"),
        ("fuori da lavoro e studio", ["12"] + INATTIVI, "fuori_lavoro_studio")]])
scarto_max = (confronto_19["F+M"] - confronto_19["T (edu)"]).abs().max()
assert scarto_max < 0.01, "le due pipeline non coincidono più: riallineare prima di citarle insieme"
print("🔗 Stesso oggetto al centesimo: il 19% del thread educazione è l'aggregato T di questa"
      f" sezione (scarto massimo {scarto_max:.4f} persone su ogni componente).")
confronto_19.round(2)

🔗 Stesso oggetto al centesimo: il 19% del thread educazione è l'aggregato T di questa sezione (scarto massimo 0.0000 persone su ogni componente).


,componente,F,M,F+M,T (edu)
0,occupati,236.00,498.00,734.00,734.00
1,in cerca,197.94,269.49,467.43,467.43
2,studenti,1875.30,1705.90,3581.20,3581.20
3,inattivi non studenti,572.77,548.61,1121.37,1121.37
4,fuori da lavoro e studio,770.70,818.10,1588.80,1588.80


# **Dentro gli "altri inattivi": casalinghe a 15-24 anni**
La decomposizione sopra mostra "altri inattivi" quasi uguali fra i generi (19.9% F contro
18.2% M nel 2024). È un'uguaglianza apparente: i codici che la compongono — casalinga/o,
percettore/rice di pensione, altra condizione — hanno distribuzioni di genere opposte,
e tenerli aggregati nasconde il meccanismo.

In [11]:
# Scissione degli "altri inattivi" nei tre codici che li compongono.
inattivi_dettaglio = pd.DataFrame({
    "casalinghe_o_i": largo["4"],
    "percettori_pensione": largo["24"],
    "altra_condizione": largo["7"],
})
quote_inattivi = (100 * inattivi_dettaglio.div(largo["99"], axis=0)).round(1)
casalinghe = pd.concat([inattivi_dettaglio["casalinghe_o_i"].rename("conteggio"),
                        quote_inattivi.add_suffix("_%")], axis=1).reset_index()
casalinghe["nome_territorio"] = casalinghe["territorio"].map(NOMI)
casalinghe.to_csv(PROCESSED / "genere_casalinghe.csv", index=False)

# Versione a sei stati della composizione, per la figura: qui "altri inattivi" è scisso,
# mentre genere_composizione_stato.csv resta a quattro stati per gli altri thread.
sei_stati = pd.DataFrame({
    "occupati": largo["1"], "in cerca": largo["12"], "studenti": largo["5"],
    "casalinghe/i": largo["4"], "altra condizione": largo["7"], "pensione": largo["24"],
})
dettaglio = (100 * sei_stati.div(largo["99"], axis=0)).round(1).reset_index()
dettaglio["nome_territorio"] = dettaglio["territorio"].map(NOMI)

# Accanto alla quota, le persone e il nodo di arrivo: li chiede il Sankey di fig02, dove il
# primo stadio è la scissione della popolazione fra i due generi. Con le sole quote i due
# rami sarebbero alti uguale (100% ciascuno) e il diagramma direbbe il falso.
# `destinazione` è la convenzione del proxy - «fuori da lavoro e istruzione», la cella più
# sotto - spezzata in tre invece che in due: chi cerca lavoro è fuori ma raggiungibile
# dalle politiche attive, chi non cerca no. Sta qui perché è una definizione, non un
# dettaglio di disegno: R legge e disegna, non ricalcola.
DESTINAZIONE = {"occupati": "dentro lavoro o studio", "studenti": "dentro lavoro o studio",
                "in cerca": "fuori ma in cerca",
                "casalinghe/i": "fuori e non in cerca",
                "altra condizione": "fuori e non in cerca",
                "pensione": "fuori e non in cerca"}
assert set(DESTINAZIONE) == set(sei_stati.columns), "stati senza nodo di arrivo"

CHIAVI = ["territorio", "anno", "genere", "stato"]
# I conteggi restano frazionari: il censimento permanente è una stima, e arrotondare
# ogni stato all'intero prima di sommarlo perde una persona per genere - i totali di fig02
# smetterebbero di combaciare con la platea di genere_platea.csv (fig09). R arrotonda solo
# quando scrive l'etichetta.
persone_stato = (sei_stati.round(1).reset_index()
                 .melt(id_vars=["territorio", "anno", "genere"],
                       var_name="stato", value_name="persone"))
composizione = (dettaglio
    .melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
          var_name="stato", value_name="quota")
    .merge(persone_stato, on=CHIAVI, validate="one_to_one"))
composizione["destinazione"] = composizione["stato"].map(DESTINAZIONE)
composizione.to_csv(PROCESSED / "genere_composizione_stato_dettaglio.csv", index=False)

serie_bag = (casalinghe[casalinghe["territorio"].eq(BAGHERIA) & casalinghe["genere"].eq("F")]
             .set_index("anno")[["conteggio", "casalinghe_o_i_%"]]
             .assign(conteggio=lambda d: d["conteggio"].round().astype(int)))
print("Bagheria — ragazze 15-24 che il censimento classifica come casalinghe:")
print(serie_bag.to_string())
print("\n2024, quote sulla popolazione 15-24 (%):")
(casalinghe[casalinghe["anno"].eq(2024) & casalinghe["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere",
                 values=["casalinghe_o_i_%", "altra_condizione_%"])
    .reindex(ORDINE)[["casalinghe_o_i_%", "altra_condizione_%"]])

Bagheria — ragazze 15-24 che il censimento classifica come casalinghe:
      conteggio  casalinghe_o_i_%
anno                             
2018        376              12.4
2019        324              10.9
2021        421              14.8
2022        364              12.8
2023        421              14.6
2024        387              13.4

2024, quote sulla popolazione 15-24 (%):


casalinghe_o_i_%      altra_condizione_%      
genere                         F    M                  F     M
nome_territorio                                               
Bagheria                    13.4  1.7                6.4  16.1
Palermo                     11.3  1.4                5.0  12.5
Sicilia                     10.1  1.2                5.1  12.7
Italia                       4.6  0.6                4.4   8.5

**📌 Risultato chiave**: nelle stime del censimento il **13.4% delle ragazze 15-24 di Bagheria risulta casalinga (387 persone)**, contro l'1.7% dei ragazzi e il 4.6% delle coetanee italiane: quasi il triplo dell'incidenza nazionale. Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2021-2024 fra 364 e 421 ragazze: è **strutturale**. Per i maschi gli "altri inattivi" sono invece quasi tutti "in altra condizione" (16.1%).

> L'uguaglianza era apparente: **il 13.4% delle ragazze 15-24 di Bagheria risulta
> casalinga (387 persone) contro l'1.7% dei ragazzi**; per i maschi gli "altri inattivi"
> sono quasi tutti "in altra condizione". E il fenomeno ha un gradiente territoriale
> netto: Italia 4.6%, Sicilia 10.1%, Palermo 11.3%, Bagheria 13.4%, il triplo
> dell'incidenza nazionale. La serie 2021-2024 dice che è strutturale, non episodico; il
> 2018-2019 viene da un altro metodo e non si mette in serie.
>
> È un'evidenza da proposal: un target definito (le ~390 ragazze casalinghe), un'etichetta
> da cui partire nel contatto e un KPI naturale (la quota casalinghe 15-24, da portare prima
> al livello di Palermo, poi verso quello nazionale). Cautela, verificata nella sezione «Le
> casalinghe sono un conteggio o una stima?»: non è una risposta delle ragazze né un
> conteggio, ma una stima di modello di ISTAT (somma di probabilità individuali, valori non
> interi, errore non pubblicato). Non misura il lavoro di cura: è un'etichetta, non una sua
> stima.

# **Chi sono le casalinghe? Un ragionamento di bounds per età**
La condizione professionale esiste solo sull'aggregato `Y15-24`; il registro demografico
dà però la popolazione femminile per **età singola**. Dal registro si prende solo la
*forma* della distribuzione per età e si delimita dove la quota di casalinghe può stare:
è un ragionamento di **bounds fra scenari estremi**, non una stima puntuale — l'età delle
387 non è osservata. Le due tavole condividono la stessa base demografica ufficiale (il
totale 15-24 coincide, come verifica la cella), quindi la ripartizione per età del
registro si applica direttamente alle quote della tavola lavoro.

In [12]:
# Struttura per età delle ragazze di Bagheria dal registro demografico (2024).
eta_f = (popolazione[popolazione["territorio"].eq(BAGHERIA)
                     & popolazione["anno"].eq(2024)
                     & popolazione["genere"].eq("F")
                     & popolazione["cittadinanza"].eq("TOTAL")
                     & popolazione["eta_anni"].notna()]
         .assign(eta=lambda d: d["eta_anni"].astype(int))
         .groupby("eta")["valore"].sum())
pop_15_24, pop_18_24, pop_20_24 = (eta_f.loc[15:24].sum(), eta_f.loc[18:24].sum(),
                                   eta_f.loc[20:24].sum())

# Denominatore della tavola lavoro: la sovrapponibilità col registro non si assume,
# si verifica — sotto l'unità le due basi coincidono e la ripartizione è trasferibile.
pop_f_lavoro = giovani.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["popolazione"].item()
assert abs(pop_15_24 - pop_f_lavoro) < 1, f"registro {pop_15_24:.0f} != tavola lavoro {pop_f_lavoro:.0f}"

riga = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")
n_cas, quota_cas = riga["conteggio"].item(), riga["casalinghe_o_i_%"].item()

scenari = pd.DataFrame([
    ("distribuzione uniforme sulle età 15-24", quota_cas),
    ("nessuna sotto i 18 anni (tutte 18-24)", quota_cas / (pop_18_24 / pop_15_24)),
    ("nessuna sotto i 20 anni (tutte 20-24)", quota_cas / (pop_20_24 / pop_15_24)),
], columns=["scenario", "quota nella fascia interessata (%)"]).round(1)
scenari.to_csv(PROCESSED / "genere_casalinghe_bounds.csv", index=False)

# Eccesso rispetto all'incidenza italiana, in persone (denominatore della tavola lavoro).
# Dai conteggi, non dalle quote arrotondate a un decimale: con quelle darebbe 254 invece di 255.
riga_ita = casalinghe.query("territorio == @ITALIA and genere == 'F' and anno == 2024")
cas_italia = riga_ita["casalinghe_o_i_%"].item()
pop_f_italia = giovani.query("territorio == @ITALIA and genere == 'F' and anno == 2024")["popolazione"].item()
eccesso = n_cas - riga_ita["conteggio"].item() / pop_f_italia * pop_f_lavoro

print(f"Registro 2024, femmine di Bagheria: 15-24 = {pop_15_24:.0f}, "
      f"di cui 18-24 = {pop_18_24:.0f} e 20-24 = {pop_20_24:.0f}")
print(f"Casalinghe 15-24 (tavola lavoro): {n_cas:.0f} = {quota_cas}% della fascia; "
      f"eccesso sull'incidenza italiana ({cas_italia}%): {eccesso:.0f} ragazze")
scenari

Registro 2024, femmine di Bagheria: 15-24 = 2882, di cui 18-24 = 2053 e 20-24 = 1498
Casalinghe 15-24 (tavola lavoro): 387 = 13.4% della fascia; eccesso sull'incidenza italiana (4.6%): 255 ragazze


,scenario,quota nella fascia interessata (%)
0,distribuzione uniforme sulle età 15-24,13.4
1,nessuna sotto i 18 anni (tutte 18-24),18.8
2,nessuna sotto i 20 anni (tutte 20-24),25.8


**📌 Risultato chiave** — Il 13.4% sull'aggregato 15-24 è compatibile con concentrazioni molto diverse: se nessuna casalinga avesse meno di 20 anni sarebbero il **25.8% delle 20-24enni — una su quattro**; se nessuna ne avesse meno di 18, il 18.8% delle 18-24enni. L'eccesso rispetto all'incidenza italiana vale **255 ragazze**. L'età resta non osservata: sono bounds, non stime.

> Lo scenario uniforme (13.4% a ogni età, comprese le quindicenni in obbligo scolastico) è
> implausibile; quello tutto-20-24 è l'estremo opposto. La realtà sta in mezzo, e già
> l'estremo inferiore plausibile — nessuna minorenne, 18.8% — descrive quasi una
> diciotto-ventiquattrenne su cinque. Con la ritenzione di coorte (le ragazze sono ancora
> qui fino ai 25 anni) il quadro è coerente: la condizione si forma *prima* dell'uscita
> dal comune, dentro la finestra in cui un servizio può ancora intercettarla.
> Se la concentrazione segua i matrimoni precoci lo dice lo stato civile per età, nella
> sezione «Le casalinghe sono coniugate?» qui sotto: non li segue.

# **Le casalinghe sono coniugate? Lo stato civile per età**

La quota di casalinghe dice *quante* ragazze stanno nel ruolo; non dice *da dove arriva*
il ruolo. Il candidato classico è il matrimonio precoce. A livello comunale il censimento
permanente non incrocia lo stato civile, ma la popolazione al 1° gennaio (`DCIS_POPRES1`,
base censuaria dal 2019) dà età singola × sesso × stato civile per tutti i comuni: fonte
diversa, dichiarata in ogni output, mai messa in serie con il censimento.

In [13]:
# Fonte diversa e dichiarata: DCIS_POPRES1 (popolazione al 1° gennaio, base censuaria dal
# 2019), perché il censimento permanente non incrocia lo stato civile a livello comunale
# (verificato in pipeline/fetch.py: la famiglia DEMCITMIG risponde NoRecordsFound).
# SETA_1 dell'anno t è lo stock al 31 dicembre, quindi coincide con POPRES al 1° gennaio
# dell'anno t+1. Il riferimento è il 1.1.2025, la fotografia di fine 2024, l'anno della tavola lavoro; il 1.1.2026 esiste
# ma pubblica solo il totale, senza dettaglio coniugale.
stato_civile = leggi("popres_stato_civile_long.csv")
ANNO_RIF = 2025

largo_sc = (stato_civile[stato_civile["eta_anni"].notna()]
            .pivot_table(index=["territorio", "anno", "genere", "eta_anni"],
                         columns="stato_civile", values="valore", aggfunc="sum"))

# Tre convenzioni della tavola, verificate e non assunte: (a) il dettaglio coniugale non è
# pubblicato dove è strutturalmente vuoto (coniugate sotto i 16 anni, unioni civili sotto
# i 18): NaN = 0; (b) sotto i 16 anni nubile == totale, esattamente; (c) con la (a) gli
# stati dettagliati ricostruiscono il totale 99 riga per riga, al centesimo.
DETTAGLIO_SC = ["1", "2", "3", "4", "15", "16", "17"]
dettaglio_sc = largo_sc[DETTAGLIO_SC].fillna(0)
con_dettaglio = largo_sc["1"].notna()          # tutto tranne il 1.1.2026, solo-totale
sotto_16 = largo_sc.index.get_level_values("eta_anni") < 16
assert (largo_sc.loc[sotto_16 & con_dettaglio, "1"]
        - largo_sc.loc[sotto_16 & con_dettaglio, "99"]).abs().max() < 0.001
assert (dettaglio_sc[con_dettaglio].sum(axis=1)
        - largo_sc.loc[con_dettaglio, "99"]).abs().max() < 0.001, \
    "gli stati dettagliati non ricostruiscono il totale"
anni_dettaglio = sorted(largo_sc[con_dettaglio].index.get_level_values("anno").unique())
assert ANNO_RIF in anni_dettaglio, f"manca il dettaglio al 1.1.{ANNO_RIF}"

# Sanità fra le tavole: le ragazze 15-24 al 1.1.2025 (POPRES) contro il 31.12.2024 (SETA_1).
# Sono lo stesso conteggio censuario: lo scarto atteso è zero.
f_15_24_popres = largo_sc.loc[(BAGHERIA, ANNO_RIF, "F"), "99"].loc[15:24].sum()
f_15_24_seta = popolazione[
    popolazione["territorio"].eq(BAGHERIA) & popolazione["anno"].eq(2024)
    & popolazione["genere"].eq("F") & popolazione["cittadinanza"].eq("TOTAL")
    & popolazione["eta_anni"].between(15, 24)]["valore"].sum()
print(f"Controllo fonti, ragazze 15-24: POPRES 1.1.{ANNO_RIF} = {f_15_24_popres:,.0f} contro "
      f"SETA_1 al 31.12.2024 = {f_15_24_seta:,.0f} ({100 * (f_15_24_popres / f_15_24_seta - 1):+.1f}%)")

# "Già coniugate": coniugate, divorziate, vedove, unioni civili incluse. Lo stato civile
# osserva il matrimonio formale, NON le convivenze né la maternità.
GIA_CONIUGATE = ["2", "3", "4", "15", "16", "17"]
gia_largo = dettaglio_sc[GIA_CONIUGATE].sum(axis=1)
righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        for anno in anni_dettaglio:
            for etichetta, (a0, a1) in {"15-24": (15, 24), "18-24": (18, 24),
                                        "20-24": (20, 24)}.items():
                gia = gia_largo.loc[(territorio, anno, genere)].loc[a0:a1].sum()
                tot = largo_sc.loc[(territorio, anno, genere), "99"].loc[a0:a1].sum()
                righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                              "anno": anno, "genere": genere, "fascia": etichetta,
                              "popolazione": round(tot), "gia_coniugate": round(gia),
                              "quota_gia_coniugate_pct": round(100 * gia / tot, 2)})
civile = pd.DataFrame(righe)
civile.to_csv(PROCESSED / "genere_stato_civile.csv", index=False)

print(f"\nQuota di ragazze già coniugate o unite civilmente al 1.1.{ANNO_RIF} (%):")
print(civile.query("genere == 'F' and anno == @ANNO_RIF")
      .pivot_table(index="nome_territorio", columns="fascia", values="quota_gia_coniugate_pct")
      .reindex(ORDINE)[["15-24", "18-24", "20-24"]].to_string())

# Il confronto che decide il meccanismo: anche se OGNI già-coniugata fosse casalinga,
# quante casalinghe restano nubili? Fonti diverse: è un bound, non un conto esatto.
gia_1524 = civile.query("territorio == @BAGHERIA and genere == 'F' "
                        "and anno == @ANNO_RIF and fascia == '15-24'")["gia_coniugate"].item()
cas_2024 = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["conteggio"].item()
print(f"\nBagheria: {cas_2024:.0f} casalinghe 15-24 (media 2024) contro {gia_1524} ragazze "
      f"già coniugate al 1.1.{ANNO_RIF}: anche nell'ipotesi estrema che ogni coniugata "
      f"sia casalinga, almeno {cas_2024 - gia_1524:.0f} casalinghe "
      f"({100 * (cas_2024 - gia_1524) / cas_2024:.0f}%) non sono sposate.")

serie_20_24 = civile.query("genere == 'F' and fascia == '20-24'")
print("\nSerie 20-24 (la fascia dove il fenomeno si concentra), quota già coniugate (%):")
print(serie_20_24.pivot_table(index="anno", columns="nome_territorio",
                              values="quota_gia_coniugate_pct")[ORDINE].to_string())

Controllo fonti, ragazze 15-24: POPRES 1.1.2025 = 2,882 contro SETA_1 al 31.12.2024 = 2,882 (+0.0%)

Quota di ragazze già coniugate o unite civilmente al 1.1.2025 (%):
fascia           15-24  18-24  20-24
nome_territorio                     
Bagheria          1.42   2.00   2.67
Palermo           1.61   2.30   3.21
Sicilia           1.52   2.15   2.90
Italia            1.16   1.65   2.25

Bagheria: 387 casalinghe 15-24 (media 2024) contro 41 ragazze già coniugate al 1.1.2025: anche nell'ipotesi estrema che ogni coniugata sia casalinga, almeno 346 casalinghe (89%) non sono sposate.

Serie 20-24 (la fascia dove il fenomeno si concentra), quota già coniugate (%):


nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2019                 6.07     6.22     5.70    4.18
2020                 4.71     5.52     5.05    3.76
2021                 3.56     4.44     4.13    3.23
2022                 3.47     3.85     3.77    2.79
2023                 3.05     3.43     3.40    2.65
2024                 2.54     3.51     3.16    2.43
2025                 2.67     3.21     2.90    2.25


**📌 Risultato chiave** — Il canale del matrimonio precoce **non regge i numeri**: al
1.1.2025 le ragazze 15-24 di Bagheria già coniugate (o in unione civile, o già uscite da
un matrimonio) sono **41, l'1.4%**, contro **387 casalinghe**: anche nell'ipotesi
estrema che ogni coniugata sia casalinga, **almeno l'89% delle casalinghe non è sposata**. E
Bagheria non si sposa presto nemmeno in senso relativo: quota di già coniugate 20-24 al
2.7%, *sotto* Palermo (3.2%) e Sicilia (2.9%), con il matrimonio under-25 in caduta
ovunque (a Bagheria dal 6.1% del 1.1.2019 al 2.7%). Le due tavole danno lo stesso
denominatore alla singola unità (2.882 ragazze): sono lo stesso conteggio censuario,
letto al 31.12.2024 e al 1.1.2025. Il ruolo di casalinga a vent'anni, qui, è nella quasi
totalità un ruolo **da non sposata**: il matrimonio precoce non lo spiega. Se passi dalla
famiglia d'origine, da una convivenza o da una maternità la fonte non lo dice: lo stato
civile non osserva convivenze né maternità; il conteggio dei nati per età della
madre (demo.istat) è il check successivo, ed è un fetch nuovo. Per la policy la
conseguenza è direzionale: il servizio per le casalinghe ventenni è un servizio di
**attivazione** (rientro in formazione e lavoro), non solo di conciliazione.

# **Gap di genere sull'istruzione, 9-24**
Composizione per titolo di studio e quota con almeno il diploma.

In [14]:
# Categorie mutuamente esclusive: la loro somma torna esattamente al totale ALL (verificato sotto).
ALMENO_DIPLOMA = ["USE_IF", "BL", "ML_RDD"]   # diploma, terziario 1° livello, terziario 2° e dottorato
TITOLI = ["NED", "PSE", "LSE"] + ALMENO_DIPLOMA

scuola = istruzione[
    istruzione["territorio"].isin(CONFRONTO)
    & istruzione["eta"].eq("Y9-24")
    & istruzione["cittadinanza"].eq("TOTAL")
    & istruzione["genere"].isin(["M", "F", "T"])]

conteggi = scuola.pivot_table(index=["territorio", "anno", "genere"],
                              columns="titolo_studio", values="valore", aggfunc="sum")

# Controllo di coerenza: le categorie di dettaglio devono ricostruire il totale.
scarto = (conteggi[TITOLI].sum(axis=1) - conteggi["ALL"]).abs().max()
assert scarto == 0, f"le categorie non ricostruiscono il totale, scarto massimo {scarto}"

titoli = pd.DataFrame({
    "popolazione_9_24": conteggi["ALL"],
    "almeno_diploma_%": 100 * conteggi[ALMENO_DIPLOMA].sum(axis=1) / conteggi["ALL"],
    "licenza_media_%": 100 * conteggi["LSE"] / conteggi["ALL"],
    "nessun_titolo_o_elementare_%": 100 * conteggi[["NED", "PSE"]].sum(axis=1) / conteggi["ALL"],
}).round(1).reset_index()
titoli["nome_territorio"] = titoli["territorio"].map(NOMI)
titoli.to_csv(PROCESSED / "genere_istruzione.csv", index=False)

ultimo_anno = titoli["anno"].max()
(titoli[titoli["anno"].eq(ultimo_anno) & titoli["genere"].isin(["M", "F"])]
    .pivot(index="nome_territorio", columns="genere", values="almeno_diploma_%")
    .reindex(ORDINE)
    .assign(**{"gap M-F (punti)": lambda d: (d["M"] - d["F"]).round(1)}))

genere,F,M,gap M-F (punti)
nome_territorio,,,
Bagheria,33.4,29.2,-4.2
Palermo,30.5,28.7,-1.8
Sicilia,33.1,30.4,-2.7
Italia,34.5,32.3,-2.2


**📌 Risultato chiave** — Gap istruzione 9-24 = **-4.2 pp**: a Bagheria le ragazze arrivano almeno al diploma più dei coetanei (33.4% contro 29.2%); sul 9-24 è il vantaggio femminile più ampio del panel (Palermo -1.8, Sicilia -2.7, Italia -2.2), sulle fasce allineate all'età del diploma no (sezione «Su 1.000 ragazze»).

> Il gap sull'istruzione è **negativo**: le ragazze arrivano al diploma più spesso dei coetanei,
> a Bagheria come altrove.
>
> Il livello assoluto va letto con cautela — la fascia parte da 9 anni e include ragazzi che il
> diploma non possono ancora averlo, quindi la percentuale è strutturalmente bassa. Il confronto
> fra generi regge solo in prima approssimazione: le ragazze 9-24 di Bagheria sono un po' più
> concentrate nelle età del diploma dei coetanei (sezione «Verifica di composizione per età»),
> e sulla stessa fascia 15-24 dell'occupazione il vantaggio è +4.8, alla pari con la Sicilia.

# **Verifica di composizione per età**
I tassi delle fasce 15-24 e 9-24 sono aggregati: se la struttura per età dentro la fascia
differisse fra generi o fra territori, parte dei gap sarebbe un artefatto di composizione
(chi ha 15 anni non lavora quasi mai, chi ne ha 12 non può avere un diploma). Le età
singole della demografia, disponibili dal 2021, permettono il controllo sul 2024.

In [15]:
singole_2024 = popolazione[
    popolazione["territorio"].isin(CONFRONTO)
    & popolazione["cittadinanza"].eq("TOTAL")
    & popolazione["genere"].isin(["M", "F"])
    & popolazione["eta_anni"].notna()
    & popolazione["anno"].eq(2024)]


def quota_fascia_alta(fascia, alta):
    """% della sottofascia più vecchia dentro la fascia, per territorio e genere (2024)."""
    dentro = singole_2024[singole_2024["eta_anni"].between(*fascia)]
    totale = dentro.groupby(["territorio", "genere"])["valore"].sum()
    parte = dentro[dentro["eta_anni"].between(*alta)].groupby(["territorio", "genere"])["valore"].sum()
    return (100 * parte / totale).unstack().rename(index=NOMI).reindex(ORDINE).round(1)


print("Fascia lavoro — quota dei 20-24 dentro i 15-24:")
print(quota_fascia_alta((15, 24), (20, 24)).to_string())
print("\nFascia istruzione — quota dei 19-24 dentro i 9-24:")
quota_fascia_alta((9, 24), (19, 24))

Fascia lavoro — quota dei 20-24 dentro i 15-24:
genere         F     M
territorio            
Bagheria    52.0  50.7
Palermo     49.2  49.6
Sicilia     50.8  50.6
Italia      50.2  50.7

Fascia istruzione — quota dei 19-24 dentro i 9-24:


genere,F,M
territorio,,
Bagheria,40.8,38.3
Palermo,38.5,38.7
Sicilia,39.5,40.1
Italia,38.8,39.7


**📌 Risultato chiave** — Il gap occupazionale **non** è un artefatto di composizione per età: le ragazze 15-24 di Bagheria sono semmai il gruppo più "vecchio" del panel (52.0% ha 20-24 anni), il che dovrebbe alzarne il tasso — il controllo è conservativo rispetto alla conclusione. Sul gap istruzione invece **~1.5 dei 4.2 punti sono mix per età**: il segno regge, la magnitudine di Bagheria va citata con questa cautela.

> - **Fascia lavoro 15-24**: la quota di 20-24enni sta fra il 49% e il 52% ovunque, per
>   entrambi i generi. Semmai le ragazze di Bagheria sono il gruppo leggermente più
>   "vecchio" (52.0%), il che dovrebbe *alzarne* il tasso di occupazione: il livello
>   femminile più basso del panel non è un artefatto di composizione, e il controllo è
>   conservativo rispetto alla conclusione.
> - **Fascia istruzione 9-24**: a Bagheria le femmine sono più concentrate nei 19-24 dei
>   maschi (40.8% contro 38.3%), mentre negli altri territori la differenza è sotto il
>   punto e spesso di segno opposto. Parte del gap di istruzione di Bagheria (-4.2) è
>   quindi composizione: assumendo che il diploma stia quasi tutto nei 19-24, l'ordine di
>   grandezza è 2.5 punti di mix × ~60 punti di differenza fra le sottofasce ≈ **1.5 punti
>   dei 4.2**. Il segno del vantaggio femminile regge (c'è anche dove il mix è pari), la
>   magnitudine di Bagheria va citata con questa cautela. La standardizzazione esatta non
>   è possibile: il titolo di studio per età singola non esiste a livello comunale.

# **I due gap a confronto**
La domanda del thread, nella forma che i dati permettono: le ragazze studiano di più e lavorano
di meno, e questo distingue Bagheria dai territori di riferimento?

In [16]:
anno_comune = min(giovani["anno"].max(), titoli["anno"].max())

occupazione_gap = (giovani[giovani["anno"].eq(anno_comune) & giovani["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="tasso_occupazione"))
istruzione_gap = (titoli[titoli["anno"].eq(anno_comune) & titoli["genere"].isin(["M", "F"])]
    .pivot_table(index="nome_territorio", columns="genere", values="almeno_diploma_%"))

quadro = pd.DataFrame({
    "occupazione M": occupazione_gap["M"],
    "occupazione F": occupazione_gap["F"],
    "gap occupazione (M-F)": occupazione_gap["M"] - occupazione_gap["F"],
    "almeno diploma M": istruzione_gap["M"],
    "almeno diploma F": istruzione_gap["F"],
    "gap istruzione (M-F)": istruzione_gap["M"] - istruzione_gap["F"],
}).reindex(ORDINE).round(1)
quadro.to_csv(PROCESSED / "genere_quadro_sintesi.csv")

print(f"Anno: {anno_comune}   —   occupazione 15-24, istruzione 9-24")
quadro

Anno: 2024   —   occupazione 15-24, istruzione 9-24


,occupazione M,occupazione F,gap occupazione (M-F),almeno diploma M,almeno diploma F,gap istruzione (M-F)
nome_territorio,,,,,,
Bagheria,16.5,8.2,8.3,29.2,33.4,-4.2
Palermo,16.5,9.6,6.9,28.7,30.5,-1.8
Sicilia,20.3,10.4,9.9,30.4,33.1,-2.7
Italia,26.9,17.3,9.6,32.3,34.5,-2.2


**📌 Risultato chiave** — Il paradosso, nella sua forma più compatta: **istruzione -4.2 pp (a favore delle ragazze), occupazione +8.3 pp (a loro sfavore)**, e i due segni sono opposti in tutti e quattro i territori. Le ragazze di Bagheria studiano più dei coetanei e lavorano la metà.

> È il risultato centrale del thread: **il gap di istruzione favorisce le ragazze, quello
> sull'occupazione le penalizza**, e i due segni sono opposti in tutti e quattro i territori.
>
> Per la proposal contano entrambi. Il confronto dice se il problema è locale: se il gap occupazionale di Bagheria è vicino
> a quello siciliano, il problema è regionale e un intervento comunale può poco. Se è più largo,
> c'è qualcosa di locale su cui agire.
>
> La risposta formale sta nella sezione LPM sopra: in punti percentuali il gap di Bagheria non è
> un'anomalia (sopra Palermo, sotto Sicilia e Italia). Lo specifico locale è il **livello**
> dell'occupazione femminile, il più basso del panel — ed è quello, insieme al conteggio della
> sezione "Il gap in persone" qui sotto, che la proposal deve bersagliare.

# **Il quadrante per genere: dove si rompe la catena**
Le stesse due misure del quadro, come otto punti M/F nel piano istruzione × occupazione:
il segmento da M a F di ogni territorio mostra direzione e ampiezza dello scarto di
genere. Le fasce restano quelle dei dati (istruzione 9-24, occupazione 15-24) e vanno
dichiarate su ogni asse. Qui si esportano anche le tabelle per le figure R
(`fig05_forbice`, `fig06_quadrante`): la "forbice" affianca il vantaggio educativo in punti,
il rapporto M/F e il livello femminile dell'occupazione. La fig05 usa il livello, l'unica delle
tre scale su cui Bagheria è ultima in ogni annata (sezione «I claim reggono al 2024?»): la scala
non si sceglie guardando dove Bagheria risulta peggiore.

In [17]:
quadrante = (giovani[giovani["anno"].eq(anno_comune) & giovani["genere"].isin(["M", "F"])]
    [["territorio", "nome_territorio", "genere", "tasso_occupazione"]]
    .merge(titoli[titoli["anno"].eq(anno_comune) & titoli["genere"].isin(["M", "F"])]
           [["territorio", "genere", "almeno_diploma_%"]], on=["territorio", "genere"])
    .assign(anno=anno_comune))
quadrante.to_csv(PROCESSED / "genere_quadrante.csv", index=False)

rapporti = ci_gap[ci_gap["anno"].eq(anno_comune)].set_index("nome_territorio")["rapporto_M_F"]
forbice = pd.DataFrame({
    "vantaggio_istruzione_F_pp": quadro["almeno diploma F"] - quadro["almeno diploma M"],
    "rapporto_M_F_occupazione": rapporti,
    "tasso_occupazione_F": quadro["occupazione F"],
    "tasso_occupazione_M": quadro["occupazione M"],
    "almeno_diploma_F": quadro["almeno diploma F"],
    "almeno_diploma_M": quadro["almeno diploma M"],
}).round(2).assign(anno=anno_comune)
forbice.index.name = "nome_territorio"
forbice.to_csv(PROCESSED / "genere_forbice.csv")

print(f"Anno {anno_comune} — istruzione 9-24 (almeno diploma), occupazione 15-24")
print(quadrante.pivot_table(index="nome_territorio", columns="genere",
                            values=["almeno_diploma_%", "tasso_occupazione"]).reindex(ORDINE).round(1).to_string())
forbice[["vantaggio_istruzione_F_pp", "rapporto_M_F_occupazione"]]

Anno 2024 — istruzione 9-24 (almeno diploma), occupazione 15-24
                almeno_diploma_%       tasso_occupazione      
genere                         F     M                 F     M
nome_territorio                                               
Bagheria                    33.4  29.2               8.2  16.5
Palermo                     30.5  28.7               9.6  16.5
Sicilia                     33.1  30.4              10.4  20.3
Italia                      34.5  32.3              17.3  26.9


,vantaggio_istruzione_F_pp,rapporto_M_F_occupazione
nome_territorio,,
Bagheria,4.2,2.01
Italia,2.2,1.56
Palermo,1.8,1.72
Sicilia,2.7,1.95


**📌 Risultato chiave** — Nel piano istruzione × occupazione tutti i segmenti M→F puntano nella stessa direzione — più istruite, meno occupate — e quello di Bagheria è il più orizzontale: il vantaggio educativo più ampio sul 9-24 (**+4.2 punti**) a fronte di uno scarto occupazionale di 8.3 punti, con l'estremo femminile più basso del piano (8.2%). La forbice fra le due classifiche è la forma compatta del paradosso, ed è la coppia esportata per `fig05`/`fig06`.

> Il primato educativo vale sul 9-24 e non regge sulle fasce allineate all'età del diploma
> (sezione «Su 1.000 ragazze»); il primato nel rapporto M/F vale nel 2024 ma non in ogni
> anno (sezione «I claim reggono al 2024?»). Regge il livello femminile, il più basso del panel. Le due tabelle esportate
> (`genere_quadrante.csv`, `genere_forbice.csv`) alimentano le figure senza ricalcoli in R.

# **Il pendolarismo ha un genere: le ragazze escono più dei coetanei per studiare, meno per lavorare**
Terza fonte sullo stesso punto di rottura, e l'unica che parla di mobilità — il terzo
focus del bando. La tavola del pendolarismo del censimento permanente (scaricata dal
thread educazione, `edu_census_commuting_long.csv`) ha la dimensione `genere`, finora
usata solo nei totali. La misura è la quota di chi **esce dal comune** sul totale di chi
si sposta giornalmente per quel motivo, separatamente per lavoro (`WK`) e studio (`STD`).

Il denominatore è già condizionato all'occupazione: chi si sposta *per lavoro* un lavoro
ce l'ha. La quota quindi **non** è un riflesso meccanico del gap occupazionale delle
sezioni precedenti — è una misura indipendente sullo stesso passaggio.

⚠️ Due limiti che vanno in caption e in proposal. La destinazione è `OMPUR`, «fuori
comune» aggregato: la tavola **non identifica Palermo**, quindi qui non si può scrivere
"pendolarismo verso Palermo" (la destinazione sta in `notebooks/mobilita.ipynb`). La tavola
non ha nemmeno l'età: le quote sono su tutti i residenti che si spostano. E la serie esiste solo per **2018 e 2019**: non si aggancia
al 2021-2024 del resto del thread e non è aggiornabile senza una rilevazione nuova.

In [18]:
# Quota di chi esce dal comune sul totale di chi si sposta giornalmente per quel motivo.
pend = leggi("edu_census_commuting_long.csv")
pend = pend[pend["territorio"].isin(CONFRONTO) & pend["genere"].isin(["M", "F"])
            & pend["motivo"].isin(["WK", "STD"]) & pend["destinazione"].isin(["ALL", "OMPUR"])]

quote = (pend.pivot_table(index=["territorio", "anno", "genere", "motivo"],
                          columns="destinazione", values="valore")
         .assign(quota_fuori_comune=lambda d: d["OMPUR"] / d["ALL"] * 100)
         .reset_index())
quote["nome_territorio"] = quote["territorio"].map(NOMI)

pendolarismo = (quote.pivot_table(index=["nome_territorio", "motivo", "anno"],
                                  columns="genere", values="quota_fuori_comune")
                .rename(columns={"M": "quota_M", "F": "quota_F"})
                .assign(gap_M_meno_F=lambda d: d["quota_M"] - d["quota_F"])
                .round(1).reset_index())
pendolarismo.to_csv(PROCESSED / "genere_pendolarismo.csv", index=False)

# Contesto 2011 (8milaCensus, 390 comuni): quanto si muove Bagheria in assoluto.
righe_mob = []
for ind in ["M2", "M4", "M6"]:
    comuni_m = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(2011)
                         & ottomila["indicatore"].eq(ind)]
                .set_index("territorio")["valore"].dropna())
    righe_mob.append({"indicatore": ind, "anno": 2011,
                      "bagheria": round(comuni_m[BAGHERIA], 1),
                      "mediana_390": round(comuni_m.median(), 1),
                      "percentile_390": round(100 * (comuni_m < comuni_m[BAGHERIA]).mean(), 1)})
mobilita_2011 = (pd.DataFrame(righe_mob)
                 .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore"))
mobilita_2011.to_csv(PROCESSED / "genere_mobilita_2011.csv", index=False)

for motivo, etichetta in [("WK", "lavoro"), ("STD", "studio")]:
    blocco = pendolarismo[pendolarismo["motivo"].eq(motivo) & pendolarismo["anno"].eq(2019)]
    print(f"{etichetta.upper()} — quota che esce dal comune (2019, % di chi si sposta per quel motivo)")
    print(blocco.set_index("nome_territorio").reindex(ORDINE)[
        ["quota_M", "quota_F", "gap_M_meno_F"]].to_string(), "\n")

print("Contesto 2011, 390 comuni siciliani:")
print(mobilita_2011.set_index("indicatore")[
    ["nome_indicatore", "bagheria", "mediana_390", "percentile_390"]].to_string())

pendolarismo.pivot_table(index="nome_territorio", columns=["motivo", "anno"],
                         values="gap_M_meno_F").reindex(ORDINE)

LAVORO — quota che esce dal comune (2019, % di chi si sposta per quel motivo)
genere           quota_M  quota_F  gap_M_meno_F
nome_territorio                                
Bagheria            41.2     33.0           8.2
Palermo              5.5      4.0           1.5
Sicilia             32.5     28.4           4.1
Italia              50.7     46.0           4.7 

STUDIO — quota che esce dal comune (2019, % di chi si sposta per quel motivo)
genere           quota_M  quota_F  gap_M_meno_F
nome_territorio                                
Bagheria            13.6     16.3          -2.7
Palermo              0.6      0.7          -0.1
Sicilia             19.7     21.5          -1.8
Italia              28.6     30.0          -1.4 

Contesto 2011, 390 comuni siciliani:
                                      nome_indicatore  bagheria  mediana_390  percentile_390
indicatore                                                                                  
M2          Mobilità fuori comune per stu

motivo           STD        WK     
anno            2018 2019 2018 2019
nome_territorio                    
Bagheria        -3.0 -2.7  8.3  8.2
Palermo         -0.2 -0.1  1.7  1.5
Sicilia         -1.7 -1.8  3.9  4.1
Italia          -1.5 -1.4  5.0  4.7

**📌 Risultato chiave** — Fra chi già si sposta per lavorare, esce dal comune il **41.2% degli uomini** di Bagheria contro il **33.0% delle donne** (2019): **8.2 punti**, il **doppio** dello scarto siciliano (4.1) e quasi il doppio di quello nazionale (4.7). Ma **per studiare il segno si inverte** — F 16.3% contro M 13.6% — ed è il vantaggio femminile più ampio del panel (+2.7 contro +1.8 Sicilia e +1.4 Italia). Stabile su entrambi gli anni disponibili.

> Le ragazze di Bagheria si muovono, e per studiare si muovono più dei coetanei e più
> che altrove. Smettono quando il motivo diventa il lavoro. È lo stesso punto di rottura
> del quadrante qui sopra — istruzione avanti, occupazione indietro — misurato su una
> tavola che non condivide né fonte né denominatore con le sezioni precedenti: per questo
> è una conferma e non una riformulazione.
>
> Contesto 2011 (390 comuni): `M2` al **25° percentile**, ma è un rapporto sull'intera
> popolazione fino a 64 anni, basso anche perché a Bagheria lavorano in pochi; a parità di
> taglia e distanza la quota di chi esce è nella media (`notebooks/mobilita.ipynb`, sezione 3). La mobilità studentesca `M4` al 18° **non** è di per sé un dato negativo:
> è un rapporto fuori/dentro comune, e Bagheria ha scuole proprie (3 sedi tecniche,
> cella MIUR più avanti).
>
> Per la proposal: la componente di mobilità non è un contorno, è una leva sullo stesso
> KPI occupazionale, e va calibrata sul motivo **lavoro** — sullo studio funziona già.

# **Il gap in persone**
Per il template della proposal (target e KPI) i tassi vanno tradotti in teste: quante
ragazze 15-24 occupate in più servirebbero a Bagheria sotto tre tassi-obiettivo. Calcolo
sui conteggi non arrotondati; la colonna "media 2022-2024" è la versione robusta alla
scelta dell'anno. Sono stock annui, non cumulati: "+40" significa 40 occupate in più
nell'anno di riferimento, a parità di popolazione.

In [19]:
def occupate_in_piu(anni: list) -> pd.Series:
    """Quante ragazze 15-24 occupate in più a Bagheria sotto ciascun tasso-obiettivo."""
    d = (giovani[giovani["genere"].isin(["M", "F"]) & giovani["anno"].isin(anni)]
         .groupby(["nome_territorio", "genere"])[["occupati", "popolazione"]].sum())
    p = d["occupati"] / d["popolazione"]
    pop_f = d.loc[("Bagheria", "F"), "popolazione"] / len(anni)  # popolazione media annua
    base = p[("Bagheria", "F")]
    return pd.Series({
        "parità con i coetanei maschi di Bagheria": int(round(pop_f * (p[("Bagheria", "M")] - base))),
        "tasso femminile di Palermo": int(round(pop_f * (p[("Palermo", "F")] - base))),
        "tasso femminile dell'Italia": int(round(pop_f * (p[("Italia", "F")] - base))),
    })


persone = pd.DataFrame({"occupate in più (2024)": occupate_in_piu([2024]),
                        "occupate in più (media 2022-2024)": occupate_in_piu([2022, 2023, 2024])})
persone.index.name = "scenario"
persone.to_csv(PROCESSED / "genere_gap_persone.csv")

base_2024 = giovani.query("nome_territorio == 'Bagheria' and genere == 'F' and anno == 2024")
# La base va anche su disco: fig09 traduce gli scenari in persone e non la ricalcola.
pd.DataFrame([{"anno": 2024,
               "popolazione_F_15_24": int(base_2024["popolazione"].item()),
               "occupate_F_15_24": int(base_2024["occupati"].item())}]).to_csv(
    PROCESSED / "genere_base_persone.csv", index=False)
print(f"Base 2024: {base_2024['popolazione'].item():.0f} ragazze 15-24, "
      f"{base_2024['occupati'].item():.0f} occupate "
      f"({base_2024['occupati'].item() / base_2024['popolazione'].item():.1%})")
persone

Base 2024: 2882 ragazze 15-24, 236 occupate (8.2%)


,occupate in più (2024),occupate in più (media 2022-2024)
scenario,,
parità con i coetanei maschi di Bagheria,239,221
tasso femminile di Palermo,40,34
tasso femminile dell'Italia,262,255


**📌 Risultato chiave** — Base 2024: **2.882 ragazze 15-24 a Bagheria, 236 occupate (8.2%)**. Allinearsi al tasso femminile di Palermo vale **+40 occupate** (l'obiettivo di convergenza, da scrivere in tasso: sezione «Il KPI netto»); la parità con i coetanei o il tasso nazionale valgono **+239 / +262**, cioè più che raddoppiare le occupate attuali — misura del problema, non obiettivo.

> La scala dell'intervento, in persone:
>
> - allinearsi al tasso femminile di **Palermo** vale **+40 occupate**: è l'obiettivo di
>   convergenza della proposta, che con la platea in calo va scritto in tasso e non in teste;
> - la **parità con i coetanei** o il **tasso femminile nazionale** valgono +239 e +262:
>   più che raddoppiare le 236 occupate attuali. È la misura del problema, non un KPI.
>
> Per il template della proposal: l'evidenza è questa tabella (più i CI della sezione
> sugli intervalli), il target sono le ~2.900 ragazze 15-24 di Bagheria, il KPI sono i
> punti di tasso chiusi verso il benchmark scelto — e ogni cifra si rigenera da questa
> cella a ogni aggiornamento dei dati.

### I canali operativi: gli istituti tecnici di Bagheria (anagrafe MIUR)
Le teste del KPI vanno raggiunte da qualche parte. Il thread educazione ha agganciato
l'anagrafe delle sedi del Ministero dell'Istruzione (SPARQL su
`dati.istruzione.it`, a.s. 2025/26; qui `edu_technical_schools.csv`): a Bagheria
risultano **tre sedi tecniche attive** — i canali fisici della presa in carico prima
dell'uscita da scuola, da incrociare con la platea per genere di questa sezione. È
un'anagrafica di sedi, non di esiti: dice dove intercettare, non quanti.

In [20]:
tecnici = leggi("edu_technical_schools.csv")
bagheria_tecnici = tecnici[tecnici["territorio"].eq(BAGHERIA)]
assert len(bagheria_tecnici) == 3, "anagrafe MIUR cambiata: aggiornare canali e conteggio nella proposal"
print(f"🔗 Sedi tecniche attive (a.s. {bagheria_tecnici['AnnoScolastico'].iloc[0]}): "
      f"{len(bagheria_tecnici)} a Bagheria, {int(tecnici['territorio'].eq(PALERMO).sum())} a Palermo.")
bagheria_tecnici[["DenominazioneScuola", "IndirizzoScuola",
                  "DescrizioneTipologiaGradoIstruzioneScuola"]]

🔗 Sedi tecniche attive (a.s. 202526): 3 a Bagheria, 23 a Palermo.


,DenominazioneScuola,IndirizzoScuola,DescrizioneTipologiaGradoIstruzioneScuola
21,IS. TEC. EC. E PER IL TUR. DON L. STURZO,Non Disponibile,ISTITUTO TECNICO COMMERCIALE
22,IST.T.COM. DON LUIGI STURZO SERALE,VIA S. IGNAZIO DI LOYOLA 7,ISTITUTO TECNICO COMMERCIALE
23,IST. TECNICO D'ACQUISTO,VIA CONSOLARE,ISTITUTO TECNICO INDUSTRIALE


# **Il KPI si può misurare? Potenza statistica e finestre di lettura**
"+40 occupate" equivale a +1.4 punti sul tasso femminile. Prima di scriverlo come KPI va
chiesto se una variazione simile sarebbe **distinguibile da come il dato si muove da solo**
nella fonte che dovrebbe misurarla. Due metri, calcolati uno accanto all'altro:

1. **binomiale**, la convenzione del notebook: i conteggi comunali trattati come un campione,
   test a due proporzioni indipendenti, α=0.05 bilaterale, potenza obiettivo 80%. Le finestre
   pooled moltiplicano la numerosità per il numero di anni: trattano come indipendenti
   annualità che contano le stesse persone;
2. **osservato**: quanto si muove, senza interventi noti, il tasso dei comuni siciliani di
   taglia simile a Bagheria (fra metà e il doppio delle sue ragazze 15-24 nel 2024, Bagheria
   esclusa), al netto della variazione mediana del gruppo. Viene dalla tavola lavoro 15-24
   dei 390 comuni, stessa fonte e stessa definizione del KPI.

Il metro osservato contiene anche le divergenze reali fra comuni: è il metro giusto per
chiedersi se un movimento di Bagheria sarebbe insolito, non una stima dell'errore statistico
della fonte. Non contiene la variabilità propria di Palermo, che il confronto dichiarato
aggiunge. Nessuno dei due metri è per costruzione più largo dell'altro: si calcolano entrambi.
La ritenzione di coorte è invece un conteggio di registro: lì il metro è la variabilità
osservata anno su anno, riportata in fondo.

In [21]:
import numpy as np
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

analisi_potenza = NormalIndPower()


def mde_pp(p0, n, potenza_obiettivo=0.80, alpha=0.05):
    """Variazione minima (pp) rilevabile all'80% confrontando due proporzioni su n persone."""
    h = analisi_potenza.solve_power(effect_size=None, nobs1=n, alpha=alpha,
                                    power=potenza_obiettivo, ratio=1.0, alternative="two-sided")
    phi0 = 2 * np.arcsin(np.sqrt(p0))
    return 100 * (np.sin((phi0 + h) / 2) ** 2 - p0)


def potenza_pct(p0, p1, n, alpha=0.05):
    h = abs(proportion_effectsize(p1, p0))
    return 100 * analisi_potenza.power(effect_size=h, nobs1=n, alpha=alpha,
                                       ratio=1.0, alternative="two-sided")


base = giovani.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")
n_f = base["popolazione"].item()
p_occ = base["occupati"].item() / n_f
p_occ_obiettivo = (giovani.query("territorio == @PALERMO and genere == 'F' and anno == 2024")
                   .pipe(lambda d: d["occupati"].item() / d["popolazione"].item()))
q_cas = casalinghe.query("territorio == @BAGHERIA and genere == 'F' and anno == 2024")["casalinghe_o_i_%"].item() / 100
q_cas_obiettivo = casalinghe.query("territorio == @PALERMO and genere == 'F' and anno == 2024")["casalinghe_o_i_%"].item() / 100

# --- Il metro osservato: i comuni siciliani di taglia simile (tavola lavoro 15-24 dei 390) ---
sic_15_24 = leggi("censpop_lavoro_15_24_sicilia_long.csv")
tassi_15_24 = (sic_15_24[sic_15_24["genere"].eq("F")
                         & sic_15_24["condizione"].isin(["1", "4", "12", "22", "99"])]
               .pivot_table(index=["territorio", "anno"], columns="condizione", values="valore")
               .rename(columns={"1": "occupate", "4": "casalinghe", "12": "in_cerca",
                                "22": "forze_lavoro", "99": "popolazione"}))
# La fonte non pubblica le celle vuote: in una ventina di comuni minuscoli manca la riga
# degli occupati. Lì le forze di lavoro coincidono con chi cerca, quindi le occupate sono
# zero, e zero si scrive invece di lasciare un buco che sparirebbe dai ranghi.
vuote = tassi_15_24["occupate"].isna()
assert ((tassi_15_24.loc[vuote, "forze_lavoro"] - tassi_15_24.loc[vuote, "in_cerca"]).abs() < 1e-6).all()
tassi_15_24["occupate"] = tassi_15_24["occupate"].fillna(0)
assert tassi_15_24[["occupate", "casalinghe", "popolazione"]].notna().all().all()
tassi_15_24["occupazione"] = 100 * tassi_15_24["occupate"] / tassi_15_24["popolazione"]
tassi_15_24["casalinghe_pct"] = 100 * tassi_15_24["casalinghe"] / tassi_15_24["popolazione"]
# Stesse celle della tavola a quattro territori (pipeline/build.py lo verifica sul raw).
assert abs(tassi_15_24.at[(BAGHERIA, 2024), "occupazione"] - 100 * p_occ) < 1e-9
assert abs(tassi_15_24.at[(BAGHERIA, 2024), "casalinghe_pct"] - 100 * q_cas) < 0.05

# Taglia simile = fra metà e il doppio delle ragazze 15-24 di Bagheria nel 2024. Bagheria
# resta fuori dal proprio gruppo di confronto; Palermo ne è fuori per taglia.
ragazze_2024 = tassi_15_24.xs(2024, level="anno")["popolazione"]
simili = ragazze_2024[ragazze_2024.between(n_f / 2, 2 * n_f)].index.drop(BAGHERIA)
assert PALERMO not in simili and len(simili) >= 30, len(simili)

# Ogni lato del confronto è la media di 1, 2 o 3 annualità. Senza il 2020 le coppie annue
# sono quattro; il triennio che i dati permettono (2018, 2019, 2021 contro 2022-2024)
# attraversa la rottura di misura del 2021, il biennio 2021-2022 contro 2023-2024 no.
FINESTRE = {1: [([2018], [2019]), ([2021], [2022]), ([2022], [2023]), ([2023], [2024])],
            2: [([2021, 2022], [2023, 2024])],
            3: [([2018, 2019, 2021], [2022, 2023, 2024])]}
ANNI_SERIE = [2018, 2019, 2021, 2022, 2023, 2024]
# L'oscillazione anno su anno si misura dopo la rottura: sulle casalinghe il 2019 -> 2021 è un
# gradino comune a tutti i comuni, non rumore, e gonfierebbe gli scarti dalla retta.
ANNI_POST = [2021, 2022, 2023, 2024]


def scarti_simili(colonna, finestra):
    """Variazione di ciascun comune simile, al netto della variazione mediana del gruppo (pp)."""
    serie = tassi_15_24[colonna].unstack("anno").loc[simili]
    pezzi = []
    for prima, dopo in FINESTRE[finestra]:
        variazione = serie[dopo].mean(axis=1) - serie[prima].mean(axis=1)
        pezzi.append(variazione - variazione.median())
    return pd.concat(pezzi, ignore_index=True)


def oscillazione_su_binomiale(conteggio):
    """Varianza dei residui attorno alla retta 2021-2024 di ogni comune simile, divisa per
    la varianza binomiale media della stessa serie: mediana sui comuni simili."""
    rapporti = []
    for territorio in simili:
        serie = tassi_15_24.loc[territorio].loc[ANNI_POST]
        p = (serie[conteggio] / serie["popolazione"]).to_numpy()
        residui = p - np.polyval(np.polyfit(ANNI_POST, p, 1), ANNI_POST)
        binomiale = np.mean(p * (1 - p) / serie["popolazione"].to_numpy())
        rapporti.append((residui ** 2).sum() / (len(ANNI_POST) - 2) / binomiale)
    return float(np.median(rapporti))


Z_ALFA = stats.norm.ppf(0.975)
Z_MDE = Z_ALFA + stats.norm.ppf(0.80)

righe = []
for kpi, colonna, conteggio, p0, p1 in [
        ("tasso di occupazione F 15-24 (obiettivo: Palermo)", "occupazione", "occupate", p_occ, p_occ_obiettivo),
        ("quota casalinghe F 15-24 (obiettivo: Palermo)", "casalinghe_pct", "casalinghe", q_cas, q_cas_obiettivo)]:
    delta = 100 * (p1 - p0)
    oscillazione = oscillazione_su_binomiale(conteggio)
    for finestra in (1, 2, 3):
        scarti = scarti_simili(colonna, finestra)
        sd = scarti.std()
        rapporto = abs(delta) / sd
        righe.append({
            "KPI": kpi, "attuale (%)": 100 * p0, "obiettivo (%)": 100 * p1,
            "delta da rilevare (pp)": delta, "anni pooled per lato": finestra,
            "MDE binomiale 80% (pp)": mde_pp(p0, finestra * n_f),
            "potenza binomiale (%)": potenza_pct(p0, p1, finestra * n_f),
            "deviazione binomiale (pp)": 100 * np.sqrt(2 * p0 * (1 - p0) / (finestra * n_f)),
            "comuni simili": len(simili), "scarti osservati": len(scarti),
            "deviazione osservata (pp)": sd,
            "MDE osservata 80% (pp)": Z_MDE * sd,
            "potenza osservata (%)": 100 * (stats.norm.sf(Z_ALFA - rapporto)
                                            + stats.norm.cdf(-Z_ALFA - rapporto)),
            "comuni simili mossi almeno del delta (%)": 100 * (scarti.abs() >= abs(delta)).mean(),
            "varianza annua attorno al trend 2021-2024 / binomiale": oscillazione})
mde = pd.DataFrame(righe).round({
    "attuale (%)": 1, "obiettivo (%)": 1, "delta da rilevare (pp)": 2,
    "MDE binomiale 80% (pp)": 2, "potenza binomiale (%)": 0, "deviazione binomiale (pp)": 2,
    "deviazione osservata (pp)": 2, "MDE osservata 80% (pp)": 2, "potenza osservata (%)": 0,
    "comuni simili mossi almeno del delta (%)": 0, "varianza annua attorno al trend 2021-2024 / binomiale": 2})
mde.to_csv(PROCESSED / "genere_mde.csv", index=False)

# Ritenzione (registro): variabilità annua osservata della coorte femminile 25-29.
eta_reg = (popolazione[popolazione["territorio"].eq(BAGHERIA)
                       & popolazione["genere"].eq("F")
                       & popolazione["cittadinanza"].eq("TOTAL")
                       & popolazione["eta_anni"].notna()]
           .assign(eta=lambda d: d["eta_anni"].astype(int))
           .groupby(["anno", "eta"])["valore"].sum())
rit_annua = {f"{a}->{a+1}": round(100 * eta_reg.loc[a + 1].loc[26:30].sum()
                                  / eta_reg.loc[a].loc[25:29].sum(), 1)
             for a in (2021, 2022, 2023)}
print("Ritenzione annua osservata, coorte F 25-29 di Bagheria (registro):", rit_annua)
print(f"Celle degli occupati non pubblicate e poste a zero: {int(vuote.sum())}")
print(f"Comuni di taglia simile: {len(simili)} (fra {n_f / 2:.0f} e {2 * n_f:.0f} ragazze 15-24 nel 2024)")
gradino = {colonna: (tassi_15_24[colonna].unstack("anno").loc[simili, 2021]
                     - tassi_15_24[colonna].unstack("anno").loc[simili, 2019]).median()
           for colonna in ("occupazione", "casalinghe_pct")}
print("Variazione mediana dei comuni simili fra 2019 e 2021, attraverso la rottura di misura (pp):",
      {k: round(v, 2) for k, v in gradino.items()})
mde[["KPI", "anni pooled per lato", "delta da rilevare (pp)", "deviazione binomiale (pp)",
     "deviazione osservata (pp)", "potenza binomiale (%)", "potenza osservata (%)",
     "MDE osservata 80% (pp)", "comuni simili mossi almeno del delta (%)",
     "varianza annua attorno al trend 2021-2024 / binomiale"]]

Ritenzione annua osservata, coorte F 25-29 di Bagheria (registro): {'2021->2022': np.float64(99.2), '2022->2023': np.float64(98.8), '2023->2024': np.float64(99.0)}
Celle degli occupati non pubblicate e poste a zero: 20
Comuni di taglia simile: 33 (fra 1441 e 5764 ragazze 15-24 nel 2024)
Variazione mediana dei comuni simili fra 2019 e 2021, attraverso la rottura di misura (pp): {'occupazione': np.float64(0.59), 'casalinghe_pct': np.float64(2.87)}


,KPI,anni pooled per lato,delta da rilevare (pp),deviazione binomiale (pp),deviazione osservata (pp),potenza binomiale (%),potenza osservata (%),MDE osservata 80% (pp),comuni simili mossi almeno del delta (%),varianza annua attorno al trend 2021-2024 / binomiale
0,tasso di occupazione F 15-24 (obiettivo: Palermo),1,1.4,0.72,0.71,46.0,50.0,2.00,5.0,0.36
1,tasso di occupazione F 15-24 (obiettivo: Palermo),2,1.4,0.51,0.64,75.0,59.0,1.80,3.0,0.36
2,tasso di occupazione F 15-24 (obiettivo: Palermo),3,1.4,0.42,0.80,90.0,41.0,2.25,9.0,0.36
3,quota casalinghe F 15-24 (obiettivo: Palermo),1,-2.1,0.90,0.98,68.0,57.0,2.76,3.0,1.41
4,quota casalinghe F 15-24 (obiettivo: Palermo),2,-2.1,0.63,0.73,93.0,82.0,2.05,0.0,1.41
5,quota casalinghe F 15-24 (obiettivo: Palermo),3,-2.1,0.52,0.97,99.0,58.0,2.72,3.0,1.41


**📌 Risultato chiave**: i due metri coincidono su un anno solo e si separano appena si
aggregano anni. Sull'occupazione femminile un comune simile si scosta dalla variazione mediana
del gruppo di 0.71 pp in un anno (binomiale: 0.72); con due e tre anni per lato resta a 0.64 e
0.80 pp, dove il binomiale scende a 0.51 e 0.42. Dopo il 2021 il tasso di un comune oscilla
attorno alla propria tendenza molto meno di un campione (0.36 volte la varianza binomiale,
mediana dei comuni simili), ma i comuni divergono fra loro in modo persistente: quella
divergenza non si media via aggregando anni.

Per il KPI "+40 occupate" (+1.40 pp):
* potenza osservata **50% su un anno, 59% sul biennio, 41% sul triennio**, contro 46%, 75% e
  90% del binomiale. Il triennio che i dati permettono attraversa la rottura del 2021; il
  biennio 2021-2022 contro 2023-2024 è la finestra pulita;
* nessuna finestra arriva all'80%: il tasso comunale dice la **direzione** della convergenza,
  non prova un effetto;
* senza interventi solo il 3-9% dei comuni simili si scosta dal gruppo di 1.4 punti: un
  movimento del genere non sarebbe banale, ma la fonte non lo distingue con la sicurezza
  convenzionale.

Per le casalinghe (-2.1 pp): 57% su un anno, **82% sul biennio**, 58% sul triennio (binomiale:
68%, 93%, 99%). È l'unico dei due KPI che su una finestra pulita supera l'80%. La quota però
oscilla più di un campione (1.41 volte la varianza binomiale dopo il 2021) e salta alla
rottura di misura (mediana dei comuni simili +2.87 punti fra 2019 e 2021): si legge solo
dentro la definizione 2021+.

> Conseguenze per la proposal:
>
> 1. le finestre di lettura restano dichiarate prima (triennio per il tasso, biennio per le
>    casalinghe), ma le potenze binomiali del 90% e del 99% escono: i KPI di popolazione si
>    leggono come direzione della convergenza, l'effetto del servizio si misura sui
>    partecipanti (sezione «Che effetto può vedere il pilota?»);
> 2. serve almeno un **indicatore di processo** a lettura annuale (utenza dei servizi per
>    età e genere, ingressi nei percorsi), che oggi nessuno rileva: il che salda questa
>    sezione alla proposta del presidio di misurazione;
> 3. la ritenzione di coorte si monitora contro la variabilità osservata: le tre letture
>    annue della coorte F 25-29 (99.2, 98.8, 99.0) stanno in mezzo punto; il segnale,
>    una perdita netta di ~1% l'anno, è stabile; ogni scarto va letto contro quel mezzo
>    punto, non contro soglie da manuale.

# **Chi se ne va? Ritenzione di coorte per genere, 2021-2024**
La fuga di talenti del bando, guardata per genere. Dalle età singole si segue ogni
coorte: chi aveva `a` anni nel 2021 ne ha `a+3` nel 2024, e il rapporto fra i due stock
è la ritenzione netta. A queste età la mortalità è trascurabile, quindi lo scarto da 100
è migrazione netta — impastata però con l'aggiustamento post-censuario delle stime, che
non è separabile: si leggono i pattern contro il benchmark nazionale, non i decimali.
"Netta" significa che conta anche chi arriva, non solo chi parte.

In [22]:
singole = (popolazione[
        popolazione["territorio"].isin(CONFRONTO)
        & popolazione["cittadinanza"].eq("TOTAL")
        & popolazione["genere"].isin(["M", "F"])
        & popolazione["eta_anni"].notna()]
    .assign(eta=lambda d: d["eta_anni"].astype(int)))

p21 = singole[singole["anno"].eq(2021)]
p24 = singole[singole["anno"].eq(2024)]

COORTI = {"15-19 nel 2021": (15, 19), "20-24 nel 2021": (20, 24), "25-29 nel 2021": (25, 29)}
righe = []
for etichetta, (a0, a1) in COORTI.items():
    base = p21[p21["eta"].between(a0, a1)].groupby(["territorio", "genere"])["valore"].sum()
    dopo = p24[p24["eta"].between(a0 + 3, a1 + 3)].groupby(["territorio", "genere"])["valore"].sum()
    parziale = (100 * dopo / base).rename("ritenzione_%").reset_index()
    parziale["coorte"] = etichetta
    righe.append(parziale)
coorti = pd.concat(righe, ignore_index=True)
coorti["nome_territorio"] = coorti["territorio"].map(NOMI)
coorti["ritenzione_%"] = coorti["ritenzione_%"].round(1)
coorti.to_csv(PROCESSED / "genere_coorti.csv", index=False)

(coorti.pivot_table(index="coorte", columns=["nome_territorio", "genere"], values="ritenzione_%")
    .reindex(columns=pd.MultiIndex.from_product([ORDINE, ["F", "M"]])))

Bagheria        Palermo        Sicilia        Italia       
                      F      M       F      M       F      M      F      M
coorte                                                                    
15-19 nel 2021    100.4   98.5   100.8  101.1   100.7  103.0  101.8  104.8
20-24 nel 2021    102.0   99.7    99.8   97.7    99.2   98.3  102.7  104.1
25-29 nel 2021     96.3  101.2    98.8   96.9    97.6   97.6  103.0  103.8

**📌 Risultato chiave** — Ritenzione di coorte 2021-2024: Bagheria sta **sotto l'Italia in ogni coorte e per entrambi i generi**. La firma di genere sta nel *quando*: i ragazzi si perdono presto (15-19: 98.5 contro 104.8), le ragazze **dopo i 25** (25-29: 96.3 contro 103.0, la cella peggiore del comune) — proprio quando il vantaggio formativo dovrebbe convertirsi in lavoro. Cautela: è la lettura di un solo triennio. Sulle coorti seguite per cinque anni la perdita all'uscita dal percorso formativo è di entrambi i generi (sulla transizione 20-24 → 25-29 i maschi perdono di più) e le ragazze di Bagheria stanno al livello di Palermo e della Sicilia (sezione «I claim reggono al 2024?», verifica 2).

> Due pattern, letti contro il benchmark nazionale (che sta sopra 100 ovunque grazie
> all'immigrazione):
>
> 1. **Bagheria non raggiunge il livello nazionale in nessuna cella**: ogni coorte, di
>    entrambi i generi, trattiene meno giovani di quanto faccia l'Italia. Il drenaggio
>    riguarda tutti.
> 2. La firma di genere sta nel **quando**: i ragazzi si perdono presto (coorte 15-19
>    nel 2021: 98.5 contro il 104.8 nazionale), le ragazze **dopo i 25** — la coorte
>    25-29 femminile è la peggiore di Bagheria (96.3 contro 103.0, quasi 7 punti sotto).
>    È l'età in cui il percorso formativo finisce: coerente con la decomposizione per
>    stato, le ragazze restano finché studiano e il territorio ne perde una quota
>    proprio quando il vantaggio educativo dovrebbe convertirsi in lavoro.
>
> La misura è netta e su una finestra di tre anni: non distingue chi parte da chi
> arriva, né dice dove vanno. Per i flussi origine-destinazione servono altre fonti —
> il thread mobilità è il posto naturale dove cercarle. La finestra di tre anni è anche
> l'unica in cui la differenza di genere compare: le coorti seguite per cinque anni non la
> ripetono (sezione «I claim reggono al 2024?»).

# **La ritenzione per età: quando esattamente, e chi resta**
Le coorti quinquennali dicono "prima i ragazzi, dopo i 25 le ragazze"; le età singole
permettono di vedere *dove* la curva si piega. Conteggi comunali per età di ~250-340
persone: il profilo usa una media mobile su tre età (somme di conteggi, poi rapporto) e
si leggono i pattern, non i decimali. In coda, il conto del gruppo che resta ma è fuori
da lavoro, studio e ricerca — il cosiddetto «19% invisibile» — scisso
per genere.

In [23]:
r21 = singole[singole["anno"].eq(2021)].groupby(["territorio", "genere", "eta"])["valore"].sum()
r24 = singole[singole["anno"].eq(2024)].groupby(["territorio", "genere", "eta"])["valore"].sum()

righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        for a in range(13, 32):
            n0, n3 = r21.get((territorio, genere, a)), r24.get((territorio, genere, a + 3))
            righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                          "genere": genere, "eta_2021": a, "eta_2024": a + 3,
                          "n_2021": n0, "n_2024": n3,
                          "ritenzione_pct": round(100 * n3 / n0, 1)})
profilo = pd.DataFrame(righe)
for _, gruppo in profilo.groupby(["territorio", "genere"]):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    profilo.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)
profilo.to_csv(PROCESSED / "genere_ritenzione_eta.csv", index=False)

print("Ritenzione 2021→2024 per età (media mobile 3 età), Bagheria e Italia:")
print(profilo[profilo["territorio"].isin([BAGHERIA, ITALIA])]
      .pivot_table(index="eta_2021", columns=["nome_territorio", "genere"],
                   values="ritenzione_rolling3_pct")
      .loc[14:30, [("Bagheria", "F"), ("Bagheria", "M"), ("Italia", "F"), ("Italia", "M")]]
      .to_string())

# Il gruppo fuori da lavoro, studio e ricerca (2024), per genere: dimensioni e composizione.
fuori = largo.loc[(BAGHERIA, 2024)]
print("\nBagheria 2024, 15-24 — fuori da lavoro, studio e ricerca attiva:")
for genere in ("F", "M"):
    r = fuori.loc[genere]
    tot = r[["4", "24", "7"]].sum()
    print(f"  {genere}: {tot:.0f} persone = {100 * tot / r['99']:.1f}% della fascia "
          f"(casalinghe/i {r['4']:.0f}, altra condizione {r['7']:.0f}, pensione {r['24']:.0f}; "
          f"in cerca, per confronto: {r['12']:.0f})")
tot_f = fuori.loc["F", ["4", "24", "7"]].sum()
tot_m = fuori.loc["M", ["4", "24", "7"]].sum()
print(f"  quota femminile del gruppo: {100 * tot_f / (tot_f + tot_m):.1f}%")

Ritenzione 2021→2024 per età (media mobile 3 età), Bagheria e Italia:
nome_territorio Bagheria        Italia       
genere                 F      M      F      M
eta_2021                                     
14                  99.8   99.4  101.2  102.8
15                 100.6   98.8  101.2  103.8
16                  99.5   99.2  101.4  104.7
17                 100.5   99.4  101.8  105.2
18                 100.3   98.3  102.1  105.2
19                 101.0   97.8  102.4  104.6
20                 103.3   99.2  102.4  104.3
21                 101.5  100.7  102.6  104.2
22                 103.3  100.4  102.8  104.2
23                 100.5   98.7  102.9  104.1
24                  99.6   97.5  102.9  104.0
25                  97.1   97.6  102.9  103.9
26                  96.9  100.0  103.0  103.9
27                  97.6  102.7  103.0  103.9
28                  96.7  103.7  103.0  103.8
29                  97.9  101.1  102.9  103.6
30                  98.4  100.0  102.8  103.4

Bagheria 

**📌 Risultato chiave** — Il profilo per età localizza le due rotture: le ragazze tengono (≈100-103) **fino ai 23 anni del 2021** e cedono da lì in poi — 96.7-97.9 sulle età 25-29 contro ~103 dell'Italia; i ragazzi stanno sotto la pari già a 17-19 e di nuovo a 23-24, ma **recuperano dopo i 26** (rientri netti alle età 27-28), le ragazze no. E il «19% invisibile» **non è un gruppo femminile**: 573 ragazze e 549 ragazzi (51% F). Femminile è l'etichetta — 387 casalinghe contro 50 — non la dimensione.

> Tre precisazioni:
>
> 1. la finestra utile per le ragazze è **22-25 anni**: prima la curva è sopra la pari,
>    dopo la perdita è già avvenuta. "Dopo i 25" era giusto ma generico — il cedimento
>    parte fra le età 24 e 25 (del 2021) e riguarda quindi le 27-32enni del 2024. È la
>    lettura del triennio 2021-2024: nel vicinato il cedimento femminile non c'è (sezione
>    «I comuni vicini») e sulle coorti seguite per cinque anni la perdita a queste età è di
>    entrambi i generi (sezione «I claim reggono al 2024?»);
> 2. per i ragazzi il profilo è **a due onde con rientro**: uscita precoce (17-19),
>    seconda uscita a 23-24, saldo positivo dopo i 26 — pendolarismo lungo di ritorno o
>    rientri, i dati non distinguono;
> 3. il gruppo fuori-da-tutto è di **entrambi i generi in parti quasi uguali**; ciò che
>    è di genere è la *composizione*: le ragazze hanno un ruolo attribuito dal censimento (casalinga),
>    i ragazzi un residuo ("altra condizione"). Un intervento "per le invisibili" che ignorasse i 549
>    ragazzi sbaglierebbe platea di metà.

# **La finestra 22-25 regge? Le transizioni una per una**

Il profilo qui sopra è un salto pooled 2021→2024: tre transizioni annuali compresse in
una. Prima di appendere una policy alla finestra 22-25 va dichiarato quanto della
finestra sta nel pooled e quanto si rivedrebbe in un anno qualsiasi.

In [24]:
# Le tre transizioni annuali dentro il salto 2021→2024: rapporti N(a+1, t+1) / N(a, t)
# dalle stesse età singole del profilo sopra. Dichiarano quanto della finestra 22-25 sta
# nel pooled triennale e quanto reggerebbe su un anno solo.
conteggi_eta = {anno: singole[singole["anno"].eq(anno)]
                  .groupby(["territorio", "genere", "eta"])["valore"].sum()
            for anno in (2021, 2022, 2023, 2024)}

righe = []
for t0 in (2021, 2022, 2023):
    for territorio in CONFRONTO:
        for genere in ("F", "M"):
            for a in range(15, 31):
                base = conteggi_eta[t0].get((territorio, genere, a))
                dopo = conteggi_eta[t0 + 1].get((territorio, genere, a + 1))
                righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                              "genere": genere, "transizione": f"{t0}-{t0 + 1}",
                              "eta_iniziale": a, "n_iniziale": round(base),
                              "rapporto_pct": round(100 * dopo / base, 1)})
transizioni = pd.DataFrame(righe)
transizioni.to_csv(PROCESSED / "genere_ritenzione_transizioni.csv", index=False)

finestra = transizioni[transizioni["territorio"].isin([BAGHERIA, ITALIA])
                       & transizioni["eta_iniziale"].between(22, 25)]
print("Rapporti annuali per età iniziale 22-25 (Italia come metro del rumore):")
print(finestra.pivot_table(index=["genere", "eta_iniziale"],
                           columns=["nome_territorio", "transizione"], values="rapporto_pct")
      [[("Bagheria", c) for c in ("2021-2022", "2022-2023", "2023-2024")]
       + [("Italia", c) for c in ("2021-2022", "2022-2023", "2023-2024")]].to_string())

# Ordine di grandezza del solo rumore di conteggio su una cella di Bagheria: se ogni
# residente restasse con probabilità r ~ 0.98 su n ~ 280, sd(rapporto) ~ sqrt(r(1-r)/n)
# ~ 0.8 pp. L'escursione osservata è più larga: le transizioni singole contengono flussi
# reali anno per anno, non solo rumore — ma nessuna da sola fa una finestra.
import numpy as np
n_tipico = transizioni.query("territorio == @BAGHERIA and 22 <= eta_iniziale <= 25")["n_iniziale"].median()
sd_conteggio = 100 * np.sqrt(0.98 * 0.02 / n_tipico)
escursione = (transizioni[transizioni["territorio"].eq(BAGHERIA)
                          & transizioni["eta_iniziale"].between(22, 25)]
              .groupby(["genere", "eta_iniziale"])["rapporto_pct"].agg(["min", "max"])
              .assign(escursione=lambda d: (d["max"] - d["min"]).round(1)))
print(f"\nBagheria, escursione min-max fra le tre transizioni (n mediano {n_tipico:.0f}, "
      f"solo conteggio darebbe ~±{sd_conteggio:.1f} pp):")
print(escursione.to_string())

sotto_cento = transizioni.query("territorio == @BAGHERIA and genere == 'F' "
                                "and 22 <= eta_iniziale <= 25 and rapporto_pct < 100")
print(f"\nCelle F 22-25 sotto quota 100: {len(sotto_cento)} su 12, "
      f"di cui {len(sotto_cento.query('transizione == \"2023-2024\"'))} nella transizione 2023-2024")

Rapporti annuali per età iniziale 22-25 (Italia come metro del rumore):
nome_territorio      Bagheria                        Italia                    
transizione         2021-2022 2022-2023 2023-2024 2021-2022 2022-2023 2023-2024
genere eta_iniziale                                                            
F      22               101.0     101.0     100.4     101.3     100.7     100.7
       23               103.9      99.0     102.0     101.3     100.9     100.8
       24               100.7     104.5      96.5     101.2     100.8     100.8
       25                97.8      99.0      96.8     101.3     100.7     100.7
M      22               100.0     101.3     100.6     101.2     101.3     101.5
       23                99.0     100.3     101.3     101.2     101.4     101.8
       24                97.8     100.0      99.3     101.2     101.4     101.5
       25                99.3      97.4      99.0     101.2     101.2     101.5



Bagheria, escursione min-max fra le tre transizioni (n mediano 290, solo conteggio darebbe ~±0.8 pp):
                       min    max  escursione
genere eta_iniziale                          
F      22            100.4  101.0         0.6
       23             99.0  103.9         4.9
       24             96.5  104.5         8.0
       25             96.8   99.0         2.2
M      22            100.0  101.3         1.3
       23             99.0  101.3         2.3
       24             97.8  100.0         2.2
       25             97.4   99.3         1.9

Celle F 22-25 sotto quota 100: 5 su 12, di cui 2 nella transizione 2023-2024


**📌 Risultato chiave** — La finestra 22-25 è una lettura **pooled, e va usata come
tale**. Nelle tre transizioni annuali il segno femminile c'è ma balla: 5 celle su 12
sotto quota 100 sulle età 22-25, con escursioni fino a **8 pp sulla stessa età** (le
24enni fanno 100.7, 104.5, 96.5) contro il ~±0.8 pp che darebbe il solo rumore di
conteggio su celle da ~290 ragazze — e contro l'Italia, che nelle stesse celle femminili
non esce mai dall'intervallo 100.7-101.3. La transizione più negativa è il 2023→2024
(96.5 e 96.8 sulle età 24 e 25). Conseguenza operativa, identica a quella dei KPI: il
claim si formula sul triennio 2021→2024 (le scale quinquennale e decennale ritrovano la
perdita, ma per entrambi i generi: sezione «I claim reggono al 2024?»), la transizione singola serve da controllo di direzione e non da titolo. Un
anno solo di dati post-intervento non potrà dire nulla sulla ritenzione: anche questa
lettura entra nella finestra triennale della sezione «Il KPI si può misurare?».

# **Il bilancio dei giovani: la platea del 2035 è già nata**

La ritenzione dice chi se ne va; il registro anagrafico dice anche chi arriverà. Chi
avrà 15-24 anni nel 2029 o nel 2034 oggi ha 10-19 o 5-14 anni ed è già contabile per
età singola: il futuro demografico del target, a migrazione ferma, è un'operazione di
conteggio. Ogni intervento della proposal agirà su questa platea, non su quella del
2024.

In [25]:
# Chi avrà 15-24 anni nel 2029 o nel 2034 è già nato e già residente: la platea futura del
# target si conta, non si prevede. Contabilità a saldo migratorio zero, con mortalità
# trascurabile a queste età (ordine di 0.1 per mille l'anno): per Bagheria, dove la
# ritenzione osservata sta sotto quota 100, il conteggio è quindi un tetto; per l'Italia,
# che sulle stesse età guadagna residenti, è semmai un pavimento.
p2024 = conteggi_eta[2024]

def platea(territorio, genere, a0, a1):
    return sum(p2024.get((territorio, genere, a)) for a in range(a0, a1 + 1))

righe = []
for territorio in CONFRONTO:
    for genere in ("F", "M"):
        oggi = platea(territorio, genere, 15, 24)
        a_2029 = platea(territorio, genere, 10, 19)   # 10-19enni del 2024
        a_2034 = platea(territorio, genere, 5, 14)    # 5-14enni del 2024
        righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                      "genere": genere,
                      "platea_2024": round(oggi), "platea_2029": round(a_2029),
                      "platea_2034": round(a_2034),
                      "var_2029_pct": round(100 * (a_2029 / oggi - 1), 1),
                      "var_2034_pct": round(100 * (a_2034 / oggi - 1), 1)})
platea_15_24 = pd.DataFrame(righe)
platea_15_24.to_csv(PROCESSED / "genere_platea.csv", index=False)

# Verifica incrociata sulla tavola indipendente delle classi quinquennali: stesso
# censimento ma query, file raw e percorso di parsing diversi. Y5-9 + Y10-14 deve
# ridare la somma delle età singole, per entrambi i generi.
quinquennali = leggi("censpop_demografia_classi_long.csv")
q_5_14 = (quinquennali[quinquennali["territorio"].eq(BAGHERIA) & quinquennali["anno"].eq(2024)
                       & quinquennali["eta"].isin(["Y5-9", "Y10-14"])
                       & quinquennali["genere"].isin(["M", "F"])
                       & quinquennali["cittadinanza"].eq("TOTAL")]
          .groupby("genere")["valore"].sum())
for genere in ("F", "M"):
    scarto = abs(q_5_14[genere] - platea(BAGHERIA, genere, 5, 14))
    assert scarto < 1, f"classi vs età singole, {genere}: scarto {scarto:.1f}"

print("Platea 15-24 futura a saldo migratorio zero (residenti 2024 fatti invecchiare):")
print(platea_15_24.pivot_table(index="nome_territorio", columns="genere",
                               values=["platea_2024", "platea_2029", "platea_2034"],
                               aggfunc="first").reindex(ORDINE).to_string())
print()
for genere in ("F", "M"):
    r = platea_15_24.query("territorio == @BAGHERIA and genere == @genere").iloc[0]
    print(f"  Bagheria {genere}: {r['platea_2024']:,} oggi -> {r['platea_2029']:,} nel 2029 "
          f"({r['var_2029_pct']:+.1f}%) -> {r['platea_2034']:,} nel 2034 ({r['var_2034_pct']:+.1f}%)")
bench = platea_15_24[platea_15_24["territorio"].ne(BAGHERIA)]
div_bag = abs(platea_15_24.query("territorio == @BAGHERIA and genere == 'F'")["var_2034_pct"].item()
              - platea_15_24.query("territorio == @BAGHERIA and genere == 'M'")["var_2034_pct"].item())
print(f"\n  Benchmark al 2034: fra {bench['var_2034_pct'].min():+.1f}% e "
      f"{bench['var_2034_pct'].max():+.1f}%, con F e M vicini in ogni territorio; "
      f"a Bagheria i due generi divergono di {div_bag:.1f} punti.")

Platea 15-24 futura a saldo migratorio zero (residenti 2024 fatti invecchiare):
                platea_2024          platea_2029          platea_2034         
genere                    F        M           F        M           F        M
nome_territorio                                                               
Bagheria               2882     3022        2651     2946        2435     2846
Palermo               32672    34363       31681    33072       28950    30260
Sicilia              243292   265707      230183   247066      210834   222117
Italia              2822968  3088740     2713088  2908421     2445360  2591788

  Bagheria F: 2,882 oggi -> 2,651 nel 2029 (-8.0%) -> 2,435 nel 2034 (-15.5%)
  Bagheria M: 3,022 oggi -> 2,946 nel 2029 (-2.5%) -> 2,846 nel 2034 (-5.8%)

  Benchmark al 2034: fra -16.4% e -11.4%, con F e M vicini in ogni territorio; a Bagheria i due generi divergono di 9.7 punti.


**📌 Risultato chiave** — La platea femminile del target **si restringe più in fretta
di quella maschile, e in modo più sbilanciato che in ogni territorio di confronto**. A
saldo migratorio zero le ragazze 15-24 di Bagheria passano da **2.882 (2024) a 2.651
nel 2029 (-8.0%) e 2.435 nel 2034 (-15.5%)**; i coetanei calano del 2.5% e del 5.8%.
Nei benchmark il calo al 2034 sta fra l'11.4% e il 16.4% ma è quasi simmetrico fra i
generi; a Bagheria i due generi divergono di 9.7 punti. Due conseguenze per la
proposal. *Dimensionamento*: i KPI espressi in teste vanno riparametrati — un
intervento tarato sulle 2.882 ragazze di oggi che parta nel 2027 lavora su una platea
già in rotta verso le 2.651 del 2029; quelli in tasso restano validi. *Urgenza*: il
conteggio è un **tetto**, perché la ritenzione osservata (sezioni precedenti) sta sotto
quota 100: il 2034 reale sarà verosimilmente sotto le 2.435 contate. L'asimmetria di
genere del calo, però, non nasce dalla migrazione dei giovani: viene da più a monte, ed
è la sezione successiva.

### La scomposizione del cambiamento: quanto pesa la platea, quanto il tasso
Il thread educazione scompone la variazione dei conteggi 2018-2024 in effetto-popolazione
ed effetto-tasso (shift-share: N₁−N₀ = (P₁−P₀)·q₀ + P₁·(q₁−q₀), identità esatta;
`edu_change_decomposition_2018_2024.csv`). Qui lo stesso formalismo si applica **per
genere** alle occupate 15-24 e si proietta in avanti sulla platea già nata: a tasso 2024
costante, il restringimento della platea è un tetto che si mangia da solo una parte del
KPI in teste. Esporta `genere_occupazione_scomposta.csv` e `genere_tetto_platea.csv`
(quest'ultimo letto da fig09).

In [26]:
def cella_lavoro(genere, anno, condizione, territorio=BAGHERIA):
    sel = istr_lav[
        istr_lav["tavola"].eq("lavoro") & istr_lav["territorio"].eq(territorio)
        & istr_lav["anno"].eq(anno) & istr_lav["eta"].eq("Y15-24")
        & istr_lav["cittadinanza"].eq("TOTAL") & istr_lav["titolo_studio"].eq("ALL")
        & istr_lav["genere"].eq(genere) & istr_lav["condizione"].eq(condizione)]
    assert len(sel) == 1
    return float(sel["valore"].iloc[0])


righe = []
for genere in ("F", "M", "T"):
    p0, p1 = cella_lavoro(genere, 2018, "99"), cella_lavoro(genere, 2024, "99")
    n0, n1 = cella_lavoro(genere, 2018, "1"), cella_lavoro(genere, 2024, "1")
    q0, q1 = n0 / p0, n1 / p1
    effetto_platea, effetto_tasso = (p1 - p0) * q0, p1 * (q1 - q0)
    assert abs((n1 - n0) - effetto_platea - effetto_tasso) < 1e-9  # identità esatta
    righe.append({"genere": genere, "occupati_2018": round(n0, 2), "occupati_2024": round(n1, 2),
                  "variazione": round(n1 - n0, 2), "effetto_platea": round(effetto_platea, 2),
                  "effetto_tasso": round(effetto_tasso, 2),
                  "quota_2018_pct": round(100 * q0, 2), "quota_2024_pct": round(100 * q1, 2)})
occupazione_scomposta = pd.DataFrame(righe)

# Cross-check sull'implementazione indipendente del thread educazione (riga "occupati",
# livello T): se diverge, uno dei due impianti è cambiato e va riallineato.
edu_dec = leggi("edu_change_decomposition_2018_2024.csv")
edu_occ = edu_dec[edu_dec["metrica"].eq("occupati")].iloc[0]
mia_t = occupazione_scomposta[occupazione_scomposta["genere"].eq("T")].iloc[0]
assert abs(mia_t["effetto_platea"] - float(edu_occ["effetto_popolazione"])) < 0.01
assert abs(mia_t["effetto_tasso"] - float(edu_occ["effetto_tasso"])) < 0.01
occupazione_scomposta.to_csv(PROCESSED / "genere_occupazione_scomposta.csv", index=False)

# Il tetto in avanti: platea 2029/2034 (già nata, celle sopra) per il tasso 2024 tenuto
# costante. Non è una previsione: è l'aritmetica del "KPI in teste riparametrato".
platea_bagheria = platea_15_24[platea_15_24["territorio"].eq(BAGHERIA)]
righe = []
for _, r in platea_bagheria.iterrows():
    occ24, pop24 = cella_lavoro(r["genere"], 2024, "1"), cella_lavoro(r["genere"], 2024, "99")
    assert round(pop24) == r["platea_2024"]  # registro e tavola lavoro: stessa base
    for orizzonte in (2029, 2034):
        platea_o = r[f"platea_{orizzonte}"]
        righe.append({"genere": r["genere"], "orizzonte": orizzonte, "platea": platea_o,
                      "tasso_2024_pct": round(100 * occ24 / pop24, 2),
                      "occupate_a_tasso_2024": round(platea_o * occ24 / pop24, 1),
                      "delta_vs_2024": round(platea_o * occ24 / pop24 - occ24, 1)})
tetto_platea = pd.DataFrame(righe)
tetto_platea.to_csv(PROCESSED / "genere_tetto_platea.csv", index=False)

f24 = occupazione_scomposta.set_index("genere").loc["F"]
tetto_f = tetto_platea[tetto_platea["genere"].eq("F")].set_index("orizzonte")
print(f"🔗 Occupate F 2018→2024: {f24['variazione']:+.0f} = effetto platea {f24['effetto_platea']:+.1f}"
      f" + effetto tasso {f24['effetto_tasso']:+.1f}. A tasso 2024 costante la sola platea vale"
      f" {tetto_f.loc[2029, 'delta_vs_2024']:+.1f} occupate al 2029"
      f" e {tetto_f.loc[2034, 'delta_vs_2024']:+.1f} al 2034.")
tetto_platea

🔗 Occupate F 2018→2024: +94 = effetto platea -7.2 + effetto tasso +101.2. A tasso 2024 costante la sola platea vale -18.9 occupate al 2029 e -36.6 al 2034.


,genere,orizzonte,platea,tasso_2024_pct,occupate_a_tasso_2024,delta_vs_2024
0,F,2029,2651,8.19,217.1,-18.9
1,F,2034,2435,8.19,199.4,-36.6
2,M,2029,2946,16.48,485.5,-12.5
3,M,2034,2846,16.48,469.0,-29.0


### Il KPI netto: l'obiettivo meno l'attrito della platea
Il KPI della proposal — allineare il tasso femminile a quello di Palermo, **+40 occupate**
— è calcolato sulla platea di **oggi**. Ma la platea del 2029 è già nata, ed è più piccola:
lo stesso tasso obiettivo applicato a meno teste produce meno occupate. Qui il target si
scompone in **lordo** (quello che si promette oggi), **attrito demografico** (quello che la
platea si riprende) e **netto** (quello che si vede alla scadenza).

Non è una previsione ed è una cosa diversa dal tetto della cella sopra: là il tasso resta
quello del 2024 e si misura il costo dell'inerzia, qui il tasso è quello obiettivo e si
misura quanto ne resta. Due tassi diversi, quindi due attriti diversi, e vanno detti
separatamente.


In [27]:
# Lordo, attrito, netto. `cella_lavoro` prende il territorio: il tasso obiettivo è quello
# osservato a Palermo sulla stessa fascia e lo stesso anno, non un parametro scelto qui.
tasso_obiettivo = (cella_lavoro("F", 2024, "1", PALERMO)
                   / cella_lavoro("F", 2024, "99", PALERMO))
occupate_oggi = cella_lavoro("F", 2024, "1")
plat_f = platea_15_24[platea_15_24["territorio"].eq(BAGHERIA)
                      & platea_15_24["genere"].eq("F")].iloc[0]

lordo = plat_f["platea_2024"] * tasso_obiettivo - occupate_oggi
# Il lordo è il numero già pubblicato in genere_gap_persone.csv: se diverge, una delle due
# celle ha cambiato definizione e la proposal citerebbe due KPI diversi con lo stesso nome.
assert abs(lordo - 40) < 1.5, f"KPI lordo {lordo:.1f}: non è più il +40 di fig09"

righe = []
for orizzonte in (2029, 2034):
    netto = plat_f[f"platea_{orizzonte}"] * tasso_obiettivo - occupate_oggi
    righe.append({"orizzonte": orizzonte,
                  "tasso_obiettivo_pct": round(100 * tasso_obiettivo, 2),
                  "occupate_2024": int(occupate_oggi),
                  "platea": int(plat_f[f"platea_{orizzonte}"]),
                  "kpi_lordo": round(lordo, 1),
                  "attrito_demografico": round(netto - lordo, 1),
                  "kpi_netto": round(netto, 1)})
kpi_netto = pd.DataFrame(righe)
# Identità: lordo + attrito = netto, riga per riga.
assert (kpi_netto["kpi_lordo"] + kpi_netto["attrito_demografico"]
        - kpi_netto["kpi_netto"]).abs().max() < 0.15
kpi_netto.to_csv(PROCESSED / "genere_kpi_netto.csv", index=False)

k29, k34 = kpi_netto.set_index("orizzonte").loc[2029], kpi_netto.set_index("orizzonte").loc[2034]
print(f"🔗 KPI al tasso di Palermo ({100 * tasso_obiettivo:.2f}%):"
      f" lordo {k29['kpi_lordo']:+.0f} occupate sulla platea di oggi;"
      f" al 2029 ne restano {k29['kpi_netto']:+.0f} ({k29['attrito_demografico']:+.0f} di attrito),"
      f" al 2034 {k34['kpi_netto']:+.0f} ({k34['attrito_demografico']:+.0f}).")
kpi_netto


🔗 KPI al tasso di Palermo (9.59%): lordo +40 occupate sulla platea di oggi; al 2029 ne restano +18 (-22 di attrito), al 2034 -2 (-43).


,orizzonte,tasso_obiettivo_pct,occupate_2024,platea,kpi_lordo,attrito_demografico,kpi_netto
0,2029,9.59,236,2651,40.4,-22.2,18.2
1,2034,9.59,236,2435,40.4,-42.9,-2.5


# **L'audit della platea: la sex ratio 5-14 che sale**

Un -15.5% femminile contro un -5.8% maschile non può venire da un calo delle nascite in
generale, che colpirebbe entrambi i generi: o fra i bambini di Bagheria mancano
specificamente le femmine, o una delle due tavole mente. Prima di usare il numero,
l'audit.

In [28]:
# Da dove viene l'asimmetria della platea: la sex ratio (maschi per 100 femmine) dei
# 5-14enni. Le classi quinquennali la seguono su 2001, 2011 e 2018-2024, con le età
# singole come tavola di controllo dal 2021: il 2001 e il 2011 sono nella norma, la
# salita è tutta successiva, nel decennio che segue il "muro" di fig10 (2001-2011).
cinque_14 = (quinquennali[quinquennali["eta"].isin(["Y5-9", "Y10-14"])
                          & quinquennali["genere"].isin(["M", "F"])
                          & quinquennali["cittadinanza"].eq("TOTAL")
                          & quinquennali["territorio"].isin(CONFRONTO)]
             .groupby(["anno", "territorio", "genere"])["valore"].sum().unstack("genere"))
sex_ratio = (cinque_14.assign(m_per_100f=lambda d: (100 * d["M"] / d["F"]).round(1))
             .reset_index())
sex_ratio["nome_territorio"] = sex_ratio["territorio"].map(NOMI)
sex_ratio.to_csv(PROCESSED / "genere_sex_ratio_5_14.csv", index=False)

print("Maschi per 100 femmine, età 5-14 (classi quinquennali):")
print(sex_ratio.pivot_table(index="anno", columns="nome_territorio", values="m_per_100f")
      [ORDINE].to_string())

# Controllo sulla tavola sorella (età singole, 2024) e scomposizione per cittadinanza:
# l'anomalia deve ritrovarsi identica e non essere un effetto di composizione straniera.
singole_5_14 = (popolazione[popolazione["territorio"].eq(BAGHERIA) & popolazione["anno"].eq(2024)
                            & popolazione["eta_anni"].between(5, 14) & popolazione["genere"].isin(["M", "F"])]
                .groupby(["cittadinanza", "genere"])["valore"].sum().unstack("genere"))
per_100 = (100 * singole_5_14["M"] / singole_5_14["F"]).round(1)
assert abs(per_100["TOTAL"] - sex_ratio.query("territorio == @BAGHERIA and anno == 2024")["m_per_100f"].item()) < 0.15
print(f"\nBagheria 2024 dalle età singole: {per_100['TOTAL']:.1f} totale, "
      f"{per_100['ITL']:.1f} fra gli italiani ({singole_5_14.loc['ITL'].sum():.0f} bambini) — "
      f"gli stranieri sono {singole_5_14.loc['FRGAPO'].sum():.0f} in tutto e non spostano il rapporto.")

# Compatibilità col caso: quota maschile osservata contro l'attesa data dalla quota
# italiana dello stesso anno. Le finestre 2001, 2011 e 2018 non condividono quasi nessuna
# coorte di nascita; il 2024 condivide col 2018 le nate 2010-2013. Un singolo z alto può
# essere sorte; una salita che attraversa finestre quasi disgiunte no.
print("\nScarto dalla quota maschile italiana (z della binomiale):")
for anno in (2001, 2011, 2018, 2024):
    m_b, f_b = cinque_14.loc[(anno, BAGHERIA), ["M", "F"]]
    m_i, f_i = cinque_14.loc[(anno, ITALIA), ["M", "F"]]
    n, p_oss, p_att = m_b + f_b, m_b / (m_b + f_b), m_i / (m_i + f_i)
    z = (p_oss - p_att) / (p_att * (1 - p_att) / n) ** 0.5
    print(f"  {anno}: {100 * p_oss:.1f}% maschi contro attesa {100 * p_att:.1f}%  "
          f"(n = {n:,.0f}, z = {z:+.1f})")

Maschi per 100 femmine, età 5-14 (classi quinquennali):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
anno                                               
2001                103.7    105.6    105.1   105.5
2011                107.2    105.4    105.8   106.1
2018                110.5    103.7    105.5   106.1
2019                111.6    104.1    105.5   106.1
2020                113.5    104.4    105.5   106.1
2021                114.0    104.1    105.0   106.0
2022                114.3    104.3    105.1   106.0
2023                117.2    104.7    105.2   106.0
2024                116.9    104.5    105.4   106.0

Bagheria 2024 dalle età singole: 116.9 totale, 116.8 fra gli italiani (5220 bambini) — gli stranieri sono 61 in tutto e non spostano il rapporto.

Scarto dalla quota maschile italiana (z della binomiale):
  2001: 50.9% maschi contro attesa 51.3%  (n = 7,029, z = -0.7)
  2011: 51.7% maschi contro attesa 51.5%  (n = 6,058, z = +0.4)
  2018: 52.5% maschi contro attesa 51.

**📌 Risultato chiave** — Fra i bambini di 5-14 anni Bagheria conta **117 maschi ogni
100 femmine (2024)** contro il 104-106 stabile di Palermo, Sicilia e Italia — e ~106 è
anche il livello naturale alla nascita. La deviazione è **recente e progressiva**:
103.7 nel 2001 e 107.2 nel 2011 (entrambi nella norma: z -0.7 e +0.4 rispetto alla
quota maschile italiana), poi 110.5 nel 2018 (z +1.5) e 116.9 nel 2024 (**z +3.5** su
~5.300 bambini). Regge ai controlli: identica su due tavole indipendenti dello stesso
censimento (classi quinquennali ed età singole), tutta dentro la popolazione italiana
(gli stranieri sono 61 in tutto), non attribuibile al caso sull'ultima finestra. La
salita parte dopo il 2011, nel decennio successivo al "muro" sul lavoro femminile di fig10. **Il
meccanismo resta aperto** — sorte accumulata sulle nascite, migrazione selettiva di
famiglie, dinamica di registro — e i due check che lo chiuderebbero sono piccoli e
dichiarati: i nati per sesso del comune (demo.istat) e la stessa serie sulle dieci
gemelle (estensione della tavola quinquennale); entrambi fonti nuove, non incluse in
questa consegna. Fino ad allora la platea femminile del 2034 si usa col suo numero (2.435 ragazze
contate oggi), ma l'asimmetria di genere si presenta **insieme a questo audit**, mai da
sola.

# **La componente straniera: il ricambio che non c'è**

Nel bilancio demografico italiano le uscite dei giovani sono in parte compensate dagli
ingressi di cittadini stranieri. Resta da misurare quanto questo canale esista a
Bagheria — con un vincolo noto: la componente straniera si può contare (`SETA_1`
incrocia cittadinanza, età singola e genere), ma non se ne può misurare l'occupazione
(nelle tavole lavoro comunali la cittadinanza è solo `TOTAL`).

In [29]:
# Il canale di ricambio che altrove attenua il calo: la popolazione straniera nel 15-34.
# Le età singole partono dal 2021, quindi la serie è 2021-2024; la scomposizione
# ITL + FRGAPO deve ricostruire il TOTAL riga per riga (stessa regola dei totali SDMX).
quindici_34 = popolazione[popolazione["territorio"].isin(CONFRONTO)
                          & popolazione["anno"].between(2021, 2024)
                          & popolazione["genere"].isin(["M", "F"])
                          & popolazione["eta_anni"].between(15, 34)]
per_citt = (quindici_34[quindici_34["cittadinanza"].isin(["ITL", "FRGAPO"])]
            .groupby(["territorio", "anno", "genere", "cittadinanza"])["valore"].sum()
            .unstack("cittadinanza"))
totale = quindici_34[quindici_34["cittadinanza"].eq("TOTAL")].groupby(
    ["territorio", "anno", "genere"])["valore"].sum()
assert (per_citt.sum(axis=1) - totale).abs().max() < 0.001, "ITL + FRGAPO non ricostruisce TOTAL"

stranieri = (per_citt.assign(totale=totale,
                             quota_stranieri_pct=lambda d: (100 * d["FRGAPO"] / d["totale"]).round(1))
             .reset_index())
stranieri["nome_territorio"] = stranieri["territorio"].map(NOMI)
stranieri.to_csv(PROCESSED / "genere_stranieri.csv", index=False)

print("Quota di cittadini stranieri nella popolazione 15-34 (%, M+F):")
insieme = (stranieri.groupby(["nome_territorio", "anno"])[["FRGAPO", "totale"]].sum()
           .assign(quota=lambda d: (100 * d["FRGAPO"] / d["totale"]).round(1)))
print(insieme["quota"].unstack("anno")[[2021, 2022, 2023, 2024]].reindex(ORDINE).to_string())

print("\nBagheria, variazione 2021 -> 2024 del 15-34 per cittadinanza e genere:")
bag = stranieri[stranieri["territorio"].eq(BAGHERIA)].set_index(["anno", "genere"])
for genere in ("F", "M"):
    d_itl = bag.loc[(2024, genere), "ITL"] - bag.loc[(2021, genere), "ITL"]
    d_frg = bag.loc[(2024, genere), "FRGAPO"] - bag.loc[(2021, genere), "FRGAPO"]
    print(f"  {genere}: italiani {d_itl:+,.0f}, stranieri {d_frg:+,.0f}")
tot_itl = bag.groupby("anno")["ITL"].sum()
tot_frg = bag.groupby("anno")["FRGAPO"].sum()
print(f"  insieme: italiani {tot_itl[2024] - tot_itl[2021]:+,.0f}, "
      f"stranieri {tot_frg[2024] - tot_frg[2021]:+,.0f} "
      f"({tot_frg[2024]:,.0f} stranieri 15-34 in tutto al 2024)")

Quota di cittadini stranieri nella popolazione 15-34 (%, M+F):
anno             2021  2022  2023  2024
nome_territorio                        
Bagheria          1.3   1.5   1.6   1.6
Palermo           4.4   4.4   4.8   5.1
Sicilia           5.4   5.6   5.9   6.4
Italia           11.7  11.8  12.0  12.4

Bagheria, variazione 2021 -> 2024 del 15-34 per cittadinanza e genere:
  F: italiani -219, stranieri +11
  M: italiani -131, stranieri +26
  insieme: italiani -350, stranieri +37 (195 stranieri 15-34 in tutto al 2024)


**📌 Risultato chiave** — Il ricambio dall'estero a Bagheria è **debole**: gli stranieri
sono l'**1.6% del 15-34** (195 persone al 2024) contro il 5.1% di Palermo, il 6.4% della
Sicilia e il 12.4% dell'Italia, un ordine di grandezza sotto il valore nazionale. Le
variazioni 2021-2024 per cittadinanza stampate sopra (italiani −350, stranieri +37) sono
però variazioni di **stock**, e lo stock non misura la fuga: mescola le partenze con la
diversa taglia delle coorti che entrano e che escono dalla fascia. La cella qui sotto
separa le due cose; per la fuga il notebook usa la ritenzione di coorte.


In [30]:
# Lo stock 15-34 non misura la fuga. Fra 2021 e 2024 cambia per due ragioni: le coorti che
# compiono 15 anni e quelle che superano i 34 non hanno la stessa taglia (ricambio d'età), e
# dentro le coorti già presenti partenze, arrivi e decessi si compensano (saldo). Con le età
# singole i due pezzi si separano: chi ha 15-34 anni nel 2024 ne aveva 12-31 nel 2021.
per_eta = (popolazione[popolazione["territorio"].isin(CONFRONTO) & popolazione["genere"].eq("T")
                       & popolazione["cittadinanza"].eq("TOTAL") & popolazione["stato_civile"].eq("ALL")
                       & popolazione["eta_anni"].notna()]
           .groupby(["territorio", "anno", "eta_anni"])["valore"].sum())


def somma_eta(territorio, anno, da, a):
    return float(per_eta.loc[(territorio, anno)].loc[da:a].sum())


righe = []
for t in CONFRONTO:
    s21, s24, stesse = somma_eta(t, 2021, 15, 34), somma_eta(t, 2024, 15, 34), somma_eta(t, 2021, 12, 31)
    righe.append({"territorio": t, "nome_territorio": NOMI[t], "stock_2021": s21, "stock_2024": s24,
                  "variazione": s24 - s21, "ricambio_eta": stesse - s21, "saldo_coorti": s24 - stesse,
                  "saldo_coorti_pct": round(100 * (s24 - stesse) / stesse, 2)})
stock_coorti = pd.DataFrame(righe)
assert (stock_coorti["ricambio_eta"] + stock_coorti["saldo_coorti"]
        - stock_coorti["variazione"]).abs().max() < 0.001
stock_coorti.to_csv(PROCESSED / "genere_stock_coorti.csv", index=False)
print("Popolazione 15-34, 2021 -> 2024: ricambio d'età e saldo dentro le stesse coorti")
print(stock_coorti.set_index("nome_territorio").reindex(ORDINE)
      [["variazione", "ricambio_eta", "saldo_coorti", "saldo_coorti_pct"]].round(2).to_string())


Popolazione 15-34, 2021 -> 2024: ricambio d'età e saldo dentro le stesse coorti
                 variazione  ricambio_eta  saldo_coorti  saldo_coorti_pct
nome_territorio                                                          
Bagheria             -313.0        -266.0         -47.0             -0.39
Palermo             -2630.0       -1867.0        -763.0             -0.56
Sicilia            -27452.0      -23559.0       -3893.0             -0.38
Italia             142291.0     -222672.0      364963.0              3.10


### Lo stock non è la fuga

**📌 Risultato chiave** — Dei 313 giovani 15-34 in meno fra 2021 e 2024, **266 sono ricambio
d'età** (le coorti che compiono 15 anni sono più piccole di quelle che superano i 34) e solo
**47 sono il saldo dentro le stesse coorti**: −0.39%, come la Sicilia (−0.38%), meno di
Palermo (−0.56%), mentre l'Italia cresce del 3.1% per immigrazione. Sull'insieme dei 15-34
Bagheria non perde più della Sicilia. Nel triennio 2021-2024 la specificità sembra stare
in **chi e quando**: le ragazze dopo i 24 anni, senza rientri (sezioni sulla ritenzione di
coorte). Sulle coorti seguite per cinque anni però la perdita all'uscita dal percorso
formativo è di entrambi i generi e al livello di Palermo e della Sicilia (sezione «I claim
reggono al 2024?»): anche il *chi* va citato come lettura di un solo triennio.


# **Contesto storico 2011**
Gli indicatori di genere di 8milaCensus. Sono calcolati sui **15 anni e più**, non sui giovani:
servono come sfondo, non come termine di paragone con le serie qui sopra.

In [31]:
GENERE_2011 = ["L1", "L2", "L6", "L7", "L10", "L11", "I1"]

storico = (ottomila[
        ottomila["territorio"].isin(CONFRONTO)
        & ottomila["anno"].eq(2011)
        & ottomila["indicatore"].isin(GENERE_2011)]
    .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
    .pivot(index=["indicatore", "nome_indicatore"], columns="territorio", values="valore"))
storico = storico[CONFRONTO].rename(columns={c: NOMI[c] for c in CONFRONTO}).round(1)
storico

,territorio,Bagheria,Palermo,Sicilia,Italia
indicatore,nome_indicatore,,,,
I1,Differenziali di genere per l'istruzione superiore,98.9,102.5,100.7,101.5
L1,Partecipazione al mercato del lavoro maschile,56.7,57.8,57.5,60.7
L10,Tasso di occupazione maschile,43.1,45.0,46.9,54.8
L11,Tasso di occupazione femminile,18.1,25.5,24.0,36.1
L2,Partecipazione al mercato del lavoro femminile,28.7,35.9,33.0,41.8
L6,Tasso di disoccupazione maschile,24.1,22.1,18.5,9.8
L7,Tasso di disoccupazione femminile,36.9,29.1,27.1,13.6


**📌 Risultato chiave** — Sfondo di lungo periodo (2011, 15+, fonte diversa): tasso di occupazione femminile a Bagheria **18.1% contro 36.1% nazionale**, la metà, e disoccupazione femminile 36.9% contro 13.6%. Lo svantaggio femminile locale non è un fatto recente. Non confrontabile con le serie 2018-2024: fascia e fonte diverse, citare sempre con entrambe.

> `L10`/`L11` danno il gap occupazionale complessivo del 2011, `L1`/`L2` quello sulla
> partecipazione. Non sono confrontabili con le serie 2018-2024 di questo notebook: fascia
> diversa (15+ contro 15-24) e fonte diversa. Vanno citati con anno e fascia, sempre.

### Verifica incrociata dei percentili storici (thread educazione)
Il thread educazione calcola gli stessi percentili sui 390 comuni con la propria pipeline
(`edu_historical_bagheria.csv`, metodo del rango percentile). Ricalcolo indipendente
dalla nostra `ottomilacensus_long.csv`: se i due impianti coincidono su ogni coppia
indicatore × anno, la posizione di Bagheria non dipende dall'implementazione.

In [32]:
edu_storico = leggi("edu_historical_bagheria.csv")
righe = []
for _, r in edu_storico.iterrows():
    comuni = ottomila[
        ottomila["livello"].eq("1") & ottomila["anno"].eq(r["anno"])
        & ottomila["indicatore"].eq(r["indicatore"])].dropna(subset=["valore"])
    mio = float(100 * comuni.set_index("territorio")["valore"].rank(pct=True).loc[BAGHERIA])
    righe.append({"indicatore": r["indicatore"], "anno": r["anno"],
                  "percentile_mio": round(mio, 1),
                  "percentile_edu": round(float(r["percentile_sicilia"]), 1)})
percentili_incrociati = pd.DataFrame(righe)
scarto = (percentili_incrociati["percentile_mio"] - percentili_incrociati["percentile_edu"]).abs().max()
assert scarto < 0.5, "i percentili delle due pipeline divergono"
p11 = percentili_incrociati[percentili_incrociati["anno"].eq(2011)].set_index("indicatore")["percentile_mio"]
print(f"🔗 Percentili identici su {len(percentili_incrociati)} coppie indicatore × anno"
      f" (scarto massimo {scarto:.2f}): il posizionamento non dipende dall'implementazione"
      f" (nel 2011: L4 {p11['L4']:.1f}° e L14 {p11['L14']:.1f}°).")
display(percentili_incrociati.pivot(index="indicatore", columns="anno", values="percentile_mio"))

# Export per fig10: la frattura del decennio 2001-2011 vista dall'istruzione. I5 è
# l'uscita precoce dal sistema di istruzione (alto = peggio) e si ferma al 2011 come
# tutto 8milaCensus. Il percentile è quello ricalcolato qui, non quello importato: le due
# pipeline coincidono (assert sopra), ma la figura cita una tavola del proprio thread.
frattura = (percentili_incrociati[percentili_incrociati["indicatore"].eq("I5")]
            .merge(edu_storico.loc[edu_storico["indicatore"].eq("I5"), ["anno", "valore"]],
                   on="anno")
            .rename(columns={"percentile_mio": "percentile_390"})
            .sort_values("anno"))
frattura["nome_indicatore"] = indicatori.set_index("indicatore").loc["I5", "nome_indicatore"]
frattura["direzione"] = "alto = sfavorevole"
frattura = frattura[["indicatore", "anno", "valore", "percentile_390",
                     "nome_indicatore", "direzione"]]
assert len(frattura) == 3
frattura.to_csv(PROCESSED / "genere_frattura_istruzione.csv", index=False)
print("🔗 Frattura educativa, uscita precoce (I5): "
      + ", ".join(f"{r.anno} {r.percentile_390:.0f}°" for r in frattura.itertuples())
      + " — nello stesso decennio in cui fig10 data il muro del lavoro.")


🔗 Percentili identici su 18 coppie indicatore × anno (scarto massimo 0.00): il posizionamento non dipende dall'implementazione (nel 2011: L4 87.8° e L14 12.9°).


anno,1991,2001,2011
indicatore,,,
I5,55.9,69.5,83.1
I6,63.5,46.9,38.7
I7,55.1,67.6,31.8
I8,3.7,17.4,33.5
L14,32.2,23.2,12.9
L4,17.2,79.0,87.8


🔗 Frattura educativa, uscita precoce (I5): 1991 56°, 2001 70°, 2011 83° — nello stesso decennio in cui fig10 data il muro del lavoro.


# **Bagheria nella distribuzione siciliana**
I quattro territori di confronto dicono se Bagheria è sopra o sotto la media, non dove si
colloca fra i comuni. L'unica fonte che copre **tutti i comuni siciliani** con un dato di
genere è 8milaCensus: indicatore `L11`, tasso di occupazione femminile, **2011, 15 anni e
più**. Fascia e anno diversi dalle serie di questo notebook — è un ritratto del contesto,
mai da accostare al 15-24 del censimento permanente.

Prepara la tabella per la mappa: valori uniti ai confini comunali (`pipeline/build.py`,
confini ISTAT 2026 generalizzati, EPSG:32633).

In [33]:
# I vertici arrivano già proiettati da pipeline/build.py: qui si uniscono ai valori e basta.
poligoni = pd.read_csv(PROCESSED / "comuni_sicilia_poligoni.csv",
                       dtype={"territorio": str, "nome_comune": str})
centroidi = pd.read_csv(PROCESSED / "comuni_sicilia_centroidi.csv",
                        dtype={"territorio": str, "nome_comune": str})

occ_femminile = (ottomila[
        ottomila["anno"].eq(2011)
        & ottomila["indicatore"].eq("L11")
        & ottomila["livello"].eq("1")][["territorio", "valore"]]
    .rename(columns={"valore": "occupazione_femminile_2011"}))

# Left join dai confini: un comune senza dato resta nella mappa come area vuota, non sparisce.
mappa = poligoni.merge(occ_femminile, on="territorio", how="left")
scoperti = sorted(set(poligoni["territorio"]) - set(occ_femminile["territorio"]))
mancanti = sorted(set(occ_femminile["territorio"]) - set(poligoni["territorio"]))
assert not mancanti, f"comuni con dato 2011 privi di confine: {mancanti}"
mappa.to_csv(PROCESSED / "genere_mappa_occupazione_femminile.csv", index=False)

# Posizione di Bagheria nella distribuzione dei 390 comuni.
valori = occ_femminile.set_index("territorio")["occupazione_femminile_2011"]
bagheria = valori[BAGHERIA]
percentile = 100 * (valori < bagheria).mean()
posizione = int((valori < bagheria).sum()) + 1

# Etichette della mappa: Bagheria, i 5 comuni più vicini e i due estremi regionali.
# Vicinanza = distanza euclidea fra centroidi già proiettati (EPSG:32633), in km:
# è la prossimità fisica, non l'adiacenza amministrativa (due comuni possono confinare
# ed essere lontani di centroide, e viceversa).
xy = centroidi.set_index("territorio")[["x", "y"]]
distanza_km = (xy - xy.loc[BAGHERIA]).pow(2).sum(axis=1).pow(0.5).div(1000)
vicini = distanza_km.drop(BAGHERIA).nsmallest(5)
estremi = [valori.idxmin(), valori.idxmax()]

etichette = pd.concat([
    pd.DataFrame({"territorio": [BAGHERIA], "ruolo": "Bagheria", "distanza_km": 0.0}),
    pd.DataFrame({"territorio": vicini.index, "ruolo": "vicino",
                  "distanza_km": vicini.to_numpy()}),
    pd.DataFrame({"territorio": estremi, "ruolo": ["minimo", "massimo"],
                  "distanza_km": distanza_km.reindex(estremi).to_numpy()}),
], ignore_index=True)
etichette = etichette.merge(centroidi, on="territorio", how="left")
etichette["valore"] = etichette["territorio"].map(valori)
assert etichette["valore"].notna().all(), "etichetta senza dato 2011"
etichette.to_csv(PROCESSED / "genere_mappa_etichette.csv", index=False)

print(f"comuni siciliani con dato 2011: {len(valori)}   confini disponibili: {poligoni['territorio'].nunique()}")
print("senza dato 2011:", [centroidi.set_index('territorio').loc[c, 'nome_comune'] for c in scoperti] or "nessuno")
print(f"\nBagheria: {bagheria:.1f}%  →  {posizione}° comune su {len(valori)} in ordine crescente "
      f"({percentile:.0f}° percentile)")
print(f"mediana siciliana {valori.median():.1f}%   min {valori.min():.1f}%   max {valori.max():.1f}%")
print("\nvicini di Bagheria (distanza fra centroidi, occupazione femminile 2011):")
for _, r in etichette[etichette["ruolo"].eq("vicino")].iterrows():
    print(f"  {r['nome_comune']:<22} {r['distanza_km']:5.1f} km   {r['valore']:.1f}%")
for ruolo in ("minimo", "massimo"):
    r = etichette[etichette["ruolo"].eq(ruolo)].iloc[0]
    print(f"  {ruolo:<8} regionale: {r['nome_comune']} ({r['valore']:.1f}%)")

comuni siciliani con dato 2011: 390   confini disponibili: 391
senza dato 2011: ['Misiliscemi']

Bagheria: 18.1%  →  49° comune su 390 in ordine crescente (12° percentile)
mediana siciliana 23.6%   min 13.0%   max 39.4%

vicini di Bagheria (distanza fra centroidi, occupazione femminile 2011):
  Santa Flavia             2.9 km   18.6%
  Ficarazzi                3.2 km   20.1%
  Villabate                4.7 km   16.6%
  Casteldaccia             7.9 km   20.1%
  Misilmeri                8.1 km   16.2%
  minimo   regionale: Francofonte (13.0%)
  massimo  regionale: Maniace (39.4%)


**📌 Risultato chiave** — Bagheria è nel **12° percentile siciliano** per occupazione femminile: solo 48 comuni su 390 stavano più in basso nel 2011 (49ª posizione in ordine crescente), contro una mediana regionale del 23.6%. Non è un comune medio della Sicilia, è nella coda bassa di una regione già ultima in Italia. Dato 2011, 15+, fonte 8milaCensus: contesto, non confrontabile con le serie 15-24 di questo notebook.

> Due avvertenze per l'uso in mappa. **Misiliscemi** (istituito nel 2021 staccandosi da
> Trapani) non ha un dato 2011 e resta in bianco: nel 2011 il suo territorio era dentro
> Trapani, e attribuirgli il valore trapanese sarebbe un'imputazione, non un dato.
> I **confini sono al 2026** mentre il dato è 2011: per tutti gli altri 390 comuni la
> corrispondenza è verificata dal join qui sopra (nessun comune con dato resta senza
> confine), e ISTAT non pubblica più le annate storiche su quello storage.

# **I comuni vicini: le stesse due serie, un pezzo di costa alla volta**
La mappa colloca Bagheria nella coda bassa siciliana insieme al suo vicinato, ma su un dato
del 2011 e sui 15 anni e più. Qui le due serie portanti del thread, il **gap occupazionale
15-24** (2018-2024) e la **ritenzione di coorte per età singola** (2021 a 2024), vengono
ricostruite con le stesse definizioni per i **cinque comuni geograficamente più vicini**,
selezionati sulla distanza fra centroidi nella cella della mappa.

I raw SDMX di questi comuni stanno in file separati (`censpop_lavoro_vicini`,
`censpop_popolazione_vicini`): le tavole condivise restano a quattro territori e gli altri
thread non cambiano numeri. Anche le uscite sono separate, `*_vicini.csv`, e le figure le
uniscono in lettura.

> **Attenzione alla taglia.** Sono comuni fra i 10.000 e i 28.000 abitanti: sulla fascia
> 15-24 i conteggi campionari sono piccoli e gli intervalli di confidenza larghi. Le loro
> serie vanno lette come contesto locale, non come stime puntuali da confrontare anno su
> anno con Bagheria.

In [34]:
# Le due serie del thread per i cinque comuni vicini. Stessi filtri e stesse formule dei
# territori di confronto: cambia solo la platea, così le curve sono confrontabili.
vicini_anagrafica = (pd.read_csv(PROCESSED / "genere_mappa_etichette.csv", dtype={"territorio": str})
                     .query("ruolo == 'vicino'")
                     .sort_values("distanza_km"))
CODICI_VICINI = vicini_anagrafica["territorio"].tolist()

CODICE_VICINATO = "VICINI5"
NOME_VICINATO = f"vicinato ({len(CODICI_VICINI)} comuni)"

istr_lav_vicini = leggi("censpop_istr_lav_vicini_long.csv")
lavoro_vicini = istr_lav_vicini[istr_lav_vicini["tavola"].eq("lavoro")]
istruzione_vicini = istr_lav_vicini[istr_lav_vicini["tavola"].eq("istruzione")]
pop_vicini = leggi("censpop_popolazione_vicini_long.csv")
assert set(lavoro_vicini["territorio"]) == set(CODICI_VICINI), "raw e selezione non coincidono"

# --- Gap occupazionale 15-24: definizione di analisi_condizione_15_24 (occupati = 1,
# popolazione = 99) e CI di Newcombe come per i territori di confronto.
base_v = lavoro_vicini[
    lavoro_vicini["eta"].eq("Y15-24")
    & lavoro_vicini["cittadinanza"].eq("TOTAL")
    & lavoro_vicini["titolo_studio"].eq("ALL")
    & lavoro_vicini["genere"].isin(["M", "F"])]
conteggi_v = (base_v.pivot_table(index=["territorio", "anno"], columns=["condizione", "genere"],
                                 values="valore", aggfunc="sum")
              .rename(columns={"1": "occupati", "99": "popolazione"}, level=0))
conteggi_v.columns = [f"{misura}_{gen}" for misura, gen in conteggi_v.columns]

def riga_gap(territorio, nome, anno, r):
    g, lo, hi = gap_con_ci(r["occupati_M"], r["popolazione_M"], r["occupati_F"], r["popolazione_F"])
    return {"territorio": territorio, "nome_territorio": nome, "anno": anno,
            "tasso_F": 100 * r["occupati_F"] / r["popolazione_F"],
            "tasso_M": 100 * r["occupati_M"] / r["popolazione_M"],
            "gap": 100 * g, "gap_lo": 100 * lo, "gap_hi": 100 * hi}


# Gli anni incompleti alla fonte (il 2020 sulla classe 15-24) restano fuori: niente riga,
# nessun valore inventato.
completi = conteggi_v[conteggi_v["popolazione_M"].gt(0) & conteggi_v["popolazione_F"].gt(0)]
righe = [riga_gap(territorio, NOMI[territorio], anno, r)
         for (territorio, anno), r in completi.iterrows()]
# Il vicinato come territorio unico: conteggi sommati e poi il rapporto, mai la media dei
# cinque rapporti (peserebbe Ficarazzi quanto Misilmeri). Il denominatore diventa dello
# stesso ordine di Bagheria e l'intervallo di confidenza si stringe di conseguenza.
righe += [riga_gap(CODICE_VICINATO, NOME_VICINATO, anno, r)
          for anno, r in completi.groupby(level="anno").sum().iterrows()]
ci_gap_vicini = pd.DataFrame(righe)
ci_gap_vicini["rapporto_M_F"] = ci_gap_vicini["tasso_M"] / ci_gap_vicini["tasso_F"]
ci_gap_vicini = ci_gap_vicini.round({"tasso_F": 1, "tasso_M": 1, "gap": 1, "gap_lo": 1,
                                     "gap_hi": 1, "rapporto_M_F": 2})
ci_gap_vicini.to_csv(PROCESSED / "genere_gap_occupazione_ci_vicini.csv", index=False)

# --- Istruzione 9-24 del vicinato e riga della forbice (fig05). Stessa definizione della
# cella sui titoli: almeno il diploma = USE_IF + BL + ML_RDD sul totale ALL. Il pivot somma
# i cinque comuni, quindi la quota è già quella del vicinato aggregato.
scuola_v = istruzione_vicini[
    istruzione_vicini["eta"].eq("Y9-24")
    & istruzione_vicini["cittadinanza"].eq("TOTAL")
    & istruzione_vicini["genere"].isin(["M", "F"])]
titoli_v = scuola_v.pivot_table(index=["anno", "genere"], columns="titolo_studio",
                                values="valore", aggfunc="sum")
scarto_v = (titoli_v[TITOLI].sum(axis=1) - titoli_v["ALL"]).abs().max()
assert scarto_v == 0, f"le categorie non ricostruiscono il totale, scarto {scarto_v}"
diploma_v = (100 * titoli_v[ALMENO_DIPLOMA].sum(axis=1) / titoli_v["ALL"]).round(1)

occ_v = ci_gap_vicini[ci_gap_vicini["territorio"].eq(CODICE_VICINATO)
                      & ci_gap_vicini["anno"].eq(anno_comune)].iloc[0]
forbice_vicini = pd.DataFrame([{
    "nome_territorio": NOME_VICINATO,
    "vantaggio_istruzione_F_pp": round(diploma_v[(anno_comune, "F")] - diploma_v[(anno_comune, "M")], 2),
    "rapporto_M_F_occupazione": occ_v["rapporto_M_F"],
    "tasso_occupazione_F": occ_v["tasso_F"],
    "tasso_occupazione_M": occ_v["tasso_M"],
    "almeno_diploma_F": diploma_v[(anno_comune, "F")],
    "almeno_diploma_M": diploma_v[(anno_comune, "M")],
    "anno": anno_comune,
}])
forbice_vicini.to_csv(PROCESSED / "genere_forbice_vicini.csv", index=False)

# --- Ritenzione per eta singola 2021 -> 2024, stessa media mobile a tre eta.
singole_v = (pop_vicini[
        pop_vicini["cittadinanza"].eq("TOTAL")
        & pop_vicini["genere"].isin(["M", "F"])
        & pop_vicini["eta_anni"].notna()]
    .assign(eta=lambda d: d["eta_anni"].astype(int)))
v21 = singole_v[singole_v["anno"].eq(2021)].groupby(["territorio", "genere", "eta"])["valore"].sum()
v24 = singole_v[singole_v["anno"].eq(2024)].groupby(["territorio", "genere", "eta"])["valore"].sum()

righe = []
for territorio in CODICI_VICINI:
    for genere in ("F", "M"):
        for a in range(13, 32):
            n0, n3 = v21.get((territorio, genere, a)), v24.get((territorio, genere, a + 3))
            righe.append({"territorio": territorio, "nome_territorio": NOMI[territorio],
                          "genere": genere, "eta_2021": a, "eta_2024": a + 3,
                          "n_2021": n0, "n_2024": n3,
                          "ritenzione_pct": round(100 * n3 / n0, 1) if n0 and n3 else None})
profilo_vicini = pd.DataFrame(righe)
for _, gruppo in profilo_vicini.groupby(["territorio", "genere"]):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    profilo_vicini.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)

# Il vicinato come un territorio solo: si sommano le coorti dei cinque comuni e poi si fa
# il rapporto, non la media dei cinque rapporti. Il denominatore diventa dello stesso
# ordine di Bagheria e la serie smette di oscillare di dieci punti per il rumore comunale.
# Le righe aggregate stanno nello stesso file con un codice territorio riconoscibile: chi
# somma la tabella per territorio deve escluderlo, altrimenti conta due volte le persone.
CODICE_VICINATO = "VICINI5"
pool = (profilo_vicini.groupby(["genere", "eta_2021", "eta_2024"], as_index=False)
        [["n_2021", "n_2024"]].sum()
        .assign(territorio=CODICE_VICINATO,
                nome_territorio=f"vicinato ({len(CODICI_VICINI)} comuni)"))
pool["ritenzione_pct"] = (100 * pool["n_2024"] / pool["n_2021"]).round(1)
for _, gruppo in pool.groupby("genere"):
    liscio = (100 * gruppo["n_2024"].rolling(3, center=True).sum()
              / gruppo["n_2021"].rolling(3, center=True).sum())
    pool.loc[gruppo.index, "ritenzione_rolling3_pct"] = liscio.round(1)
profilo_vicini = pd.concat([profilo_vicini, pool[profilo_vicini.columns]], ignore_index=True)
profilo_vicini.to_csv(PROCESSED / "genere_ritenzione_eta_vicini.csv", index=False)

# --- Composizione per stato, 15-24 (fig02). Stessi codici CL_FORZE_LAV della cella
# condivisa: 1 occupati, 12 in cerca, 5 studenti, 4 casalinghe/i, 7 altra condizione,
# 24 pensione, denominatore 99. Il pivot somma i cinque comuni prima della quota.
base_stati = lavoro_vicini[
    lavoro_vicini["eta"].eq("Y15-24")
    & lavoro_vicini["cittadinanza"].eq("TOTAL")
    & lavoro_vicini["titolo_studio"].eq("ALL")
    & lavoro_vicini["genere"].isin(["M", "F", "T"])]
largo_v = base_stati.pivot_table(index=["anno", "genere"], columns="condizione",
                                 values="valore", aggfunc="sum")
sei_stati_v = pd.DataFrame({
    "occupati": largo_v["1"], "in cerca": largo_v["12"], "studenti": largo_v["5"],
    "casalinghe/i": largo_v["4"], "altra condizione": largo_v["7"], "pensione": largo_v["24"],
})
composizione_vicini = ((100 * sei_stati_v.div(largo_v["99"], axis=0)).round(1)
    .reset_index()
    .assign(territorio=CODICE_VICINATO, nome_territorio=NOME_VICINATO)
    .melt(id_vars=["territorio", "nome_territorio", "anno", "genere"],
          var_name="stato", value_name="quota")
    # il 2020 manca alla fonte sulla classe 15-24: niente riga, come nel file condiviso
    .dropna(subset=["quota"]))
composizione_vicini.to_csv(PROCESSED / "genere_composizione_stato_dettaglio_vicini.csv", index=False)

# --- Ritenzione di coorte (fig03): stesse tre coorti della cella condivisa, coorti dei
# cinque comuni sommate prima del rapporto.
righe = []
for etichetta, (a0, a1) in COORTI.items():
    base = (singole_v[singole_v["anno"].eq(2021) & singole_v["eta"].between(a0, a1)]
            .groupby("genere")["valore"].sum())
    dopo = (singole_v[singole_v["anno"].eq(2024) & singole_v["eta"].between(a0 + 3, a1 + 3)]
            .groupby("genere")["valore"].sum())
    parziale = (100 * dopo / base).rename("ritenzione_%").reset_index()
    parziale["coorte"] = etichetta
    righe.append(parziale)
coorti_vicini = pd.concat(righe, ignore_index=True).assign(
    territorio=CODICE_VICINATO, nome_territorio=NOME_VICINATO)
coorti_vicini["ritenzione_%"] = coorti_vicini["ritenzione_%"].round(1)
coorti_vicini[["territorio", "genere", "ritenzione_%", "coorte", "nome_territorio"]].to_csv(
    PROCESSED / "genere_coorti_vicini.csv", index=False)

print(f"Composizione femminile 15-24 nel {anno_comune}, vicinato contro Bagheria e Palermo:")
confronto_stati = pd.concat([
    composizione_vicini,
    leggi("genere_composizione_stato_dettaglio.csv").query("territorio in [@BAGHERIA, @PALERMO]"),
])
confronto_stati["quota"] = pd.to_numeric(confronto_stati["quota"])
print(confronto_stati.query("anno == @anno_comune and genere == 'F'")
      .pivot_table(index="stato", columns="nome_territorio", values="quota").to_string())

print("\nRitenzione di coorte del vicinato:")
print(coorti_vicini.pivot_table(index="coorte", columns="genere", values="ritenzione_%").to_string())

print("\nGap occupazionale M-F 15-24, ultimo anno disponibile per comune:")
ultimo_anno = ci_gap_vicini.sort_values("anno").groupby("nome_territorio").tail(1)
print(ultimo_anno.set_index("nome_territorio")
      .reindex(vicini_anagrafica["nome_comune"])[["anno", "tasso_F", "tasso_M", "gap", "gap_lo", "gap_hi"]]
      .to_string())
print(f"\nVicinato aggregato {anno_comune}: gap {occ_v['gap']:.1f} punti",
      f"(CI {occ_v['gap_lo']:.1f}-{occ_v['gap_hi']:.1f}), rapporto {occ_v['rapporto_M_F']:.2f}x,",
      f"vantaggio educativo femminile {forbice_vicini['vantaggio_istruzione_F_pp'].iloc[0]:.1f} punti")
print("\nAmpiezza mediana del CI sul gap:",
      f"{(ci_gap_vicini['gap_hi'] - ci_gap_vicini['gap_lo']).median():.1f} punti nei vicini",
      f"contro {(ci_gap['gap_hi'] - ci_gap['gap_lo']).median():.1f} nei quattro territori di confronto")
donne_2225 = pool[pool["genere"].eq("F") & pool["eta_2021"].between(22, 25)]["n_2021"].sum()
print(f"\nVicinato aggregato: coorte femminile 22-25 del 2021 = {int(donne_2225)} persone,",
      "contro le", int(profilo[profilo["territorio"].eq(BAGHERIA) & profilo["genere"].eq("F")
                               & profilo["eta_2021"].between(22, 25)]["n_2021"].sum()), "di Bagheria")
print(pd.concat([profilo, pool])
      .query("genere == 'F' and 22 <= eta_2021 <= 25")
      .pivot_table(index="eta_2021", columns="nome_territorio", values="ritenzione_rolling3_pct")
      .to_string())

print("\nRitenzione femminile 22-25 anni (media mobile), per comune:")
print(profilo_vicini[profilo_vicini["genere"].eq("F") & profilo_vicini["eta_2021"].between(22, 25)]
      .pivot_table(index="eta_2021", columns="nome_territorio", values="ritenzione_rolling3_pct")
      .reindex(columns=vicini_anagrafica["nome_comune"]).to_string())

Composizione femminile 15-24 nel 2024, vicinato contro Bagheria e Palermo:
nome_territorio   Bagheria  Palermo  vicinato (5 comuni)
stato                                                   
altra condizione       6.4      5.0                  6.0
casalinghe/i          13.4     11.3                 13.6
in cerca               6.9      7.2                  8.0
occupati               8.2      9.6                  8.9
pensione               0.1      0.1                  0.1
studenti              65.1     66.8                 63.5

Ritenzione di coorte del vicinato:
genere              F     M
coorte                     
15-19 nel 2021   98.6  99.1
20-24 nel 2021  100.1  98.0
25-29 nel 2021  102.4  99.7

Gap occupazionale M-F 15-24, ultimo anno disponibile per comune:
              anno  tasso_F  tasso_M   gap  gap_lo  gap_hi
nome_comune                                               
Santa Flavia  2024      7.8     17.9  10.1     6.3    13.9
Ficarazzi     2024      9.1     14.7   5.6     2.4

**📌 Risultato chiave** — Il vicinato aggregato somiglia a Bagheria sul lavoro e se ne
separa sull'istruzione e sulla ritenzione. Sul lavoro: gap 2024 di 8.8 punti (CI 7.4-10.1),
rapporto M/F 1.98, casalinghe al 13.6% delle ragazze 15-24 contro il 13.4% di Bagheria.
Sul vantaggio educativo femminile no: 0.5 punti contro 4.2. E il cedimento femminile dopo i
24 anni **non c'è**: nel vicinato la coorte femminile 25-29 del 2021 è a 102.4 (Bagheria
96.3), e sulle età 22-25 la media mobile sta fra 100.4 e 101.7 (Bagheria da 103.3 a 97.1).

> Due letture compatibili con questi numeri, che la ritenzione netta non separa: la perdita
> femminile dopo i 24 anni è specifica di Bagheria anche rispetto ai cinque comuni più
> vicini, oppure è in parte un trasloco dentro l'area, a pochi chilometri, che conta come
> uscita per Bagheria e come ingresso per il vicinato. I singoli comuni oscillano troppo
> (fra 92.8 e 110.9 sulle stesse età) per decidere. In entrambi i casi la perdita femminile
> del triennio non va presentata come un tratto dell'area.

In [35]:
# Il proxy del NEET secondo la convenzione di repo — «giovani 15-24 fuori da lavoro e
# istruzione»: tutti meno occupati (1) e studenti (5), cioè in cerca (12) + casalinghe/i
# (4) + altra condizione (7) + pensione (24). Quota e persone, per la graffa di fig02.
# La convenzione di calcolo sta qui (docs/relazione/RELAZIONE_DATAPOLIS.md §1.1): R legge, non ricalcola.
FUORI = ["12", "4", "7", "24"]

def fuori_lavoro_istruzione(conteggi: pd.DataFrame) -> pd.DataFrame:
    valido = conteggi[conteggi["99"].gt(0)]  # il 2020 manca alla fonte sulla classe 15-24
    fuori = valido[FUORI].sum(axis=1)
    assert (fuori - (valido["99"] - valido["1"] - valido["5"])).abs().max() < 0.001, \
        "fuori ≠ popolazione − occupati − studenti: la partizione non torna"
    return pd.DataFrame({"quota_pct": (100 * fuori / valido["99"]).round(1),
                         "persone": fuori.round().astype(int)})

fuori_tutti = pd.concat([
    fuori_lavoro_istruzione(largo).reset_index(),
    fuori_lavoro_istruzione(largo_v).reset_index().assign(territorio=CODICE_VICINATO),
], ignore_index=True)
fuori_tutti["nome_territorio"] = fuori_tutti["territorio"].map(NOMI | {CODICE_VICINATO: NOME_VICINATO})
fuori_tutti = fuori_tutti[["territorio", "nome_territorio", "anno", "genere", "quota_pct", "persone"]]
fuori_tutti.to_csv(PROCESSED / "genere_fuori_lavoro_istruzione.csv", index=False)

print("Fuori da lavoro e istruzione, 15-24 — quota % e persone (2024):")
print(fuori_tutti[fuori_tutti["anno"].eq(2024) & fuori_tutti["genere"].isin(["F", "M"])]
      .pivot_table(index="nome_territorio", columns="genere", values=["quota_pct", "persone"])
      .reindex(ORDINE + [NOME_VICINATO]).to_string())


Fuori da lavoro e istruzione, 15-24 — quota % e persone (2024):
                      persone           quota_pct      
genere                      F         M         F     M
nome_territorio                                        
Bagheria                771.0     818.0      26.7  27.1
Palermo                7708.0    8131.0      23.6  23.7
Sicilia               53104.0   60359.0      21.8  22.7
Italia               421772.0  485149.0      14.9  15.7
vicinato (5 comuni)    1275.0    1345.0      27.6  27.2


# **Su 1.000 ragazze: istruzione e lavoro sulla stessa fascia**
La tavola istruzione pubblica solo il 9-24, quella del lavoro solo il 15-24: finora i due
gap convivevano su fasce diverse. Ma sotto i 15 anni un diploma è **strutturalmente
impossibile** (la secondaria di II grado si completa a 18-19 anni, le qualifiche IFP non
prima dei 17): il conteggio delle diplomate 9-24 *è* il conteggio 15-24, identico. Il
denominatore 15-24 arriva dalle età singole della demografia (disponibili dal 2021), e la
coerenza fra le due tavole si verifica con un assert, non si assume.

Così i due tassi vivono sulla **stessa popolazione** e si possono leggere su base 1.000.
Quello che l'estrazione **non** compra è la condizionalità: l'incrocio titolo × condizione
non esiste a livello comunale (sezione «Verifica di fattibilità»), quindi "quante delle
diplomate lavorano" resta non calcolabile. Le due quote si mostrano **in parallelo, mai
come stadi di un funnel** — una 16enne può lavorare senza diploma, gli insiemi non sono
annidati.

Il 18-24 usa la stessa logica dei bounds sulle casalinghe: nessun diploma sotto i 18 è un
**bound superiore** (qualche qualifica IFP si ottiene a 17).

In [36]:
# Diplomate/i 9-24 = 15-24 (nessun titolo sotto i 15); denominatori dalle età singole.
def pop_eta_singole(df, da_eta, a_eta, chiavi):
    """Somma delle età singole [da, a] — richiede il dettaglio per anno di età (2021+)."""
    base = df[df["eta_anni"].between(da_eta, a_eta)
              & df["cittadinanza"].eq("TOTAL") & df["stato_civile"].eq("ALL")
              & df["genere"].isin(["M", "F"])]
    return base.groupby(chiavi)["valore"].sum()


def diplomati_9_24(df, chiavi):
    base = df[df["eta"].eq("Y9-24") & df["cittadinanza"].eq("TOTAL")
              & df["genere"].isin(["M", "F"])]
    tab = base.pivot_table(index=chiavi, columns="titolo_studio", values="valore", aggfunc="sum")
    return pd.DataFrame({"diplomati": tab[ALMENO_DIPLOMA].sum(axis=1), "pop_9_24": tab["ALL"]})


def blocco_fasce(istr, pop, chiavi):
    return (diplomati_9_24(istr, chiavi)
            .join(pop_eta_singole(pop, 9, 14, chiavi).rename("pop_9_14"))
            .join(pop_eta_singole(pop, 15, 24, chiavi).rename("pop_15_24"))
            .join(pop_eta_singole(pop, 18, 24, chiavi).rename("pop_18_24")))


istruzione_confronto = istr_lav[istr_lav["tavola"].eq("istruzione")
                                & istr_lav["territorio"].isin(CONFRONTO)]
# Vicinato come territorio unico: conteggi dei cinque comuni sommati prima dei rapporti.
vicinato_fasce = (blocco_fasce(istruzione_vicini, pop_vicini, ["anno", "genere"])
                  .reset_index().assign(territorio=CODICE_VICINATO))
fasce = (pd.concat([
    blocco_fasce(istruzione_confronto, popolazione[popolazione["territorio"].isin(CONFRONTO)],
                 ["territorio", "anno", "genere"]).reset_index(),
    vicinato_fasce,
], ignore_index=True)
    .dropna(subset=["pop_15_24"])  # le età singole partono dal 2021: il 2018-2019 esce qui
    .set_index(["territorio", "anno", "genere"]))

# Coerenza fra le tavole: la popolazione 9-24 di istruzione deve ricostruirsi ESATTAMENTE
# dalle età singole della demografia. Se un giorno divergono, l'estrazione non è più lecita.
scarto = (fasce["pop_9_24"] - fasce["pop_9_14"] - fasce["pop_15_24"]).abs().max()
assert scarto == 0, f"tavole incoerenti: scarto massimo {scarto} persone"
assert (fasce["diplomati"] <= fasce["pop_15_24"]).all()

# Occupati 15-24: territori di confronto da `condizione`, vicinato dai conteggi già sommati.
occ_vicinato = (completi.reset_index().groupby("anno")[["occupati_F", "occupati_M"]].sum()
                .reset_index()
                .melt(id_vars="anno", var_name="misura", value_name="occupati")
                .assign(genere=lambda d: d["misura"].str[-1], territorio=CODICE_VICINATO))
occ = (pd.concat([
    condizione[condizione["territorio"].isin(CONFRONTO) & condizione["genere"].isin(["M", "F"])]
        [["territorio", "anno", "genere", "occupati"]],
    occ_vicinato[["territorio", "anno", "genere", "occupati"]],
], ignore_index=True)
    .set_index(["territorio", "anno", "genere"])["occupati"])

per_1000 = fasce.join(occ, how="inner").reset_index()
per_1000["nome_territorio"] = per_1000["territorio"].map(NOMI | {CODICE_VICINATO: NOME_VICINATO})
per_1000["almeno_diploma_15_24_%"] = (100 * per_1000["diplomati"] / per_1000["pop_15_24"]).round(1)
per_1000["occupazione_15_24_%"] = (100 * per_1000["occupati"] / per_1000["pop_15_24"]).round(1)
per_1000["almeno_diploma_18_24_bound_%"] = (100 * per_1000["diplomati"] / per_1000["pop_18_24"]).round(1)
per_1000["per_1000_diploma"] = (1000 * per_1000["diplomati"] / per_1000["pop_15_24"]).round().astype(int)
per_1000["per_1000_occupati"] = (1000 * per_1000["occupati"] / per_1000["pop_15_24"]).round().astype(int)
per_1000 = per_1000[["territorio", "nome_territorio", "anno", "genere", "pop_15_24",
                     "diplomati", "occupati", "almeno_diploma_15_24_%", "occupazione_15_24_%",
                     "per_1000_diploma", "per_1000_occupati", "pop_18_24",
                     "almeno_diploma_18_24_bound_%"]].sort_values(["territorio", "anno", "genere"])
per_1000.to_csv(PROCESSED / "genere_per_1000.csv", index=False)

ultimo = per_1000[per_1000["anno"].eq(per_1000["anno"].max())]
print(f"=== {ultimo['anno'].iloc[0]}, per 1.000 residenti 15-24 ===")
print(ultimo.pivot_table(index="nome_territorio", columns="genere",
                         values=["per_1000_diploma", "per_1000_occupati"]).astype(int))

# Il primato del vantaggio educativo, riletto sulle fasce allineate all'età da diploma.
vantaggio = ultimo.pivot_table(index="nome_territorio", columns="genere",
                               values=["almeno_diploma_15_24_%", "almeno_diploma_18_24_bound_%"])
print("\n=== vantaggio educativo F-M (pp) ===")
print(pd.DataFrame({
    "15-24": vantaggio["almeno_diploma_15_24_%", "F"] - vantaggio["almeno_diploma_15_24_%", "M"],
    "18-24 (bound)": vantaggio["almeno_diploma_18_24_bound_%", "F"] - vantaggio["almeno_diploma_18_24_bound_%", "M"],
}).round(1))

=== 2024, per 1.000 residenti 15-24 ===
                    per_1000_diploma      per_1000_occupati     
genere                             F    M                 F    M
nome_territorio                                                 
Bagheria                         510  462                82  165
Italia                           534  495               173  269
Palermo                          473  445                96  165
Sicilia                          509  462               104  203
vicinato (5 comuni)              470  455                89  177

=== vantaggio educativo F-M (pp) ===
                     15-24  18-24 (bound)
nome_territorio                          
Bagheria               4.8            5.7
Italia                 3.9            6.2
Palermo                2.8            4.5
Sicilia                4.7            7.1
vicinato (5 comuni)    1.5            2.5


**📌 Risultato chiave** — Su 1.000 ragazze 15-24 di Bagheria (2024): **510 con almeno il
diploma, 82 occupate**; sui coetanei: 462 e 165. La coerenza fra tavola istruzione ed età
singole è esatta (scarto zero, verificato dall'assert), quindi i due tassi poggiano sulla
stessa popolazione. Sulla fascia allineata il vantaggio educativo femminile resta (+4,8 pp
sul 15-24, +5,7 sul 18-24) ma il **primato di fig05 non regge fuori dal 9-24**: sul 18-24
Sicilia (+7,1) e Italia (+6,2) superano Bagheria — parte del primato era composizione per
età, come la «Verifica di composizione» sospettava. Regge su ogni fascia il distacco dal
vicinato (+5,7 contro +2,5) e la conversione peggiore: i claim della proposal usano questi
due, non il primato assoluto. Serie 2021-2024 stabile (48,8 → 51,0 sul 15-24 F).

In [37]:
# Export per fig05 (pannello del quadrante): vantaggio nel diploma e occupazione sulla
# STESSA fascia 15-24, dalle colonne di genere_per_1000.csv. La differenza F − M si
# calcola qui, non in R; la fotografia deve coincidere con genere_forbice.csv sul tasso
# femminile (denominatori diversi ma tavole coerenti a scarto zero: l'assert lo pinna).
largo_q = per_1000.pivot_table(index=["territorio", "nome_territorio", "anno"],
                               columns="genere",
                               values=["almeno_diploma_15_24_%", "occupazione_15_24_%"])
quadrante_forbice = pd.DataFrame({
    "vantaggio_diploma_15_24_pp": (largo_q[("almeno_diploma_15_24_%", "F")]
                                   - largo_q[("almeno_diploma_15_24_%", "M")]).round(1),
    "almeno_diploma_F": largo_q[("almeno_diploma_15_24_%", "F")],
    "almeno_diploma_M": largo_q[("almeno_diploma_15_24_%", "M")],
    "tasso_occupazione_F": largo_q[("occupazione_15_24_%", "F")],
    "tasso_occupazione_M": largo_q[("occupazione_15_24_%", "M")],
}).reset_index()
quadrante_forbice.to_csv(PROCESSED / "genere_forbice_quadrante.csv", index=False)

foto_q = quadrante_forbice[quadrante_forbice["anno"].eq(anno_comune)].set_index("nome_territorio")
assert abs(foto_q.loc["Bagheria", "tasso_occupazione_F"]
           - forbice.loc["Bagheria", "tasso_occupazione_F"]) < 0.11, "quadrante ≠ fotografia"
print(f"Quadrante {anno_comune} — vantaggio diploma 15-24 (pp) e tasso di occupazione F (%):")
print(foto_q[["vantaggio_diploma_15_24_pp", "tasso_occupazione_F"]].round(1).to_string())


Quadrante 2024 — vantaggio diploma 15-24 (pp) e tasso di occupazione F (%):
                     vantaggio_diploma_15_24_pp  tasso_occupazione_F
nome_territorio                                                     
Bagheria                                    4.8                  8.2
Palermo                                     2.8                  9.6
Italia                                      3.9                 17.3
Sicilia                                     4.7                 10.4
vicinato (5 comuni)                         1.5                  8.9


### La base è (quasi) universale da prima: I8 nel 2011
Contesto dal thread educazione (`edu_historical_benchmarks_2011.csv`): già nel 2011 il
titolo di base era pressoché universale — licenza media fra i 15-19enni al 96,7% a
Bagheria, in linea con tutti i benchmark. Il collo di bottiglia della catena "su 1.000
ragazze" non è mai stato l'accesso all'istruzione di base: è la conversione del titolo
in lavoro. Fonte 2011/8milaCensus: contesto, mai in serie con il 2021-2024 di questa
sezione.

In [38]:
edu_bench = leggi("edu_historical_benchmarks_2011.csv")
i8_edu = edu_bench[edu_bench["indicatore"].eq("I8")].set_index("territorio")["valore"]
i8_mio = ottomila[
    ottomila["anno"].eq(2011) & ottomila["indicatore"].eq("I8")
    & ottomila["territorio"].isin(CONFRONTO)].set_index("territorio")["valore"]
assert (i8_mio - i8_edu).abs().max() < 0.05  # stessa fonte, due pipeline
print("🔗 I8 2011 (almeno licenza media 15-19): "
      + ", ".join(f"{NOMI[t]} {i8_mio[t]:.1f}%" for t in CONFRONTO)
      + " — la base c'era già nel 2011, quando la coorte di oggi era alle elementari.")

🔗 I8 2011 (almeno licenza media 15-19): Bagheria 96.7%, Palermo 95.6%, Sicilia 96.5%, Italia 97.9% — la base c'era già nel 2011, quando la coorte di oggi era alle elementari.


# **Il gap delle madri: tre censimenti, 1991-2011**
La sezione precedente fotografa il 2011; qui si usa la profondità dei tre censimenti.
La domanda: il vantaggio educativo femminile è una
novità, o c'era già ai tempi delle madri — e si è mai convertito in occupazione?
Indicatori **15+ (I1: 6+)**, mai giovanili: la traiettoria si legge accanto alle serie
2018-2024, mai dentro. `I1` è un rapporto M/F × 100 sull'"almeno diploma": sotto 100 le
donne sono più istruite. Il percentile è la posizione fra i 390 comuni in ordine
crescente del valore: per disoccupazione (e per I1 letto come vantaggio maschile) alto
significa peggio. Caveat: le definizioni possono variare fra censimenti; la lettura è
sulla traiettoria delle posizioni, non sui decimali dei livelli.

In [39]:
MADRI = ["L2", "L11", "L10", "L7", "I1"]

valori_confronto = (ottomila[ottomila["territorio"].isin(CONFRONTO)
                             & ottomila["indicatore"].isin(MADRI)]
                    .pivot_table(index=["indicatore", "anno"], columns="territorio", values="valore")
                    [CONFRONTO].rename(columns=NOMI).round(1)
                    .reindex(MADRI, level=0))

comuni_madri = ottomila[ottomila["livello"].eq("1") & ottomila["indicatore"].isin(MADRI)]
righe = []
for (ind, anno), d in comuni_madri.groupby(["indicatore", "anno"]):
    v = d.set_index("territorio")["valore"].dropna()
    righe.append({"indicatore": ind, "anno": anno,
                  "percentile_390": round(100 * (v < v[BAGHERIA]).mean(), 1)})
percentili = pd.DataFrame(righe)

gap_madri = (valori_confronto.reset_index()
             .merge(percentili, on=["indicatore", "anno"])
             .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore"))
gap_madri.to_csv(PROCESSED / "genere_gap_madri.csv", index=False)

print("Valori (15+; I1: 6+) e percentile di Bagheria fra i 390 comuni:")
print(gap_madri.set_index(["indicatore", "anno"])
      [["Bagheria", "percentile_390", "Palermo", "Sicilia", "Italia"]].to_string())

Valori (15+; I1: 6+) e percentile di Bagheria fra i 390 comuni:
                 Bagheria  percentile_390  Palermo  Sicilia  Italia
indicatore anno                                                    
L2         1991      20.6             7.9     28.9     28.0    35.3
           2001      27.2            40.8     33.5     30.0    37.6
           2011      28.7            31.5     35.9     33.0    41.8
L11        1991      10.9            22.1     18.0     15.7    27.7
           2001      15.1            23.6     21.5     19.5    32.0
           2011      18.1            12.3     25.5     24.0    36.1
L10        1991      44.2            63.6     45.3     45.0    55.8
           2001      43.8            54.9     44.4     44.6    54.8
           2011      43.1            14.6     45.0     46.9    54.8
L7         1991      47.3            45.9     37.6     44.1    21.7
           2001      44.5            84.4     35.7     34.8    14.8
           2011      36.9            94.6     29.1  

**📌 Risultato chiave** — In trent'anni le donne di Bagheria **sono entrate nel mercato del lavoro e il mercato non le ha assorbite**: la partecipazione femminile sale da 20.6 a 28.7 (dall'8° al 32° percentile siciliano), ma l'occupazione femminile scivola dal 22° al **12° percentile** e la disoccupazione femminile crolla dal 46° al **95°** — peggio di quasi tutti. Intanto il differenziale educativo si chiude e si ribalta: I1 da 102.6 a **98.9**, sotto la parità già nel 2011, unico territorio del panel. Il vantaggio educativo non convertito **non è un incidente del censimento permanente: è il regime del territorio da almeno un decennio**.

> Tre letture della traiettoria:
>
> 1. **partecipazione su, assorbimento no** — L2 +8.1 punti in vent'anni mentre L7 passa
>    da 47.3 a 36.9 *restando* fra i peggiori comuni della Sicilia (95° percentile 2011):
>    il calo del tasso è il ciclo, la posizione è la struttura;
> 2. **il sorpasso educativo è del 2011**, non di oggi: ai tempi delle madri (1991) gli
>    uomini erano ancora più istruiti (I1 102.6). Le ragazze del censimento permanente
>    sono la prima generazione figlia del sorpasso — e trovano lo stesso muro;
> 3. l'occupazione femminile assoluta migliora (10.9 → 18.1) ma **la posizione peggiora**:
>    è la «corsa che arretra», osservata sul lato femminile. E non è solo
>    femminile: anche la posizione maschile crolla nel decennio 2001-2011 (L10 dal 55°
>    al 15° percentile) — lo scivolamento relativo è dell'intero mercato locale; la
>    specificità femminile è il muro della disoccupazione (95°) e il sorpasso educativo
>    che non si converte.

# **Le gemelle di Bagheria**
Palermo, Sicilia e Italia dicono se Bagheria è sotto la media, non se è anomala *fra i
comuni che le somigliano*. Il gruppo di controllo si costruisce sui **390 comuni al
2011** con variabili strutturali **non-outcome**: dimensione (`P1`), densità (`P7`),
struttura per età (`P11`, `P12`), stranieri (`S1`), patrimonio abitativo (`A1`, `A4`),
distanza dal capoluogo. Escluse per costruzione: istruzione e lavoro (sono gli esiti da
confrontare), famiglie `F*` (confrontate nella sezione successiva), mobilità `M*` (è il
meccanismo in esame nel thread mobilità), vulnerabilità `V*` (l'indice incorpora
istruzione e occupazione). Distanza di Mahalanobis (le variabili sono correlate; le code
asimmetriche entrano in log), k=10; robustezza su metodo e leave-one-variable-out.
Il matching è al 2011; la serie 2018-2024 delle gemelle sta nella sezione «Il ponte fra i
due censimenti».

In [40]:
import numpy as np

com_2011 = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(2011)]
            .pivot_table(index="territorio", columns="indicatore", values="valore"))
xy = centroidi.set_index("territorio")
com_2011["dist_palermo_km"] = np.hypot(xy["x"] - xy.loc[PALERMO, "x"],
                                       xy["y"] - xy.loc[PALERMO, "y"]) / 1000

MATCHING = ["P1", "P7", "P11", "P12", "S1", "A1", "A4", "dist_palermo_km"]
X = com_2011[MATCHING].copy()
X["P1"], X["P7"], X["S1"] = np.log10(X["P1"]), np.log10(X["P7"]), np.log1p(X["S1"])
assert not X.isna().any().any(), "matrice di matching incompleta"


def distanze_da_bagheria(matrice, metodo="mahalanobis"):
    if metodo == "mahalanobis":
        inversa = np.linalg.inv(np.cov(matrice.values.T))
        diff = matrice.values - matrice.loc[BAGHERIA].values
        d = pd.Series(np.sqrt(np.einsum("ij,jk,ik->i", diff, inversa, diff)), index=matrice.index)
    elif metodo == "pca4":  # kNN sulle prime 4 componenti principali, sferizzate
        Z = ((matrice - matrice.mean()) / matrice.std(ddof=0)).values
        _, S, Vt = np.linalg.svd(Z - Z.mean(0), full_matrices=False)
        proi = pd.DataFrame((Z @ Vt.T[:, :4]) / S[:4], index=matrice.index)
        d = np.sqrt(((proi - proi.loc[BAGHERIA]) ** 2).sum(axis=1))
    else:  # euclide su z-score
        Z = (matrice - matrice.mean()) / matrice.std(ddof=0)
        d = np.sqrt(((Z - Z.loc[BAGHERIA]) ** 2).sum(axis=1))
    return d.drop(BAGHERIA).sort_values()


K = 10
dist = distanze_da_bagheria(X)
gemelle = dist.head(K)
PROVINCE = {"081": "TP", "082": "PA", "083": "ME", "084": "AG", "085": "CL",
            "086": "EN", "087": "CT", "088": "RG", "089": "SR"}
anagrafica = pd.DataFrame({
    "rank": range(1, K + 1),
    "territorio": gemelle.index,
    "nome_comune": [NOMI[t] for t in gemelle.index],
    "provincia": [PROVINCE[t[:3]] for t in gemelle.index],
    "distanza_matching": gemelle.round(2).values,
    "popolazione_2011": com_2011.loc[gemelle.index, "P1"].astype(int).values,
    "dist_palermo_km": com_2011.loc[gemelle.index, "dist_palermo_km"].round(1).values,
})
anagrafica.to_csv(PROCESSED / "genere_gemelle.csv", index=False)
print(anagrafica.to_string(index=False))

overlap = {metodo: len(set(distanze_da_bagheria(X, metodo).head(K).index) & set(gemelle.index))
           for metodo in ("zscore", "pca4")}
lovo = {v: len(set(distanze_da_bagheria(X.drop(columns=v)).head(K).index) & set(gemelle.index))
        for v in MATCHING}
print(f"\nRobustezza — overlap col gruppo base: z-score {overlap['zscore']}/{K}, "
      f"PCA-4 {overlap['pca4']}/{K}; leave-one-variable-out min "
      f"{min(lovo.values())}/{K} (senza {min(lovo, key=lovo.get)})")

 rank territorio     nome_comune provincia  distanza_matching  popolazione_2011  dist_palermo_km
    1     082070 Termini Imerese        PA               1.74             26201             40.0
    2     082067    Santa Flavia        PA               2.21             10751             20.5
    3     082020          Capaci        PA               2.24             11030              7.4
    4     082073          Trabia        PA               2.41             10360             30.5
    5     082048       Misilmeri        PA               2.51             27570             17.5
    6     082005       Altofonte        PA               2.68             10266             12.2
    7     082071       Terrasini        PA               2.84             11985             20.4
    8     081008           Erice        TP               2.84             28012             64.6
    9     084028 Porto Empedocle        AG               2.84             16841             93.3
   10     084041         Sciac

In [41]:
# Bagheria dentro il suo gruppo: posizione sugli indicatori femminili, giovanili e
# familiari del 2011. "Gemelle sotto Bagheria" = quante hanno un valore più basso.
POSIZIONE = ["L11", "L7", "I1", "L4", "L14", "F4", "F7", "M2"]
righe = []
for ind in POSIZIONE:
    nel_gruppo = com_2011.loc[gemelle.index, ind]
    valore = com_2011.loc[BAGHERIA, ind]
    righe.append({"indicatore": ind,
                  "nome": indicatori.set_index("indicatore").loc[ind, "nome_indicatore"][:52],
                  "Bagheria": round(valore, 1),
                  "mediana gemelle": round(nel_gruppo.median(), 1),
                  "min": round(nel_gruppo.min(), 1), "max": round(nel_gruppo.max(), 1),
                  "gemelle sotto Bagheria": int((nel_gruppo < valore).sum())})
pd.DataFrame(righe).set_index("indicatore")

,nome,Bagheria,mediana gemelle,min,max,gemelle sotto Bagheria
indicatore,,,,,,
L11,Tasso di occupazione femminile,18.1,19.8,16.2,26.8,3
L7,Tasso di disoccupazione femminile,36.9,30.5,21.3,41.3,8
I1,Differenziali di genere per l'istruzione super...,98.9,101.2,97.0,107.0,1
L4,Incidenza giovani 15-29 anni che non studiano ...,40.1,41.2,30.4,46.8,4
L14,Tasso di occupazione 15-29 anni,20.0,22.5,19.3,25.0,2
F4,Incidenza di giovani che vivono da soli,3.6,4.4,3.6,8.6,0
F7,Incidenza di coppie giovani con figli,11.7,11.8,7.7,14.0,5
M2,Mobilità fuori comune per studio o lavoro,14.3,24.0,3.6,30.7,2


**📌 Risultato chiave** — Le dieci gemelle sono riconoscibili (Santa Flavia, Capaci, Misilmeri, Altofonte, Trabia, Terrasini, Termini Imerese nella cintura/costa palermitana, più Erice, Porto Empedocle, Sciacca: costieri di taglia media) e il gruppo è stabile (overlap 8/10 e 6/10 sugli altri metodi, LOVO mai sotto 7/10). Dentro il gruppo Bagheria è **nella norma su NEET 15-29 e occupazione femminile (2011)** — ma è **estrema dove il thread ha già puntato**: il differenziale educativo più favorevole alle donne (I1 98.9, una sola gemella più in basso), la disoccupazione femminile fra le peggiori (8/10 sotto), **l'autonomia giovanile al minimo del gruppo** (F4 3.6%: nessuna gemella più in basso) e quasi la mobilità minima (M2: 2/10 sotto).

> La risposta all'obiezione "ma non è così in tutta la Sicilia?": sui livelli generali sì,
> Bagheria sta come chi le somiglia — il problema femminile giovanile è di scala regionale
> e la proposal deve dirlo. Ma il **profilo** di Bagheria dentro il gruppo è specifico:
> più capitale umano femminile relativo, più donne in coda al mercato, meno giovani
> autonomi, meno mobilità. È il ritratto strutturale della talent trap di genere, e rende
> i target dei KPI difendibili: "la mediana delle gemelle" è un obiettivo che "la media
> nazionale" non può essere. Matching al 2011; la serie 2018-2024 delle gemelle è
> nella sezione «Il ponte fra i due censimenti».

In [42]:
# Export per fig08: la stessa posizione della cella sopra, ma con il gruppo intero
# (min-q1-mediana-q3-max) e il percentile sui 390 comuni, così la figura disegna la
# distribuzione e non solo un rank. `verso` è l'unica scelta interpretativa e sta qui,
# non in R: +1 = alto è meglio, -1 = alto è peggio, 0 = descrittivo, nessun verso "buono".
# F7 e M2 sono a 0 di proposito: il notebook mostra che sono tratti di fascia costiera
# (F7) e che la mobilità (M2) non ha un segno univoco senza il thread pendolarismo.
#
# Due lenti, non una. Accanto alle gemelle strutturali entra il gruppo di pari del thread
# educazione, appaiato su variabili diverse — include il profilo educativo, esclude gli
# esiti. I due gruppi condividono un solo comune: la seconda colonna è quindi un test di
# robustezza vero, non una replica. Si leggono affiancate e non si fondono mai in una
# classifica sola. L14 (occupazione 15-29) è l'indicatore su cui divergono: senza, la
# differenza fra le due definizioni di "simile" non si vedrebbe.
VERSO = {"L11": 1, "L7": -1, "I1": -1, "L4": -1, "L14": 1, "F4": 1, "F7": 0, "M2": 0}
assert set(VERSO) == set(POSIZIONE)

pari_edu = leggi("edu_matched_peers_2011.csv")
pari_edu = list(pari_edu[pari_edu["ruolo"].eq("Peer")]["territorio"])
assert len(pari_edu) == 10 and set(pari_edu) <= set(com_2011.index)

nomi = indicatori.set_index("indicatore")["nome_indicatore"]
LENTI = (("gemelle", list(gemelle.index)), ("istruiti", pari_edu))
righe = []
for ind in POSIZIONE:
    valore = com_2011.loc[BAGHERIA, ind]
    regione = com_2011[ind].dropna()
    riga = {"indicatore": ind, "nome": nomi.loc[ind], "verso": VERSO[ind],
            "bagheria": round(valore, 1)}
    for prefisso, gruppo in LENTI:
        nel_gruppo = com_2011.loc[gruppo, ind]
        riga |= {f"{prefisso}_min": round(nel_gruppo.min(), 1),
                 f"{prefisso}_q1": round(nel_gruppo.quantile(0.25), 1),
                 f"{prefisso}_mediana": round(nel_gruppo.median(), 1),
                 f"{prefisso}_q3": round(nel_gruppo.quantile(0.75), 1),
                 f"{prefisso}_max": round(nel_gruppo.max(), 1),
                 f"{prefisso}_sotto": int((nel_gruppo < valore).sum()),
                 f"n_{prefisso}": int(nel_gruppo.notna().sum())}
    riga |= {"percentile_390": round((regione < valore).mean() * 100, 1),
             "n_regione": int(regione.size)}
    righe.append(riga)

posizionamento = pd.DataFrame(righe)
# Il valore 2011 di L14 deve coincidere con quello che il thread educazione pubblica:
# stesso indicatore, due percorsi di lettura indipendenti.
edu_l14 = float(leggi("edu_matched_peers_2011.csv").query("ruolo == 'Bagheria'")["L14"].iloc[0])
assert abs(posizionamento.set_index("indicatore").loc["L14", "bagheria"] - edu_l14) < 0.05
posizionamento.to_csv(PROCESSED / "genere_posizionamento.csv", index=False)
posizionamento


,indicatore,nome,verso,bagheria,gemelle_min,gemelle_q1,gemelle_mediana,gemelle_q3,gemelle_max,gemelle_sotto,n_gemelle,istruiti_min,istruiti_q1,istruiti_mediana,istruiti_q3,istruiti_max,istruiti_sotto,n_istruiti,percentile_390,n_regione
0,L11,Tasso di occupazione femminile,1,18.1,16.2,17.8,19.8,22.7,26.8,3,10,16.2,19.9,21.4,21.8,25.1,1,10,12.3,390
1,L7,Tasso di disoccupazione femminile,-1,36.9,21.3,28.0,30.5,35.1,41.3,8,10,23.0,29.6,32.0,34.9,41.3,8,10,94.6,390
2,I1,Differenziali di genere per l'istruzione super...,-1,98.9,97.0,100.1,101.2,103.4,107.0,1,10,87.0,94.5,98.7,102.7,107.0,5,10,37.4,390
3,L4,Incidenza giovani 15-29 anni che non studiano ...,-1,40.1,30.4,37.4,41.2,42.7,46.8,4,10,31.4,36.2,38.8,42.8,45.5,6,10,87.4,390
4,L14,Tasso di occupazione 15-29 anni,1,20.0,19.3,21.2,22.5,23.3,25.0,2,10,19.9,21.5,24.1,26.6,32.4,2,10,12.6,390
5,F4,Incidenza di giovani che vivono da soli,1,3.6,3.6,4.0,4.4,4.9,8.6,0,10,3.6,4.0,4.3,6.0,9.6,0,10,3.3,390
6,F7,Incidenza di coppie giovani con figli,0,11.7,7.7,10.6,11.8,13.0,14.0,5,10,9.7,11.3,13.4,14.2,14.9,4,10,81.0,390
7,M2,Mobilità fuori comune per studio o lavoro,0,14.3,3.6,16.3,24.0,27.4,30.7,2,10,4.6,6.1,11.6,23.3,26.7,6,10,25.4,390


### La seconda lente sui pari: i peer "istruiti" del thread educazione
Il thread educazione costruisce dieci comparabili con un disegno diverso
(`edu_matched_peers_2011.csv`): caliper di popolazione 0,5-2× su tutta l'isola e distanza
euclidea che **include il profilo educativo** (I5, I6, I7) ed esclude gli esiti L14/L4.
Le gemelle di questa sezione rispondono a "comuni strutturalmente simili"; i suoi a
"comuni ugualmente scolarizzati". L'overlap è un solo comune, e le due lenti sono
complementari, non concorrenti: fra i pari strutturali l'occupazione femminile di
Bagheria è nella norma (lo svantaggio è di fascia costiera), fra i pari a pari istruzione
l'occupazione giovanile resta sotto la mediana — il titolo non manca, non si converte.
Nella proposal le due lenti vanno dichiarate insieme, mai fuse in un'unica classifica.

In [43]:
edu_peer = leggi("edu_matched_peers_2011.csv")
peer_suoi = edu_peer[edu_peer["ruolo"].eq("Peer")].copy()
peer_suoi["L14"] = pd.to_numeric(peer_suoi["L14"])
gemelle_csv = leggi("genere_gemelle.csv")
overlap = peer_suoi.merge(gemelle_csv, on="territorio")
mediana_l14 = peer_suoi["L14"].median()
bagheria_l14 = float(edu_peer[edu_peer["ruolo"].eq("Bagheria")]["L14"].iloc[0])
assert len(peer_suoi) == 10 and abs(mediana_l14 - 24.1) < 0.05
print("peer del thread educazione:", ", ".join(sorted(peer_suoi["nome_territorio"])))
print("overlap con le gemelle strutturali:",
      ", ".join(overlap["nome_comune"]) if len(overlap) else "nessuno")
print(f"🔗 Due lenti: a pari istruzione Bagheria fa {bagheria_l14:.1f}% di occupazione 15-29"
      f" contro una mediana peer di {mediana_l14:.1f}% ({bagheria_l14 - mediana_l14:+.1f} p.p.);"
      " fra le gemelle strutturali è nella norma su L11. Il titolo c'è, la conversione no.")

# Export per fig08: chi sta in quale lente. La figura conta l'overlap da qui e nomina il
# comune condiviso — nessun elenco ricopiato a mano in R.
lenti = pd.concat([
    pd.DataFrame({"territorio": list(gemelle.index), "lente": "strutturale"}),
    pd.DataFrame({"territorio": list(peer_suoi["territorio"]), "lente": "istruzione"}),
])
lenti = (lenti.groupby("territorio")["lente"]
         .agg(lambda s: "entrambe" if len(s) == 2 else s.iloc[0])
         .reset_index())
lenti["nome_comune"] = lenti["territorio"].map(NOMI)
assert int(lenti["lente"].eq("entrambe").sum()) == len(overlap)
lenti.sort_values(["lente", "nome_comune"]).to_csv(
    PROCESSED / "genere_pari_lenti.csv", index=False)


peer del thread educazione: Canicattì, Carini, Castelvetrano, Mazara del Vallo, Misilmeri, Misterbianco, Monreale, Partinico, Paternò, Vittoria
overlap con le gemelle strutturali: Misilmeri
🔗 Due lenti: a pari istruzione Bagheria fa 20.0% di occupazione 15-29 contro una mediana peer di 24.1% (-4.1 p.p.); fra le gemelle strutturali è nella norma su L11. Il titolo c'è, la conversione no.


# **La nuvola dei 390: istruzione × occupazione femminile (2011)**
Il quadrante della sezione "I due gap a confronto" ha quattro territori; qui lo stesso
piano si costruisce per **tutti i comuni siciliani** — `I1` (differenziale educativo,
6+) contro `L11` (occupazione femminile, 15+), 2011. Serve a dare scala: Bagheria smette
di essere un caso isolato e diventa un punto in una distribuzione. Indicatori 15+/6+ e
fonte 2011: contesto strutturale, mai da unire al quadrante giovanile 2018-2024 — in
`fig06` i due pannelli sono affiancati e etichettati, non sovrapposti.

In [44]:
from scipy import stats

nuvola = (com_2011[["I1", "L11"]].dropna().reset_index()
          .assign(nome_comune=lambda d: d["territorio"].map(NOMI),
                  gemella=lambda d: d["territorio"].isin(gemelle.index),
                  evidenzia=lambda d: d["territorio"].map({BAGHERIA: "Bagheria", PALERMO: "Palermo"})))
nuvola.to_csv(PROCESSED / "genere_nuvola_390.csv", index=False)

rho, p_rho = stats.spearmanr(nuvola["I1"], nuvola["L11"])
vantaggio_f = nuvola["I1"] < 100
print(f"Spearman I1 × L11 su {len(nuvola)} comuni (2011): rho = {rho:.2f} (p = {p_rho:.1e})")
print(f"Comuni con vantaggio educativo femminile (I1 < 100): {vantaggio_f.sum()} su {len(nuvola)}")
print(f"  occupazione femminile mediana: {nuvola.loc[vantaggio_f, 'L11'].median():.1f}% fra questi, "
      f"{nuvola.loc[~vantaggio_f, 'L11'].median():.1f}% fra gli altri")
print(f"Bagheria: I1 = {com_2011.loc[BAGHERIA, 'I1']:.1f}, L11 = {com_2011.loc[BAGHERIA, 'L11']:.1f} "
      f"— vantaggio educativo femminile con occupazione femminile al 12° percentile")

Spearman I1 × L11 su 390 comuni (2011): rho = -0.24 (p = 1.2e-06)
Comuni con vantaggio educativo femminile (I1 < 100): 170 su 390
  occupazione femminile mediana: 24.9% fra questi, 22.4% fra gli altri
Bagheria: I1 = 98.9, L11 = 18.1 — vantaggio educativo femminile con occupazione femminile al 12° percentile


In [45]:
# La stessa nuvola con la mobilità fuori comune (M2) al posto dell'istruzione: è la
# correlazione ecologica da cui parte l'ipotesi «più mobilità, più lavoro femminile»,
# che la relazione (sezione 5.5) e la policy (componente E) citano e poi smontano.
mob_l11 = com_2011[["M2", "L11"]].dropna()
rho_m2, p_m2 = stats.spearmanr(mob_l11["M2"], mob_l11["L11"])
print(f"Spearman M2 × L11 su {len(mob_l11)} comuni (2011): rho = {rho_m2:+.2f} (p = {p_m2:.1e})")

Spearman M2 × L11 su 390 comuni (2011): rho = +0.32 (p = 1.1e-10)


**📌 Risultato chiave** — Sui 390 comuni la relazione va nel verso "giusto": **dove le donne sono relativamente più istruite, l'occupazione femminile è più alta** (Spearman fra I1 e L11: -0.24, p<0.001; mediana L11 24.9% nei 170 comuni con I1<100 contro 22.4% negli altri). Bagheria contraddice il pattern: sta **nel quadrante vantaggio-educativo-femminile / occupazione-bassa**, con L11 al 12° percentile. La nuvola è il pannello di contesto di `fig06`, con le gemelle evidenziate.

> Correlazione **ecologica** su dati 2011: orienta l'ipotesi ("l'istruzione femminile di
> solito si converte, qui no"), non dimostra alcun nesso individuale. Il valore aggiunto è
> la scala: il paradosso di Bagheria non è "la Sicilia va così" — la maggioranza dei
> comuni col suo profilo educativo occupa di più.

### Il residuo che la nuvola non quantifica: i modelli comunali del thread educazione
La nuvola dice che Bagheria contraddice il pattern I1 × L11; i modelli del thread
educazione (`edu_model_robustness_2011.csv`) mettono un numero sullo scarto: tre
regressioni sui 389 comuni (Bagheria esclusa dal training) prevedono dal profilo
educativo e strutturale un'occupazione giovanile L14 più alta dell'osservata di ~6
punti, e un NEET 15-29 (`L4`, 2011) più basso dell'osservato. Con le stesse cautele dichiarate lì:
R² out-of-sample debole (CV 10-fold ≤ 0,05 su L14, e negativo in due modelli su tre), confronto ecologico, l'intervallo
bootstrap quantifica l'incertezza dei coefficienti e non è un intervallo di previsione.
Un controllo di robustezza convergente, non una prova.

In [46]:
edu_modelli = leggi("edu_model_robustness_2011.csv")
for colonna in ("r2_cv_10fold", "previsto_bagheria", "residuo_bagheria",
                "residuo_ci95_basso", "residuo_ci95_alto"):
    edu_modelli[colonna] = pd.to_numeric(edu_modelli[colonna])
assert edu_modelli["n_comuni_training"].astype(int).eq(389).all()
l14_c = edu_modelli[edu_modelli["outcome"].eq("L14")
                    & edu_modelli["modello"].eq("C_contesto_territoriale")].iloc[0]
assert abs(l14_c["residuo_bagheria"] - (-5.9)) < 0.1  # pin: se il suo impianto cambia, qui si vede
print(f"🔗 L14 osservato − previsto (modello C): {l14_c['residuo_bagheria']:+.1f} p.p."
      f" [bootstrap 95%: {l14_c['residuo_ci95_basso']:+.1f}, {l14_c['residuo_ci95_alto']:+.1f}],"
      f" CV R² {l14_c['r2_cv_10fold']:.2f} — convergente con la nuvola, da citare con le sue cautele.")
edu_modelli[["outcome", "modello", "r2_cv_10fold", "previsto_bagheria", "residuo_bagheria"]].round(2)

🔗 L14 osservato − previsto (modello C): -5.9 p.p. [bootstrap 95%: -7.7, -4.3], CV R² 0.05 — convergente con la nuvola, da citare con le sue cautele.


,outcome,modello,r2_cv_10fold,previsto_bagheria,residuo_bagheria
0,L14,A_istruzione,-0.05,25.95,-5.95
1,L14,B_istruzione_mobilita,-0.04,25.66,-5.66
2,L14,C_contesto_territoriale,0.05,25.93,-5.93
3,L4,A_istruzione,0.38,36.12,3.98
4,L4,B_istruzione_mobilita,0.38,36.12,3.98
5,L4,C_contesto_territoriale,0.45,38.28,1.82


# **Formazione familiare precoce e autonomia abitativa (2011)**
Il canale ipotizzato della formazione familiare precoce, misurato con gli indicatori `F` mai usati finora: giovani soli (`F4`), monogenitori
giovani (`F5`), coppie giovani senza e con figli (`F6`, `F7`). Avvertenza di
denominatore: `F5`-`F7` sono quote sul **totale delle famiglie**, quindi risentono della
struttura per età del comune (un comune più giovane ha meccanicamente più coppie
giovani); `F4` è invece rapportato alla popolazione giovane ed è il più pulito dei
quattro. Dati 1991-2011, mai in serie con il 2018-2024.

In [47]:
FAMIGLIA = ["F4", "F5", "F6", "F7"]
righe = []
for ind in FAMIGLIA:
    for anno in (1991, 2001, 2011):
        comuni_f = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(anno)
                             & ottomila["indicatore"].eq(ind)]
                    .set_index("territorio")["valore"].dropna())
        confronto = (ottomila[ottomila["territorio"].isin(CONFRONTO)
                              & ottomila["anno"].eq(anno) & ottomila["indicatore"].eq(ind)]
                     .set_index("territorio")["valore"])
        righe.append({"indicatore": ind, "anno": anno,
                      "bagheria": round(comuni_f[BAGHERIA], 1),
                      "percentile_390": round(100 * (comuni_f < comuni_f[BAGHERIA]).mean(), 1),
                      "mediana_gemelle": round(comuni_f[gemelle.index].median(), 1),
                      "palermo": round(confronto[PALERMO], 1),
                      "italia": round(confronto[ITALIA], 1)})
famiglia = (pd.DataFrame(righe)
            .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore"))
famiglia.to_csv(PROCESSED / "genere_famiglia_precoce.csv", index=False)
famiglia.set_index(["indicatore", "anno"]).drop(columns="nome_indicatore")

bagheria  percentile_390  mediana_gemelle  palermo  italia
indicatore anno                                                            
F4         1991       1.3             9.0              2.0      2.1     2.9
           2001       1.7             6.4              2.6      2.6     4.6
           2011       3.6             3.3              4.4      4.0     7.0
F5         1991       1.3            54.4              1.0      1.7     1.0
           2001       0.9            56.9              0.7      0.8     1.0
           2011       0.9            49.0              0.8      0.7     1.0
F6         1991       5.0            80.0              4.8      4.5     5.4
           2001       4.7            84.4              4.2      4.1     5.0
           2011       3.4            81.8              3.6      2.9     3.4
F7         1991      25.6            88.5             24.6     20.9    16.3
           2001      18.4            86.7             17.8     14.3    11.0
           2011      11.7            81.0             11.8      9.3     7.4

**📌 Risultato chiave** — Il canale familiare ha numeri estremi in entrambe le direzioni: **giovani che vivono da soli al 3° percentile siciliano** (F4 = 3.6%, la metà dell'Italia, ultima anche fra le gemelle) e **coppie giovani con figli all'81°** (F7 = 11.7% delle famiglie, contro il 7.4% nazionale) — con la stessa posizione relativa già nel 1991 (9° e 88° percentile). A Bagheria la transizione all'età adulta passa dalla famiglia — formata presto — e quasi mai dall'autonomia abitativa. È il contesto strutturale in cui "casalinga a vent'anni" è un esito di sistema, non una scelta anomala.

> Cautele: `F7` risente della struttura per età (denominatore = tutte le famiglie) e il
> confronto giusto è il percentile e le gemelle, non l'Italia — e fra le gemelle F7 è
> nella norma (mediana 11.8): la formazione familiare precoce è un tratto di tutta la
> fascia costiera comparabile, non un'anomalia di Bagheria; `F4` non ne risente ed è
> al minimo del gruppo di controllo. La traiettoria 1991→2011 (F7 25.6 → 11.7) segue il
> calo generale della fecondità: a cambiare è il livello nazionale, non la posizione
> relativa di Bagheria, che resta nel quinto più alto. Il **motivo** (scelta, vincolo,
> norma) non è nei dati — resta il limite dichiarato del thread. Questi indicatori sono del
> 2011 e descrivono le generazioni precedenti: per le ragazze di oggi lo stato civile per età
> (sezione «Le casalinghe sono coniugate?») esclude il matrimonio precoce come canale.

# **Trend paralleli: il controfattuale, testato sul passato**
La proposta prevede un disegno differenza-nelle-differenze per la valutazione. Il DiD post-intervento oggi non esiste (nessun intervento è partito), ma il
suo prerequisito — **trend paralleli nel pre-periodo** — si testa già: (a) sul tasso di
occupazione femminile 15-24, 2018-2024, contro i tre benchmark (GLM binomiale a link
identità, stessa convenzione LPM del notebook: pendenze in pp/anno); (b) sul lungo
periodo, `L11` di Bagheria contro le gemelle sui tre censimenti; (c) senza modello
binomiale, la differenza di pendenza con Palermo contro la dispersione delle pendenze dei
comuni di taglia simile (sezione «Il KPI si può misurare?»). Sei punti temporali e
conteggi comunali: il test ha la potenza che ha — "non rifiutato" non è "dimostrato".

In [48]:
femmine = giovani[giovani["genere"].eq("F")].assign(
    p=lambda d: d["occupati"] / d["popolazione"], anno_c=lambda d: d["anno"] - 2021)
parallelo = smf.glm("p ~ anno_c * C(nome_territorio, Treatment('Bagheria'))", data=femmine,
                    family=sm.families.Binomial(link=sm.families.links.Identity()),
                    var_weights=femmine["popolazione"]).fit()
intervalli = parallelo.conf_int()

righe = [{"termine": "pendenza Bagheria (pp/anno)",
          "stima": 100 * parallelo.params["anno_c"],
          "CI 95% basso": 100 * intervalli.loc["anno_c", 0],
          "CI 95% alto": 100 * intervalli.loc["anno_c", 1],
          "p": parallelo.pvalues["anno_c"]}]
for nome in BENCHMARK:
    chiave = f"anno_c:C(nome_territorio, Treatment('Bagheria'))[T.{nome}]"
    righe.append({"termine": f"differenza di pendenza: {nome} - Bagheria",
                  "stima": 100 * parallelo.params[chiave],
                  "CI 95% basso": 100 * intervalli.loc[chiave, 0],
                  "CI 95% alto": 100 * intervalli.loc[chiave, 1],
                  "p": parallelo.pvalues[chiave]})
pendenze_f = pd.DataFrame(righe).round({"stima": 2, "CI 95% basso": 2, "CI 95% alto": 2, "p": 3})
pendenze_f.to_csv(PROCESSED / "genere_pretrend.csv", index=False)
print("(a) Tasso di occupazione femminile 15-24, 2018-2024 (2020 mancante alla fonte):")
print(pendenze_f.to_string(index=False))

righe = []
for anno in (1991, 2001, 2011):
    serie = (ottomila[ottomila["livello"].eq("1") & ottomila["anno"].eq(anno)
                      & ottomila["indicatore"].eq("L11")].set_index("territorio")["valore"])
    righe.append({"anno": anno, "bagheria": serie[BAGHERIA],
                  "gemelle_mediana": serie[gemelle.index].median(),
                  "gemelle_q1": serie[gemelle.index].quantile(0.25),
                  "gemelle_q3": serie[gemelle.index].quantile(0.75)})
storico_gemelle = pd.DataFrame(righe).round(1)
storico_gemelle.to_csv(PROCESSED / "genere_pretrend_gemelle.csv", index=False)
print("\n(b) Occupazione femminile 15+ (L11), Bagheria contro le gemelle:")
print(storico_gemelle.to_string(index=False))

(a) Tasso di occupazione femminile 15-24, 2018-2024 (2020 mancante alla fonte):
                                   termine  stima  CI 95% basso  CI 95% alto     p
               pendenza Bagheria (pp/anno)   0.65          0.48         0.81 0.000
differenza di pendenza: Palermo - Bagheria  -0.10         -0.27         0.08 0.291
differenza di pendenza: Sicilia - Bagheria  -0.17         -0.33         0.00 0.054
 differenza di pendenza: Italia - Bagheria  -0.08         -0.25         0.08 0.328



(b) Occupazione femminile 15+ (L11), Bagheria contro le gemelle:
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3
 1991      10.9             10.8         9.5        14.5
 2001      15.1             15.0        14.2        17.9
 2011      18.1             19.8        17.8        22.7


In [49]:
# Lo stesso prerequisito senza il modello binomiale. Il GLM qui sopra tratta ogni annata come
# un campione indipendente, la stessa ipotesi che nella sezione sul KPI non regge quando si
# aggregano anni. Qui la differenza di pendenza fra Palermo e Bagheria si confronta con quanto
# divergono fra loro le pendenze dei comuni di taglia simile (OLS 2018-2024, stessa tavola).
pendenze_390 = tassi_15_24["occupazione"].unstack("anno")[ANNI_SERIE].apply(
    lambda serie: np.polyfit(ANNI_SERIE, serie.to_numpy(), 1)[0], axis=1)
scarto_pa = pendenze_390[PALERMO] - pendenze_390[BAGHERIA]
pretrend_390 = pd.DataFrame([{
    "pendenza Bagheria OLS (pp/anno)": pendenze_390[BAGHERIA],
    "pendenza Palermo OLS (pp/anno)": pendenze_390[PALERMO],
    "differenza Palermo - Bagheria (pp/anno)": scarto_pa,
    "comuni simili": len(simili),
    "deviazione delle pendenze dei comuni simili (pp/anno)":
        (pendenze_390[simili] - pendenze_390[simili].median()).std(),
    "comuni simili più lontani da Palermo di Bagheria (%)":
        100 * ((pendenze_390[simili] - pendenze_390[PALERMO]).abs() >= abs(scarto_pa)).mean(),
}]).round(2).round({"comuni simili più lontani da Palermo di Bagheria (%)": 0})
pretrend_390.to_csv(PROCESSED / "genere_pretrend_390.csv", index=False)
print(pretrend_390.T.to_string(header=False))

pendenza Bagheria OLS (pp/anno)                         0.64
pendenza Palermo OLS (pp/anno)                          0.55
differenza Palermo - Bagheria (pp/anno)                -0.09
comuni simili                                          33.00
deviazione delle pendenze dei comuni simili (pp/anno)   0.19
comuni simili più lontani da Palermo di Bagheria (%)   67.00


**📌 Risultato chiave** — Il prerequisito del disegno di valutazione non è smentito: la pendenza del tasso femminile di Bagheria (+0.65 pp/anno, CI 0.48-0.81) è **indistinguibile da Palermo e Italia** (differenze -0.10 e -0.08, p=0.29 e 0.33; la Sicilia è al margine, -0.17, p=0.054). E sul lungo periodo Bagheria era **identica alle sue gemelle** nel 1991 (10.9 contro 10.8) e nel 2001 (15.1 contro 15.0), poi **se ne stacca nel decennio 2001-2011** (18.1 contro una mediana di 19.8, con 7 gemelle su 10 sopra): lo scarto dalle comparabili è recente, non eterno.

Senza modello binomiale (terza cella): la differenza di pendenza Palermo - Bagheria è
-0.09 pp/anno (OLS), più piccola di quella del 67% dei comuni di taglia simile, le cui
pendenze si disperdono con una deviazione di 0.19 pp/anno. Il confronto contiene le
divergenze reali fra comuni, quindi è indulgente: dice che la distanza da Palermo è
ordinaria, non che le tendenze siano parallele. Il p = 0.29 del GLM tratta le annate come
campioni indipendenti, la stessa ipotesi che sul KPI non regge quando si aggregano anni.

> Per la proposal: il controfattuale dichiarato in anticipo è **Palermo** (trend parallelo
> non rifiutato sul pre-periodo, stessa fonte, lettura annuale), con le gemelle come
> ancora di lungo periodo per i target. Limiti scritti: sei punti e potenza bassa — il
> test non dimostra i paralleli, non li rifiuta; il confronto 2018-2024 con le gemelle è
> nella sezione «Il ponte fra i due censimenti». La divergenza 2001-2011
> dalle gemelle data anche il problema: qualunque spiegazione della talent trap femminile
> deve essere compatibile con un peggioramento **relativo** concentrato in quel decennio.

# **Il ponte fra i due censimenti: dal 2011 al 2024**

Il lungo periodo di questo thread si ferma al 2011 perché 8milaCensus è l'ultimo censimento
decennale. Ma la classe **15 anni e più** esiste anche nel censimento permanente
(`AGE_NOCLASS = Y_GE15`), con gli stessi numeratori e denominatori del codebook: i quattro
indicatori di lavoro si ricalcolano per il 2018-2024, e scaricando i 390 comuni siciliani
anche i percentili regionali.

Serve a tre cose:

1. la posizione di Bagheria fra i 390 comuni arriva al 2024, non più al 2011;
2. il confronto con le dieci gemelle diventa una serie recente e non solo storica —
   è il pre-periodo del DiD, misurato invece che assunto;
3. il salto 2011 → 2018 mette a confronto **due rilevazioni con disegni diversi** —
   censimento decennale universale a questionario contro censimento permanente campionario
   appoggiato ai registri. Se danno lo stesso livello, il livello non è un artefatto del
   disegno della rilevazione.

Il punto 3 non è una validazione esterna: i numeri restano ISTAT in entrambi i casi, e la
coerenza va misurata indicatore per indicatore, non dichiarata. Le due fonti restano due:
mai una linea continua fra 2011 e 2018, sempre uno stacco visibile e la fonte per blocco.

In [50]:
# I quattro indicatori 15+ di fig10 ricostruiti dal censimento permanente. Le formule sono
# quelle del codebook 8milaCensus (data/processed/indicatori.csv), non un'approssimazione:
#   L2  = attive F / residenti F 15+       L11 = occupate F / residenti F 15+
#   L10 = occupati M / residenti M 15+     L7  = in cerca F / attive F 15+
# Codici CUR_ACT_STAT: 1 occupato, 12 in cerca di occupazione, 22 forze di lavoro, 99 totale.
# I1 resta fuori: 8milaCensus lo calcola sulla popolazione 6+, la tavola istruzione del
# permanente parte da 9+ e non ha una classe 15+. Sarebbe un altro indicatore, non un seguito.
RECENTI = ["L2", "L11", "L10", "L7"]


def tassi_15piu(tabella: pd.DataFrame) -> pd.DataFrame:
    """L2/L7/L10/L11 dalla tavola lavoro sulla classe Y_GE15, definizioni 8milaCensus."""
    base = tabella[tabella["eta"].eq("Y_GE15") & tabella["cittadinanza"].eq("TOTAL")
                   & tabella["titolo_studio"].eq("ALL") & tabella["genere"].isin(["M", "F"])]
    c = base.pivot_table(index=["territorio", "anno"], columns=["genere", "condizione"],
                         values="valore", aggfunc="sum")
    return pd.DataFrame({"L2":  100 * c[("F", "22")] / c[("F", "99")],
                         "L11": 100 * c[("F", "1")] / c[("F", "99")],
                         "L10": 100 * c[("M", "1")] / c[("M", "99")],
                         "L7":  100 * c[("F", "12")] / c[("F", "22")]})


sicilia_390 = leggi("censpop_lavoro_15piu_sicilia_long.csv")
assert sicilia_390["territorio"].nunique() == 390, "platea diversa da quella dei percentili 2011"
tassi_390 = tassi_15piu(sicilia_390)
tassi_confronto = tassi_15piu(istr_lav[istr_lav["tavola"].eq("lavoro")])

# --- (a) coerenza fra le due rilevazioni: 2011 decennale contro 2018 permanente --------
val_2011 = (ottomila[ottomila["anno"].eq(2011) & ottomila["indicatore"].isin(RECENTI)
                     & ottomila["territorio"].isin(CONFRONTO)]
            .pivot_table(index="territorio", columns="indicatore", values="valore"))
val_2018 = tassi_confronto.xs(2018, level="anno")

# --- (b) il salto interno al permanente fra 2019 e 2021, sugli stessi indicatori --------
# Il 2020 manca alla fonte su ogni classe che contenga i 15-24: il confronto utile è 2019-2021.
salto = tassi_confronto.xs(2021, level="anno") - tassi_confronto.xs(2019, level="anno")

coerenza = pd.DataFrame([
    {"territorio": NOMI[t], "indicatore": ind,
     "decennale_2011": round(val_2011.loc[t, ind], 1),
     "permanente_2018": round(val_2018.loc[t, ind], 1),
     "scarto_2011_2018": round(val_2018.loc[t, ind] - val_2011.loc[t, ind], 1),
     "salto_2019_2021": round(salto.loc[t, ind], 1)}
    for ind in RECENTI for t in CONFRONTO])
coerenza.to_csv(PROCESSED / "genere_coerenza_fonti.csv", index=False)

print("Due rilevazioni indipendenti sullo stesso indicatore e la stessa fascia (15+):")
print(coerenza.pivot(index="indicatore", columns="territorio")
      [["decennale_2011", "permanente_2018", "scarto_2011_2018", "salto_2019_2021"]]
      .reindex(RECENTI).to_string())

Due rilevazioni indipendenti sullo stesso indicatore e la stessa fascia (15+):
           decennale_2011                        permanente_2018                        scarto_2011_2018                        salto_2019_2021                       
territorio       Bagheria Italia Palermo Sicilia        Bagheria Italia Palermo Sicilia         Bagheria Italia Palermo Sicilia        Bagheria Italia Palermo Sicilia
indicatore                                                                                                                                                            
L2                   28.7   41.8    35.9    33.0            30.4   44.0    37.3    35.9              1.7    2.2     1.4     2.9            -4.7   -1.6    -4.2    -4.3
L11                  18.1   36.1    25.5    24.0            18.8   36.8    25.0    24.8              0.7    0.7    -0.5     0.8             1.2    0.5     1.8     1.0
L10                  43.1   54.8    45.0    46.9            40.4   53.8    43.5    44.

In [51]:
# Percentile di Bagheria fra i 390 comuni, anno per anno, sulla stessa platea del 2011
# (i comuni ai confini 2011: Misiliscemi, nato nel 2021, resta fuori per non cambiare il
# denominatore fra le due epoche). Il percentile è un rango calcolato dentro l'anno, quindi
# assorbe lo scarto di definizione fra le fonti, che sposta tutti i comuni nello stesso verso:
# è il motivo per cui il pannello dei percentili può attraversare il 2011 e i livelli no.
percentili_recenti = pd.DataFrame([
    {"indicatore": ind, "anno": anno,
     "percentile_390": round(100 * (v < v[BAGHERIA]).mean(), 1), "comuni": int(v.notna().sum())}
    for ind in RECENTI
    for anno, serie in tassi_390[ind].groupby("anno")
    for v in [serie.droplevel("anno").dropna()]])

madri_recente = (tassi_confronto.loc[CONFRONTO, RECENTI].round(1).stack()
                 .rename("valore").reset_index().rename(columns={"level_2": "indicatore"})
                 .pivot_table(index=["indicatore", "anno"], columns="territorio", values="valore")
                 [CONFRONTO].rename(columns=NOMI).reset_index()
                 .merge(percentili_recenti, on=["indicatore", "anno"])
                 .merge(indicatori[["indicatore", "nome_indicatore"]], on="indicatore")
                 .assign(fonte="censimento permanente")
                 .sort_values(["indicatore", "anno"], key=lambda c: c.map(
                     {v: i for i, v in enumerate(RECENTI)}).fillna(c) if c.name == "indicatore" else c))
madri_recente.to_csv(PROCESSED / "genere_madri_recente.csv", index=False)

print("Percentile di Bagheria fra i 390 comuni, 2018-2024 (2020 assente alla fonte):")
print(madri_recente.pivot(index="anno", columns="indicatore", values="percentile_390")
      [RECENTI].to_string())
print("\nPer confronto, gli stessi percentili ai tre censimenti (8milaCensus):")
print(gap_madri[gap_madri["indicatore"].isin(RECENTI)]
      .pivot(index="anno", columns="indicatore", values="percentile_390")[RECENTI].to_string())

Percentile di Bagheria fra i 390 comuni, 2018-2024 (2020 assente alla fonte):
indicatore    L2   L11   L10    L7
anno                              
2018        21.5   8.2  15.4  92.6
2019        21.3   9.7  19.2  91.8
2021        17.9  13.3  34.9  95.6
2022        15.1  13.1  35.6  84.4
2023        14.1  12.6  28.5  83.1
2024        16.9  16.9  29.5  73.6

Per confronto, gli stessi percentili ai tre censimenti (8milaCensus):
indicatore    L2   L11   L10    L7
anno                              
1991         7.9  22.1  63.6  45.9
2001        40.8  23.6  54.9  84.4
2011        31.5  12.3  14.6  94.6


In [52]:
# Le dieci gemelle dentro il censimento permanente: la banda interquartile del gruppo
# diventa una serie 2018-2024 invece di tre punti storici. Bagheria viene dal file dei 390,
# le gemelle dal loro raw: stessa formula, stessa fascia, stessa fonte.
lavoro_gemelle = leggi("censpop_lavoro_gemelle_long.csv")
tassi_gemelle = tassi_15piu(lavoro_gemelle[lavoro_gemelle["tavola"].eq("lavoro")])
assert set(tassi_gemelle.index.get_level_values("territorio")) == set(gemelle.index), \
    "il raw delle gemelle non coincide con il gruppo del matching"

gemelle_recente = pd.DataFrame([
    {"anno": anno, "bagheria": tassi_390.loc[(BAGHERIA, anno), "L11"],
     "gemelle_mediana": v.median(), "gemelle_q1": v.quantile(0.25), "gemelle_q3": v.quantile(0.75)}
    for anno, serie in tassi_gemelle["L11"].groupby("anno")
    for v in [serie.droplevel("anno")]]).round(1).assign(fonte="censimento permanente")
gemelle_recente.to_csv(PROCESSED / "genere_pretrend_gemelle_recente.csv", index=False)

scarto = (gemelle_recente["bagheria"] - gemelle_recente["gemelle_mediana"]).round(1)
print("Occupazione femminile 15+ (L11), Bagheria contro le dieci gemelle:")
print(gemelle_recente.assign(scarto=scarto).to_string(index=False))
print(f"\nStorico per confronto (8milaCensus, scarto finale "
      f"{storico_gemelle['bagheria'].iloc[-1] - storico_gemelle['gemelle_mediana'].iloc[-1]:+.1f}):")
print(storico_gemelle.to_string(index=False))

Occupazione femminile 15+ (L11), Bagheria contro le dieci gemelle:
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3                 fonte  scarto
 2018      18.8             21.1        19.1        23.0 censimento permanente    -2.3
 2019      19.4             21.5        19.6        24.2 censimento permanente    -2.1
 2021      20.6             23.1        20.8        25.3 censimento permanente    -2.5
 2022      21.1             23.7        21.3        25.4 censimento permanente    -2.6
 2023      22.2             25.3        22.7        27.0 censimento permanente    -3.1
 2024      23.7             26.0        23.8        27.9 censimento permanente    -2.3

Storico per confronto (8milaCensus, scarto finale -1.7):
 anno  bagheria  gemelle_mediana  gemelle_q1  gemelle_q3
 1991      10.9             10.8         9.5        14.5
 2001      15.1             15.0        14.2        17.9
 2011      18.1             19.8        17.8        22.7


**📌 Risultato chiave** — Il muro si segue fino al **2024**, e **non si è richiuso**.

*(a) Le due rilevazioni concordano — su due indicatori su quattro.* Sul tasso di occupazione
femminile 15+ il censimento decennale 2011 (universale, a questionario) e il censimento
permanente 2018 (campionario, sui registri) danno **18,1 e 18,8** a Bagheria, e lo scarto resta
entro ±1 punto anche su Palermo, Sicilia e Italia: il livello femminile non è un artefatto del
disegno della rilevazione. La stessa verifica **boccia L7 e L2**: dentro il permanente, fra 2019
e 2021, la disoccupazione femminile crolla di **15,5 punti** a Bagheria e di 4,5 in Italia —
è un cambio di misura, non un mercato che guarisce, e in figura va marcato.

*(b) Percentili sui 390 comuni.* L'occupazione femminile scende ancora dopo il 2011
(12° → **8° percentile nel 2018**) e risale solo in parte (**17° nel 2024**). Nel frattempo
l'occupazione **maschile** risale dal 15° al 30° e la **partecipazione femminile continua a
scendere** (32° nel 2011 → **17° nel 2024**). La lettura del 2011 — «un mercato che si è
ristretto per tutti» — non regge più al 2024: gli uomini recuperano, le donne no.

*(c) Le gemelle.* Bagheria sta sotto la mediana del gruppo in **tutti** gli anni 2018-2024, di
2,1-3,1 punti, contro i −1,7 del 2011. Lo stacco aperto nel decennio 2001-2011 non si è chiuso:
tredici anni dopo Bagheria è ancora appoggiata al primo quartile delle sue gemelle. Il
controfattuale del DiD ha adesso un pre-periodo **misurato** e non solo assunto.

# **I claim reggono al 2024? Tre verifiche**

Il ponte qui sopra ha portato *fig10* fino al 2024. Le altre figure non sono tutte
altrettanto fresche, e la freschezza non è l'unico problema: una fotografia del 2011
può essere obsoleta, ma una fotografia del 2024 costruita su **un anno solo** può
essere altrettanto fragile. Tre verifiche, una per figura:

1. **fig04** legge la posizione di Bagheria fra i 390 comuni siciliani al 2011. La
   graduatoria del 2011 predice ancora quella del 2024, o si è rimescolata?
2. **fig07** data la fuga con il profilo per età singola 2021-2024, tre anni di braccio.
   Le classi quinquennali della demografia (2001, 2011, 2018-2024) permettono di
   chiedersi **da quando**.
3. **fig05** è già al 2024, ma è una fotografia: le due classifiche che racconta reggono
   su tutti gli anni disponibili, o le decide l'ultimo?

In [53]:
# ---- (1) fig04: la graduatoria dei 390 comuni sopravvive al cambio di rilevazione? ----
# Spearman e non Pearson perché la domanda è sull'ordine, non sui livelli: i livelli le
# due rilevazioni li misurano in modo diverso (cella sopra), il rango dentro l'anno no.
# Se il rango del 2011 predice quello del 2024, un claim di posizionamento costruito sul
# 2011 non era una scommessa sul passato — era una previsione, e qui si verifica.
from scipy.stats import spearmanr

occ_390 = tassi_390["L11"].unstack("anno")
occ_390.insert(0, 2011, occ_femminile.set_index("territorio")["occupazione_femminile_2011"])
assert occ_390.notna().all().all(), "la platea 2011 e quella 2018-2024 non coincidono"
ANNI_390 = list(occ_390.columns)


def percentile_interno(s: pd.Series) -> pd.Series:
    """Quota di comuni strettamente sotto: la stessa convenzione di `percentili_recenti`."""
    return (100 * (s.rank(method="min") - 1) / len(s)).round(1)


def quintile_basso(anno: int) -> set:
    return set(occ_390.index[occ_390[anno].rank(pct=True) <= 0.2])


def rho(a: int, b: int) -> float:
    return round(float(spearmanr(occ_390[a], occ_390[b]).statistic), 3)


distribuzione_390 = pd.DataFrame([
    {"anno": anno,
     "fonte": "8milaCensus" if anno == 2011 else "censimento permanente",
     "bagheria": round(occ_390.loc[BAGHERIA, anno], 1),
     "percentile": percentile_interno(occ_390[anno])[BAGHERIA],
     "comuni_sotto": int((occ_390[anno] < occ_390.loc[BAGHERIA, anno]).sum()),
     "mediana": round(occ_390[anno].median(), 1),
     "q1": round(occ_390[anno].quantile(0.25), 1),
     "q3": round(occ_390[anno].quantile(0.75), 1),
     "minimo": round(occ_390[anno].min(), 1),
     "massimo": round(occ_390[anno].max(), 1),
     "rho_vs_2011": rho(2011, anno),
     "rho_vs_2024": rho(anno, 2024),
     "quintile_basso_ancora_tale_nel_2024_pct":
         round(100 * len(quintile_basso(anno) & quintile_basso(2024)) / len(quintile_basso(anno)), 1)}
    for anno in ANNI_390])
distribuzione_390.to_csv(PROCESSED / "genere_distribuzione_390.csv", index=False)

# Le due convenzioni di percentile (questa e quella di `percentili_recenti`) devono
# coincidere: se un giorno divergono, fig04 e fig10 raccontano ranghi diversi.
atteso = percentili_recenti.set_index(["indicatore", "anno"]).loc[("L11", 2024), "percentile_390"]
assert distribuzione_390.set_index("anno").loc[2024, "percentile"] == atteso, "due percentili diversi"

# Tavola per la mappa: gli stessi comuni ai due estremi della serie. `ruolo` marca chi va
# etichettato — Bagheria, i cinque vicini e gli estremi regionali di CIASCUN anno, che nel
# 2024 non sono gli stessi comuni del 2011.
ruolo = pd.Series("", index=occ_390.index)
for anno in (2011, 2024):
    ruolo[occ_390[anno].idxmin()] = f"minimo {anno}"
    ruolo[occ_390[anno].idxmax()] = f"massimo {anno}"
ruolo[CODICI_VICINI] = "vicino"
ruolo[BAGHERIA] = "Bagheria"

# Il denominatore del tasso 2024: le residenti di 15 anni e più. Serve a fig04c, che nomina
# gli estremi della graduatoria — e in coda ci stanno sia Adrano (una città) sia Bompensiere
# (poche centinaia di abitanti). Senza il denominatore la figura li mette allo stesso peso
# visivo e invita a leggere come un dato quello che nei comuni minuscoli è rumore.
base_390 = sicilia_390[sicilia_390["eta"].eq("Y_GE15") & sicilia_390["cittadinanza"].eq("TOTAL")
                       & sicilia_390["titolo_studio"].eq("ALL") & sicilia_390["genere"].eq("F")
                       & sicilia_390["anno"].eq(2024)]
conteggi_390 = base_390.pivot_table(index="territorio", columns="condizione",
                                    values="valore", aggfunc="sum")
donne_15piu = conteggi_390["99"]
# Il controllo che conta: la platea esportata deve essere QUELLA del tasso in figura, non
# un'altra popolazione con lo stesso nome.
assert (100 * conteggi_390["1"] / donne_15piu - occ_390[2024]).abs().max() < 1e-6, \
    "il denominatore esportato non ricostruisce il tasso"

mappa_2011_2024 = (pd.DataFrame(index=occ_390.index)
    .assign(nome_comune=centroidi.set_index("territorio")["nome_comune"],
            x=xy["x"], y=xy["y"], distanza_km=distanza_km.round(1),
            occ_2011=occ_390[2011].round(1), occ_2024=occ_390[2024].round(1),
            pct_2011=percentile_interno(occ_390[2011]),
            pct_2024=percentile_interno(occ_390[2024]),
            donne_15piu_2024=donne_15piu.round().astype(int),
            # Il rango esplicito: fig04c lo stampa accanto a ogni riga, perché le righe sono
            # ordinate per valore ma spaziate uniformemente e la distanza fra due righe
            # vicine non dice quanti comuni ci stanno in mezzo.
            rango_2024=occ_390[2024].rank(ascending=False, method="min").astype(int),
            ruolo=ruolo)
    .reset_index())
mappa_2011_2024.to_csv(PROCESSED / "genere_mappa_2011_2024.csv", index=False)

print("Occupazione femminile 15+, distribuzione dei 390 comuni siciliani:")
print(distribuzione_390.set_index("anno").to_string())
print(f"\nLa graduatoria del 2011 predice quella del 2024: rho = {rho(2011, 2024)}",
      f"su {len(occ_390)} comuni, attraverso 13 anni e due rilevazioni diverse.")
print(f"Dentro il solo permanente, 2018 vs 2024: rho = {rho(2018, 2024)}.")
print(f"Del quintile più basso del 2011, il "
      f"{distribuzione_390.set_index('anno').loc[2011, 'quintile_basso_ancora_tale_nel_2024_pct']:.0f}%"
      f" è ancora nel quintile più basso nel 2024 — Bagheria compresa: "
      f"{BAGHERIA in quintile_basso(2011) and BAGHERIA in quintile_basso(2024)}")
print("\nEstremi regionali e vicini etichettati sulla mappa:")
print(mappa_2011_2024[mappa_2011_2024["ruolo"].ne("")]
      [["nome_comune", "ruolo", "occ_2011", "occ_2024", "pct_2011", "pct_2024"]]
      .sort_values("occ_2024").to_string(index=False))

Occupazione femminile 15+, distribuzione dei 390 comuni siciliani:
                      fonte  bagheria  percentile  comuni_sotto  mediana    q1    q3  minimo  massimo  rho_vs_2011  rho_vs_2024  quintile_basso_ancora_tale_nel_2024_pct
anno                                                                                                                                                                    
2011            8milaCensus      18.1        12.3            48     23.6  19.7  27.4    13.0     39.4        1.000        0.848                                     74.0
2018  censimento permanente      18.8         8.2            32     24.3  20.9  27.4    14.8     36.0        0.883        0.919                                     83.3
2019  censimento permanente      19.4         9.7            38     25.0  21.6  28.0    14.8     35.9        0.879        0.925                                     83.3
2021  censimento permanente      20.6        13.3            52     25.7  22.7  28.7    

In [54]:
# ---- (1b) fig06b: la stessa distribuzione, una cresta per annata ----------------------
# Le creste della fig06b. La densità la calcola Python e non R: la convenzione del repo è
# che R legga tabelle già pronte, e su questa macchina ggridges non è installabile.
# Solo censimento permanente: il 2011 è un'altra rilevazione, e una cresta appaiata alle
# altre inviterebbe a leggere lo scarto di definizione come movimento. Il confronto fra
# le due epoche ha la sua figura (fig04), dove lo stacco è dichiarato in caption.
ANNI_CRESTE = [a for a in ANNI_390 if a != 2011]

# Una banda unica per tutte le annate, regola di Silverman sul pool. Applicata anno per
# anno ogni cresta avrebbe il suo lisciamento, e parte delle differenze di forma sarebbe
# l'effetto della banda invece che del dato: qui le creste si confrontano fra loro.
pool = occ_390[ANNI_CRESTE].to_numpy().ravel()
banda = 0.9 * min(pool.std(ddof=1), stats.iqr(pool) / 1.349) * len(pool) ** -0.2
griglia = np.linspace(pool.min() - 3 * banda, pool.max() + 3 * banda, 512)

creste_390 = pd.concat(
    pd.DataFrame({"anno": anno, "x": griglia.round(3),
                  "densita": stats.gaussian_kde(
                      occ_390[anno], bw_method=banda / occ_390[anno].std(ddof=1))(griglia)})
    for anno in ANNI_CRESTE)
creste_390.to_csv(PROCESSED / "genere_creste_390.csv", index=False)

# L'area sotto ogni cresta vale 1 per costruzione: se una si scosta, la griglia sta
# tagliando una coda e la figura confronterebbe altezze non confrontabili.
aree = {anno: float(np.trapezoid(g["densita"], g["x"])) for anno, g in creste_390.groupby("anno")}
assert all(0.99 <= a <= 1.01 for a in aree.values()), f"code fuori griglia: {aree}"

# Il claim della figura in numeri: la distribuzione si sposta, la posizione di Bagheria no.
# L'assert è sul titolo: se un'annata uscisse dal quinto più basso, il titolo mentirebbe.
recenti = distribuzione_390[distribuzione_390["anno"].ne(2011)].set_index("anno")
assert recenti["percentile"].lt(20).all(), "Bagheria esce dal quinto più basso: titolo da riscrivere"

print(f"Creste per fig06b: {len(ANNI_CRESTE)} annate, banda comune {banda:.2f} punti.")
print(f"Mediana regionale {recenti['mediana'].iloc[0]:.1f}% -> {recenti['mediana'].iloc[-1]:.1f}%, "
      f"Bagheria {recenti['bagheria'].iloc[0]:.1f}% -> {recenti['bagheria'].iloc[-1]:.1f}%: "
      f"guadagna più della mediana e resta fra il {recenti['percentile'].min():.0f}° e il "
      f"{recenti['percentile'].max():.0f}° percentile.")
print(f"Nel 2024 Bagheria ({recenti['bagheria'].iloc[-1]:.1f}%) non ha ancora raggiunto la "
      f"mediana siciliana del {recenti.index[0]} ({recenti['mediana'].iloc[0]:.1f}%).")

Creste per fig06b: 6 annate, banda comune 0.88 punti.
Mediana regionale 24.3% -> 28.3%, Bagheria 18.8% -> 23.7%: guadagna più della mediana e resta fra il 8° e il 17° percentile.
Nel 2024 Bagheria (23.7%) non ha ancora raggiunto la mediana siciliana del 2018 (24.3%).


In [55]:
# ---- (2) fig07: da quando si perde chi ------------------------------------------------
# Le classi quinquennali della demografia coprono 2001, 2011 e 2018-2024: una coorte si
# segue spostandosi di una classe ogni cinque anni (chi ha 15-19 anni in t ne ha 25-29 in
# t+10). Il profilo per età singola di `genere_ritenzione_eta.csv` dice DOVE si perde chi;
# questo dice DA QUANDO, che tre anni di braccio non possono dire.
CLASSI_QUINQ = ["Y10-14", "Y15-19", "Y20-24", "Y25-29", "Y30-34", "Y35-39", "Y40-44", "Y45-49"]
PERIODI = [(2001, 2011), (2011, 2021), (2018, 2023), (2019, 2024)]

classi = leggi("censpop_demografia_classi_long.csv")
conteggi_classi = (classi[classi["genere"].isin(["M", "F"]) & classi["eta"].isin(CLASSI_QUINQ)]
                   .pivot_table(index=["territorio", "genere", "eta"], columns="anno", values="valore"))


def coorti_quinquennali(anno_da: int, anno_a: int) -> list[dict]:
    passi, resto = divmod(anno_a - anno_da, 5)
    assert resto == 0, "le classi sono quinquennali: il passo dev'essere multiplo di cinque"
    righe = []
    for t in CONFRONTO:
        for g in ("F", "M"):
            for i, eta_da in enumerate(CLASSI_QUINQ[:-passi]):
                eta_a = CLASSI_QUINQ[i + passi]
                n_da = conteggi_classi.loc[(t, g, eta_da), anno_da]
                n_a = conteggi_classi.loc[(t, g, eta_a), anno_a]
                righe.append({
                    "territorio": t, "nome_territorio": NOMI[t], "genere": g,
                    "anno_da": anno_da, "anno_a": anno_a, "anni": anno_a - anno_da,
                    "eta_da": eta_da, "eta_a": eta_a,
                    "n_da": int(n_da), "n_a": int(n_a),
                    "ritenzione_pct": round(100 * n_a / n_da, 1),
                    # 2011 è decennale e 2021 permanente: quel rapporto ha una gamba per
                    # fonte e va marcato. La distorsione nota va nel verso prudente: il
                    # censimento 2011 contò meno dell'anagrafe, quindi sta al denominatore
                    # del decennio che crolla e al numeratore di quello che tiene — il
                    # divario fra i due decenni è una stima per difetto in entrambe le gambe.
                    "fonti_diverse": anno_da < 2018 <= anno_a})
    return righe


ritenzione_decennale = pd.DataFrame([r for da, a in PERIODI for r in coorti_quinquennali(da, a)])
ritenzione_decennale.to_csv(PROCESSED / "genere_ritenzione_decennale.csv", index=False)

COORTE = "Y15-19"
decenni = ritenzione_decennale[ritenzione_decennale["eta_da"].eq(COORTE)]
for g, etichetta in (("F", "femmine"), ("M", "maschi")):
    print(f"Ritenzione della coorte {COORTE} — {etichetta} (100 = la coorte si conserva):")
    print(decenni[decenni["genere"].eq(g)]
          .assign(periodo=lambda d: d["anno_da"].astype(str) + "-" + d["anno_a"].astype(str))
          .pivot(index="periodo", columns="nome_territorio", values="ritenzione_pct")
          [ORDINE].to_string(), "\n")

ribaltamento = (decenni[decenni["anno_da"].isin([2001, 2011]) & decenni["territorio"].eq(BAGHERIA)]
                .pivot(index="genere", columns="anno_da", values="ritenzione_pct")
                .assign(scarto=lambda d: (d[2011] - d[2001]).round(1)))
print("Bagheria, scarto fra i due decenni sulla stessa coorte:")
print(ribaltamento.to_string())

# Il controllo sulla finestra del profilo per età singola: le coorti che escono dal percorso
# formativo, seguite per cinque anni dentro il solo censimento permanente. Entrambi i generi,
# perché il claim da verificare è di genere.
uscita = ritenzione_decennale[ritenzione_decennale["eta_da"].isin(["Y20-24", "Y25-29"])
                              & ritenzione_decennale["anno_da"].ge(2018)]
print("\nUscita dal percorso formativo, coorti seguite per cinque anni (censimento permanente):")
print(uscita.assign(periodo=lambda d: d["anno_da"].astype(str) + "-" + d["anno_a"].astype(str))
      .pivot_table(index=["eta_da", "periodo"], columns=["nome_territorio", "genere"],
                   values="ritenzione_pct")
      .reindex(columns=pd.MultiIndex.from_product([ORDINE, ["F", "M"]])).to_string())

Ritenzione della coorte Y15-19 — femmine (100 = la coorte si conserva):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
periodo                                            
2001-2011           102.9     87.6     97.8   113.1
2011-2021            88.6     90.2     92.1   104.4
2018-2023            97.9     95.8     98.1   101.7
2019-2024           100.1     97.0     99.4   102.2 

Ritenzione della coorte Y15-19 — maschi (100 = la coorte si conserva):
nome_territorio  Bagheria  Palermo  Sicilia  Italia
periodo                                            
2001-2011            98.4     86.3     95.1   108.1
2011-2021            83.9     86.3     91.0   104.8
2018-2023            96.1     94.9     97.4   103.6
2019-2024            98.1     97.1    100.2   105.5 

Bagheria, scarto fra i due decenni sulla stessa coorte:
anno_da   2001  2011  scarto
genere                      
F        102.9  88.6   -14.3
M         98.4  83.9   -14.5

Uscita dal percorso formativo, coorti seguite per cinque

In [56]:
# ---- (3) fig05: la forbice è un anno o una tendenza? ----------------------------------
# Stesse formule di `genere_forbice.csv` (istruzione 9-24 «almeno il diploma», occupazione
# 15-24) ripetute su ogni anno disponibile, vicinato compreso. Ricalcolate qui e non
# riprese dalle tabelle a monte, che sono già filtrate sull'anno comune.
def serie_forbice(lavoro_t: pd.DataFrame, istruzione_t: pd.DataFrame, territorio: str) -> pd.DataFrame:
    occ = (lavoro_t[lavoro_t["eta"].eq("Y15-24") & lavoro_t["genere"].isin(["M", "F"])]
           .pivot_table(index="anno", columns=["condizione", "genere"], values="valore", aggfunc="sum"))
    # Il 2020 manca alla fonte su ogni classe che contenga i 15-24: niente riga, nessuna stima.
    occ = occ[occ[("99", "F")].gt(0) & occ[("99", "M")].gt(0)]
    istr = (istruzione_t[istruzione_t["eta"].eq("Y9-24") & istruzione_t["genere"].isin(["M", "F"])]
            .pivot_table(index="anno", columns=["titolo_studio", "genere"], values="valore", aggfunc="sum")
            .reindex(occ.index))
    tasso = lambda g: 100 * occ[("1", g)] / occ[("99", g)]
    diploma = lambda g: 100 * sum(istr[(k, g)] for k in ALMENO_DIPLOMA) / istr[("ALL", g)]
    return pd.DataFrame({
        "tasso_occupazione_F": tasso("F"), "tasso_occupazione_M": tasso("M"),
        "rapporto_M_F_occupazione": tasso("M") / tasso("F"),
        "almeno_diploma_F": diploma("F"), "almeno_diploma_M": diploma("M"),
        "vantaggio_istruzione_F_pp": diploma("F") - diploma("M"),
    }).assign(territorio=territorio,
              nome_territorio=NOMI.get(territorio, NOME_VICINATO)).round(2)


lavoro_confronto = istr_lav[istr_lav["tavola"].eq("lavoro")]
istruzione_tavola = istr_lav[istr_lav["tavola"].eq("istruzione")]
forbice_serie = pd.concat(
    [serie_forbice(lavoro_confronto[lavoro_confronto["territorio"].eq(t)],
                   istruzione_tavola[istruzione_tavola["territorio"].eq(t)], t) for t in CONFRONTO]
    + [serie_forbice(lavoro_vicini, istruzione_vicini, CODICE_VICINATO)]).reset_index()
forbice_serie.to_csv(PROCESSED / "genere_forbice_serie.csv", index=False)

# La serie deve riprodurre esattamente la fotografia che fig05 già disegna.
for colonna in ("rapporto_M_F_occupazione", "vantaggio_istruzione_F_pp", "tasso_occupazione_F"):
    atteso_fig = forbice.loc["Bagheria", colonna]
    ottenuto = forbice_serie.set_index(["territorio", "anno"]).loc[(BAGHERIA, anno_comune), colonna]
    assert abs(ottenuto - atteso_fig) < 0.02, f"{colonna}: serie {ottenuto} contro fotografia {atteso_fig}"

ORDINE_SERIE = ["Bagheria", NOME_VICINATO, "Palermo", "Sicilia", "Italia"]
for colonna, verso, titolo in (
        ("tasso_occupazione_F", "min", "tasso di occupazione femminile 15-24 (%)"),
        ("rapporto_M_F_occupazione", "max", "rapporto M/F sull'occupazione 15-24"),
        ("vantaggio_istruzione_F_pp", "max", "vantaggio educativo femminile 9-24 (punti)")):
    tabella = forbice_serie.pivot_table(index="anno", columns="nome_territorio", values=colonna)[ORDINE_SERIE]
    testa = tabella.idxmin(axis=1) if verso == "min" else tabella.idxmax(axis=1)
    print(f"{titolo} — in testa alla classifica «sbagliata»: "
          f"Bagheria in {(testa == 'Bagheria').sum()} anni su {len(testa)}")
    print(tabella.to_string())
    print(f"   per anno: {testa.to_dict()}\n")

tasso di occupazione femminile 15-24 (%) — in testa alla classifica «sbagliata»: Bagheria in 6 anni su 6
nome_territorio  Bagheria  vicinato (5 comuni)  Palermo  Sicilia  Italia
anno                                                                    
2018                 4.68                 5.94     6.34     7.52   14.07
2019                 5.16                 6.54     6.94     7.96   14.48
2021                 6.17                 7.46     7.53     8.42   15.00
2022                 7.69                 7.87     8.63     9.34   16.40
2023                 8.02                 8.09     9.21     9.94   16.90
2024                 8.19                 8.90     9.59    10.41   17.27
   per anno: {2018: 'Bagheria', 2019: 'Bagheria', 2021: 'Bagheria', 2022: 'Bagheria', 2023: 'Bagheria', 2024: 'Bagheria'}

rapporto M/F sull'occupazione 15-24 — in testa alla classifica «sbagliata»: Bagheria in 4 anni su 6
nome_territorio  Bagheria  vicinato (5 comuni)  Palermo  Sicilia  Italia
anno           

**📌 Risultato chiave** — Le tre verifiche danno tre esiti diversi, e vale la pena
tenerli distinti.

**fig04 — il claim del 2011 era una previsione, e ha tenuto.** Sui 390 comuni la
graduatoria del 2011 predice quella del 2024 con **ρ di Spearman 0.848**, attraverso
tredici anni e due disegni di rilevazione; dentro il solo censimento permanente
(2018 vs 2024) ρ sale a 0.919. Il **74%** dei comuni nel quintile più basso del 2011 è
ancora lì nel 2024, Bagheria compresa. Il livello però va aggiornato: Bagheria passa da
18.1% (12° percentile) a **23.7% (17° percentile)**, mentre la mediana regionale sale da
23.6% a 28.3%. Detta così: **nel 2024 Bagheria arriva dove stava la mediana siciliana nel
2011**, e i cinque comuni vicini restano tutti sotto la mediana anche adesso.

**fig07 — la fuga è databile, e cade nel decennio successivo al muro sul lavoro.** Sulla
coorte 15-19 anni seguita per dieci anni, Bagheria passa da **102.9% (2001-2011) a 88.6%
(2011-2021)** sulle femmine e da 98.4% a **83.9%** sui maschi: un ribaltamento di circa
quattordici punti. Nel primo decennio Bagheria *guadagnava* coorti mentre Palermo ne
perdeva il 13%; nel secondo ne perde più di Palermo sui maschi. I controlli interni al
permanente (2018-2023 e 2019-2024) confermano che la perdita è **attuale** e sta dove il
profilo per età singola la trova, all'uscita dal percorso formativo: transizione
20-24 → 25-29, femmine 92.8% e 93.3% contro il 102% italiano. **Non la confermano come
femminile.** Sulla stessa transizione i maschi perdono di più (89.2% e 91.8%), e le
femmine di Bagheria stanno al livello di Palermo (90.9% e 92.9%) e della Sicilia (93.4% e
94.4%). Sulla coorte successiva, 25-29 → 30-34, i generi si equivalgono nel 2018-2023
(91.9% contro 91.7%) e divergono di 1.8 punti nel 2019-2024 (93.5% contro 95.3%). Dove lo
scarto di genere c'è, viene dai **maschi**, che a Bagheria restano più che a Palermo e in
Sicilia; vale anche per il triennio 2021-2024 (coorte 25-29: maschi 101.2 contro 96.9 di
Palermo e 97.6 della Sicilia, femmine 96.3 contro 98.8 e 97.6). Regge la perdita a 20-29
anni, 9-12 punti sotto l'Italia per entrambi i generi; una perdita *specificamente*
femminile dopo i 25 anni resta la lettura di un solo triennio.

**fig05 — qui il problema non è l'età del dato, è che poggia su un anno solo.** Delle due
classifiche che la figura racconta, una regge e una no. Il **tasso di occupazione
femminile 15-24 di Bagheria è il minimo del panel in ognuno dei sei anni
disponibili** (4.7% →
8.2%), e il vantaggio educativo femminile cresce da 3.1 a 4.2 punti mentre nel vicinato
crolla da 2.9 a 0.5: la forbice si allarga davvero. Il **rapporto M/F sull'occupazione**
invece oscilla — 2.47 nel 2018, **1.89 nel 2023** (il migliore del gruppo locale, meglio
di vicinato e Sicilia), 2.01 nel 2024: in testa alla classifica «sbagliata»
in quattro anni su sei. «Prima in entrambe le classifiche» è vero nel 2024 per tre
centesimi e non lo era né nel 2022 né nel 2023: la figura va appoggiata sul livello
femminile e sulla forbice, non sul rapporto.

# **Robustezza: le domande di approfondimento**
Verifiche aggiunte il 23 settembre 2026, ciascuna per una domanda che la giuria può fare e a
cui le sezioni precedenti non rispondevano con un numero. La tavola lavoro 15-24 dei 390
comuni e i comuni di taglia simile vengono dalla sezione «Il KPI si può misurare?».

### Bagheria è anomala fra i comuni siciliani?
Il panel a quattro territori dice che Bagheria è ultima fra Bagheria, Palermo, Sicilia e
Italia. Qui la stessa misura (occupazione e casalinghe, ragazze 15-24) su tutti i 390 comuni,
e dentro il gruppo dei comuni di taglia simile con Bagheria. È un confronto descrittivo:
rango e quartili, nessun modello di varianza.

In [57]:
righe = []
for colonna, nome in [("occupazione", "tasso di occupazione F 15-24"),
                      ("casalinghe_pct", "quota casalinghe F 15-24")]:
    for anno in (2018, 2021, 2024):
        valori = tassi_15_24[colonna].xs(anno, level="anno")
        valore_b = valori[BAGHERIA]
        gruppo = valori[simili.append(pd.Index([BAGHERIA]))]
        righe.append({"indicatore": nome, "anno": anno, "Bagheria (%)": valore_b,
                      "comuni": len(valori), "rango dal basso": int((valori < valore_b).sum() + 1),
                      "comuni sotto Bagheria (%)": 100 * (valori < valore_b).mean(),
                      "mediana (%)": valori.median(), "primo quartile (%)": valori.quantile(0.25),
                      "terzo quartile (%)": valori.quantile(0.75),
                      "comuni simili con Bagheria": len(gruppo),
                      "rango dal basso fra i simili": int((gruppo < valore_b).sum() + 1),
                      "mediana dei simili (%)": gruppo.median()})
rango_390 = pd.DataFrame(righe).round(1)
rango_390.to_csv(PROCESSED / "genere_rango_390_15_24.csv", index=False)
rango_390

,indicatore,anno,Bagheria (%),comuni,rango dal basso,comuni sotto Bagheria (%),mediana (%),primo quartile (%),terzo quartile (%),comuni simili con Bagheria,rango dal basso fra i simili,mediana dei simili (%)
0,tasso di occupazione F 15-24,2018,4.7,390,60,15.1,7.1,5.3,9.1,34,2,7.5
1,tasso di occupazione F 15-24,2021,6.2,390,108,27.4,7.8,5.9,9.7,34,3,8.7
2,tasso di occupazione F 15-24,2024,8.2,390,112,28.5,9.7,7.8,12.0,34,2,10.6
3,quota casalinghe F 15-24,2018,12.4,390,339,86.7,8.2,6.1,10.8,34,25,10.0
4,quota casalinghe F 15-24,2021,14.8,390,337,86.2,10.3,8.0,12.8,34,27,12.5
5,quota casalinghe F 15-24,2024,13.4,390,358,91.5,8.2,6.4,10.7,34,29,9.5


**📌 Risultato chiave**: sull'occupazione femminile 15-24 Bagheria non è un'eccezione
siciliana: nel 2024 è 112ª dal basso su 390 (il 28.5% dei comuni sta sotto), contro una
mediana regionale del 9.7%. Lo è fra i comuni della sua taglia: 2ª dal basso su 34 (mediana
10.6%), e lo era anche nel 2018 e nel 2021 (2ª e 3ª). Sulle casalinghe sta nella coda alta in
qualunque confronto: 358ª dal basso su 390 nel 2024 (sopra il 91.5% dei comuni), 29ª su 34
fra i simili.

> Per la proposta: «il tasso più basso» vale nel panel a quattro territori e fra i comuni
> della sua taglia, non fra tutti i comuni siciliani, dove molti comuni piccoli stanno sotto.
> La quota di casalinghe è alta in ogni confronto.

### Le casalinghe sono un conteggio o una stima?
La cifra di testa del focus femminile è la quota di ragazze 15-24 che il censimento classifica
come casalinghe. Prima di costruirci un KPI conviene sapere come nasce. Un conteggio di persone
è un numero intero; una stima ponderata o imputata di solito no. Qui la quota di celle intere,
per condizione e anno, sui 390 comuni (ragazze e ragazzi 15-24, tavola lavoro).

In [58]:
celle = sic_15_24[sic_15_24["genere"].isin(["M", "F"])].assign(
    intera=lambda d: np.isclose(d["valore"], d["valore"].round(), atol=1e-6))
interi = (celle.groupby(["anno", "condizione"])["intera"].mean().mul(100).round(1)
          .rename("celle_intere_pct").reset_index()
          .assign(etichetta=lambda d: d["condizione"].map(
              codici[codici["dimensione"].eq("condizione")].set_index("codice")["etichetta"])))
assert interi["etichetta"].notna().all()
interi.to_csv(PROCESSED / "genere_interi_condizione.csv", index=False)
interi.pivot_table(index="anno", columns="etichetta", values="celle_intere_pct")

etichetta,casalinga/o,forze di lavoro,in altra condizione,in cerca di occupazione,non forze di lavoro,occupato,percettore/rice di una o più pensioni per effetto di attività lavorativa precedente o di redditi da capitale,studente/ssa,totale
anno,,,,,,,,,
2018,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2019,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
2021,0.4,0.4,0.0,0.1,1.0,100.0,0.0,0.5,100.0
2022,0.1,0.4,0.0,0.1,1.3,100.0,0.0,0.8,100.0
2023,0.1,0.4,0.1,0.0,1.4,100.0,0.0,0.5,100.0
2024,0.0,0.5,0.3,0.4,1.3,100.0,0.0,0.3,100.0


**📌 Risultato chiave**: nel 2018 e nel 2019 tutte le celle sono intere. Dal 2021 restano
intere solo **occupati e totale**; in cerca, casalinghe, studenti, pensionati, altra condizione
e non forze di lavoro non lo sono quasi mai (a Bagheria le casalinghe 15-24 del 2024 sono
386.84, arrotondate a 387). Dal 2021 l'occupazione è un conteggio, le condizioni non
professionali sono **stime**: la rottura di misura fra 2019 e 2021 è questo passaggio, visto
dai dati.

ISTAT lo dice nei metadati del censimento 2021: prima si stima se una persona è occupata (sì o
no), poi, per chi non lo è, le probabilità delle altre condizioni; il numero comunale è la
somma di quelle probabilità, fra 0 e 1 per persona, e l'errore standard non è calcolato
(metadati ESMS del censimento 2021 per Eurostat). Il modello è un logit multinomiale stimato
sulle risposte del campione censuario, con covariate amministrative (età, cittadinanza,
istruzione, segnali di lavoro, pensione, iscrizione a corsi, redditi) e le stime comunali
precedenti; «casalinga» sta nella stessa categoria del modello di «altra condizione» (Chianella,
Ciccaglioni, Ercolani, RIEDS 2024). Nessuna covariata misura il lavoro domestico.

> Per la proposta:
>
> 1. «si dichiarano casalinghe» è inesatto: dal 2021 la condizione è attribuita da una stima
>    del censimento, non dichiarata da ciascuna ragazza. Si scrive «risultano casalinghe» o
>    «il censimento classifica come casalinghe»;
> 2. la quota di casalinghe è una stima, e la tavola comunale non ne riporta
>    l'errore: oscilla più di un campione (sezione «Il KPI si può misurare?») e può muoversi
>    per ragioni di metodo. Come KPI regge solo dentro la definizione 2021+, accompagnata
>    dal tasso di occupazione, che è un conteggio;
> 3. lo stesso vale per tutte le componenti del «fuori da lavoro e studio» diverse dagli
>    occupati: la platea di 1.121 inattivi non studenti è una stima, non un conteggio.

### Quante diplomate lavorano? Il tetto dei margini
L'incrocio titolo di studio × condizione professionale non esiste a livello comunale (sezione
«Verifica di fattibilità»). I margini però vincolano la risposta (limiti di Fréchet): se tutte
le occupate fossero diplomate, lavorerebbe al massimo min(occupate, diplomate) / diplomate; il
minimo è max(0, occupate + diplomate − residenti) / diplomate. Vale solo se diplomate e
occupate sono sottoinsiemi della stessa popolazione: il totale 15-24 della tavola lavoro deve
coincidere con la somma delle età singole usata come denominatore (asserito qui sotto), e i
diplomati della tavola 9-24 hanno tutti almeno 15 anni. Sono limiti, non stime.

In [59]:
lavoro_15_24 = istr_lav[istr_lav["tavola"].eq("lavoro") & istr_lav["eta"].eq("Y15-24")
                        & istr_lav["cittadinanza"].eq("TOTAL") & istr_lav["titolo_studio"].eq("ALL")
                        & istr_lav["condizione"].eq("99") & istr_lav["genere"].isin(["M", "F"])]
denominatori = per_1000.merge(
    lavoro_15_24.rename(columns={"valore": "residenti_tavola_lavoro"})
    [["territorio", "anno", "genere", "residenti_tavola_lavoro"]],
    on=["territorio", "anno", "genere"])
assert len(denominatori) == 32, len(denominatori)  # 4 territori x 2021-2024 x 2 generi
assert (denominatori["pop_15_24"] - denominatori["residenti_tavola_lavoro"]).abs().max() < 0.5
frechet = denominatori.assign(
    minimo_pct=lambda d: 100 * np.maximum(0, d["occupati"] + d["diplomati"] - d["pop_15_24"]) / d["diplomati"],
    tetto_pct=lambda d: 100 * np.minimum(d["occupati"], d["diplomati"]) / d["diplomati"])
frechet = frechet[["territorio", "nome_territorio", "anno", "genere", "pop_15_24", "diplomati",
                   "occupati", "minimo_pct", "tetto_pct"]].round(1)
frechet.to_csv(PROCESSED / "genere_frechet.csv", index=False)
frechet[frechet["anno"].eq(2024)].pivot_table(index="nome_territorio", columns="genere",
                                               values="tetto_pct").loc[ORDINE]

genere,F,M
nome_territorio,,
Bagheria,16.1,35.7
Palermo,20.3,37.2
Sicilia,20.4,43.9
Italia,32.3,54.4


**📌 Risultato chiave**: nel 2024 lavora al massimo il 16.1% delle ragazze 15-24 di
Bagheria con almeno il diploma (236 occupate su 1.469 diplomate), contro un tetto del 35.7%
per i ragazzi; il minimo, per entrambi, è zero. Il tetto femminile di Bagheria è il più basso
dei quattro territori (Palermo 20.3%, Sicilia 20.4%, Italia 32.3%). Non dice quante diplomate
lavorino: dice che non sono più di una su sei.

### Il NEET del bando, a scala regionale
Il bando parla di NEET 15-34. A livello comunale non esiste; la rilevazione sulle forze di
lavoro (RCFL) lo pubblica per regione. È una **fonte diversa**: campionaria, regionale, con la
definizione europea (fuori da occupazione, istruzione e formazione, anche non formale). Serve
come riferimento per la cifra del bando e, sulla fascia 15-24, come controllo esterno del proxy
censuario «fuori da lavoro e studio» della Sicilia. Mai in serie con il dato comunale.

In [60]:
neet = leggi("rcfl_neet_regionale_long.csv")
neet["neet_pct"] = pd.to_numeric(neet["neet_pct"])
neet["nome_territorio"] = neet["territorio"].map(NOMI)
neet[["territorio", "nome_territorio", "anno", "genere", "eta", "neet_pct"]].to_csv(
    PROCESSED / "genere_neet_rcfl.csv", index=False)

fuori_li = leggi("genere_fuori_lavoro_istruzione.csv")
proxy_sicilia = (fuori_li[fuori_li["territorio"].eq(SICILIA) & fuori_li["anno"].eq(2024)]
                 .set_index("genere")["quota_pct"].astype(float))
neet_2024 = neet[neet["anno"].eq(2024)].pivot_table(index=["nome_territorio", "eta"],
                                                   columns="genere", values="neet_pct")[["F", "M", "T"]]
print("Sicilia 2024, fascia 15-24: RCFL NEET contro proxy censuario «fuori da lavoro e studio»")
print(pd.DataFrame({"NEET RCFL (%)": neet_2024.loc[("Sicilia", "Y15-24")],
                    "proxy censuario (%)": proxy_sicilia}).round(1).to_string())
neet_2024

Sicilia 2024, fascia 15-24: RCFL NEET contro proxy censuario «fuori da lavoro e studio»
        NEET RCFL (%)  proxy censuario (%)
genere                                    
F                17.4                 21.8
M                21.5                 22.7
T                19.5                 22.3


genere                          F          M          T
nome_territorio eta                                    
Italia          Y15-24  11.596079  12.297676  11.959324
                Y15-29  16.633438  13.846822  15.190796
                Y15-34  20.648138  14.195112  17.323425
                Y18-29  20.311037  16.673310  18.426117
Sicilia         Y15-24  17.400803  21.468405  19.499739
                Y15-29  27.393836  24.185596  25.742155
                Y15-34  35.257603  25.219984  30.118529
                Y18-29  33.539896  29.288454  31.351632

**📌 Risultato chiave**: nel 2024 il NEET 15-34 della Sicilia (RCFL) è al 30.1%: 35.3%
fra le donne, 25.2% fra gli uomini; in Italia 17.3%. Sulla fascia 15-24 la RCFL dà 19.5% per
la Sicilia, il proxy censuario 22.3%: stesso ordine di grandezza, con il censimento più alto
di 2.8 punti (fonti e definizioni diverse). Il divario di genere del NEET siciliano si apre
dopo i 24 anni: fra 15 e 24 le donne stanno sotto gli uomini (17.4% contro 21.5%), fra 15 e
34 sopra di dieci punti. Va nella stessa direzione della finestra femminile dopo i 22 anni
osservata a Bagheria, ma è un dato regionale di un'altra fonte.

### Che effetto può vedere il pilota?
La valutazione sui partecipanti (policy, sezione 7) confronta chi comincia subito con chi
comincia sei mesi dopo: 200 posti in due coorti da 100. L'esito primario è la quota di
partecipanti occupati, in istruzione o in formazione qualificante a sei mesi. Il suo livello
senza servizio non è noto, quindi la soglia si calcola su tre basi plausibili (20%, 30%, 40%)
e, per il caso di adesioni dimezzate, anche con 50 persone per coorte. Stessa formula della
sezione sul KPI (arcoseno, α=0.05 bilaterale, potenza 80%): qui il modello binomiale è quello
giusto, perché i partecipanti sono persone assegnate a caso, non una stima comunale.

In [61]:
pilota = pd.DataFrame([{"persone per coorte": n, "esito a sei mesi senza servizio (%)": 100 * p0,
                        "effetto minimo rilevabile (pp)": mde_pp(p0, n)}
                       for n in (100, 50) for p0 in (0.20, 0.30, 0.40)]).round(1)
pilota.to_csv(PROCESSED / "genere_potenza_pilota.csv", index=False)
pilota

,persone per coorte,esito a sei mesi senza servizio (%),effetto minimo rilevabile (pp)
0,100,20.0,17.8
1,100,30.0,19.2
2,100,40.0,19.7
3,50,20.0,25.8
4,50,30.0,27.4
5,50,40.0,27.6


**📌 Risultato chiave**: con 100 persone per coorte il confronto vede solo effetti di
18-20 punti percentuali o più (17.8 con un esito base del 20%, 19.7 con il 40%); con adesioni
dimezzate la soglia sale a 26-28 punti. Nel primo anno il pilota distingue solo effetti
grandi: un effetto più piccolo resterebbe non dimostrato, non assente. Il confronto esiste
solo sull'esito a sei mesi, perché la seconda coorte comincia al sesto mese: sull'esito a
dodici mesi non c'è gruppo di controllo, e se le domande non superano i posti non c'è
sorteggio.

### Quanto costa il pilota?
La proposta fissa la dotazione minima (4 case manager e 1 data manager per 12 mesi, policy
§3) ma non un importo. Qui un ordine di grandezza in due scenari, personale comunale o
affidamento a una cooperativa sociale, con parametri presi dalle tabelle ufficiali e
trascritti con la loro fonte. Sono ipotesi di costo, non dati: stanno in una cella perché
ogni cifra della proposta si rigeneri da qui (fonti in `docs/sources.md` §13.4).

In [62]:
PARAMETRI_COSTO = pd.DataFrame([
    ("cooperativa_d2", "costo annuo cooperativa sociale, livello D2 (educatore professionale, assistente sociale), senza indennità di turno",
     35812.94, "euro/anno", "Ministero del Lavoro, D.D. n. 30 del 14/6/2024, tabella gennaio 2026, riga COSTO ANNUO"),
    ("comune_funzionari", "costo lordo annuo ente locale, area dei Funzionari e dell'Elevata Qualificazione",
     44539.99, "euro/anno", "Ministero del Lavoro, Nota metodologica UCS costi del personale (D.D. 105/2026, agg. D.D. 130/2026), Tabella 1"),
    ("indiretti", "tasso forfettario dei costi indiretti sul costo del personale",
     0.15, "quota", "Reg. (UE) 2021/1060, art. 54, par. 1, lett. b), come nella nota metodologica"),
    ("tirocinio_mese", "indennità minima mensile di tirocinio extracurriculare in Sicilia",
     300.0, "euro/mese", "Deliberazione della Giunta regionale siciliana n. 292 del 19/7/2017"),
    ("gol_ucs_ora", "unità di costo standard GOL, orientamento specialistico e accompagnamento al lavoro",
     39.94, "euro/ora", "ANPAL, Delibera del Commissario straordinario n. 5 del 12/4/2023, Allegato B"),
    ("gol_ore_p4", "ore massime GOL, percorso 4: orientamento specialistico (10) e accompagnamento (20)",
     30.0, "ore", "ANPAL, Delibera del Commissario straordinario n. 5 del 12/4/2023, Allegato B"),
], columns=["chiave", "parametro", "valore", "unita", "fonte"])
PARAMETRI_COSTO.to_csv(PROCESSED / "genere_costo_parametri.csv", index=False)
par = PARAMETRI_COSTO.set_index("chiave")["valore"]

PERSONE, POSTI, POSIZIONI_ESPERIENZA = 5, 200, 30  # policy §3 e §5 (soglia del modulo esperienza)
righe = []
for scenario, chiave in [("personale comunale, area Funzionari", "comune_funzionari"),
                         ("affidamento a cooperativa sociale, livello D2", "cooperativa_d2")]:
    personale = PERSONE * par[chiave]
    indiretti = personale * par["indiretti"]
    righe.append({"scenario": scenario, "persone": PERSONE, "costo annuo per persona": par[chiave],
                  "personale": personale, "costi indiretti": indiretti,
                  "totale annuo": personale + indiretti, "costo per posto": (personale + indiretti) / POSTI})
costo_pilota = pd.DataFrame(righe).round(2)
costo_pilota.to_csv(PROCESSED / "genere_costo_pilota.csv", index=False)
print(f"Riferimento GOL, percorso 4: {par['gol_ore_p4'] * par['gol_ucs_ora']:.2f} euro per partecipante "
      f"(solo orientamento specialistico e accompagnamento)")
print(f"Esperienze retribuite, non incluse: almeno {POSIZIONI_ESPERIENZA * par['tirocinio_mese']:.0f} euro "
      f"per ogni mese, con {POSIZIONI_ESPERIENZA} posizioni all'indennità minima siciliana")
costo_pilota

Riferimento GOL, percorso 4: 1198.20 euro per partecipante (solo orientamento specialistico e accompagnamento)
Esperienze retribuite, non incluse: almeno 9000 euro per ogni mese, con 30 posizioni all'indennità minima siciliana


,scenario,persone,costo annuo per persona,personale,costi indiretti,totale annuo,costo per posto
0,"personale comunale, area Funzionari",5,44539.99,222699.95,33404.99,256104.94,1280.52
1,"affidamento a cooperativa sociale, livello D2",5,35812.94,179064.70,26859.70,205924.41,1029.62


**📌 Risultato chiave**: il pilota costa in ordine di grandezza fra 206.000 euro l'anno
(cooperativa sociale) e 256.000 (personale comunale), cioè fra 1.030 e 1.281 euro per posto.
Il percorso più intensivo del programma GOL paga orientamento specialistico e accompagnamento
fino a 1.198 euro per partecipante: stesso ordine di grandezza. Restano fuori le esperienze
retribuite, che partono solo su posizioni verificate (almeno 9.000 euro per ogni mese con 30
posizioni), l'IVA di un eventuale affidamento, il tempo del personale già in servizio di
Comune, scuole e Centro per l'impiego, e la valutazione esterna. È una stima parametrica, non
un piano economico.

### Dopo i 25 anni: ritardo o mancata conversione?
La forbice del 15-24 (più istruite, meno occupate) la prevedono due ipotesi diverse. Il
**ritardo**: le ragazze studiano più a lungo, quindi a 15-24 anni lavorano meno, e dopo
recuperano. La **mancata conversione**: finito lo studio il lavoro non arriva, e il divario
resta. Sul 15-24 le due ipotesi predicono gli stessi dati; dopo il percorso formativo no.

La tavola lavoro comunale ha una sola classe adulta sotto i 50 anni, **25-49**. Qui non è un
proxy dei giovani (la regola delle definizioni resta): è il test di che cosa succede *dopo*
la fascia target. Lo scarto si misura contro Palermo e la Sicilia, per genere: se il
deficit femminile di Bagheria fosse solo un ritardo, a 25-49 dovrebbe chiudersi.

In [63]:
# Tasso di occupazione e quota di casalinghe per classe d'età, genere e territorio: stessa
# tavola lavoro del 15-24, stesse convenzioni (occupati = 1, casalinghe = 4, totale = 99).
CLASSI = ["Y15-24", "Y25-49", "Y50-64"]
per_classe = (istr_lav[istr_lav["tavola"].eq("lavoro") & istr_lav["territorio"].isin(CONFRONTO)
                       & istr_lav["eta"].isin(CLASSI) & istr_lav["cittadinanza"].eq("TOTAL")
                       & istr_lav["titolo_studio"].eq("ALL") & istr_lav["genere"].isin(["M", "F"])]
              .pivot_table(index=["territorio", "anno", "eta", "genere"], columns="condizione",
                           values="valore", aggfunc="sum"))
per_classe = per_classe[per_classe["99"].gt(0)]  # il 2020 manca alla fonte sulla classe 15-24
dopo_25 = pd.DataFrame({"popolazione": per_classe["99"], "occupati": per_classe["1"],
                        "tasso_occupazione": 100 * per_classe["1"] / per_classe["99"],
                        "quota_casalinghe": 100 * per_classe["4"] / per_classe["99"]}).reset_index()
dopo_25["nome_territorio"] = dopo_25["territorio"].map(NOMI)
dopo_25.round({"tasso_occupazione": 4, "quota_casalinghe": 4}).to_csv(
    PROCESSED / "genere_dopo_25.csv", index=False)

# Lo scarto locale: Bagheria meno ciascun benchmark, stessa classe, genere e anno.
tasso_classe = dopo_25.pivot_table(index=["anno", "eta", "genere"], columns="territorio",
                                   values="tasso_occupazione")
scarti_classe = pd.DataFrame({"vs Palermo": tasso_classe[BAGHERIA] - tasso_classe[PALERMO],
                              "vs Sicilia": tasso_classe[BAGHERIA] - tasso_classe[SICILIA]})
scarti_classe.round(4).reset_index().to_csv(PROCESSED / "genere_dopo_25_scarti.csv", index=False)

# Controllo di composizione: se le donne 25-49 di Bagheria fossero più anziane di quelle dei
# benchmark, parte dello scarto sarebbe età e non territorio.
classi5 = (quinquennali[quinquennali["territorio"].isin(CONFRONTO) & quinquennali["genere"].eq("F")
                        & quinquennali["cittadinanza"].eq("TOTAL")]
           .pivot_table(index=["anno", "territorio"], columns="eta", values="valore", aggfunc="sum"))
BANDA = ["Y25-29", "Y30-34", "Y35-39", "Y40-44", "Y45-49"]
quota_35_49 = (100 * classi5[BANDA[2:]].sum(axis=1) / classi5[BANDA].sum(axis=1)).unstack("territorio")

# In persone: le occupate in più che Bagheria avrebbe al tasso femminile di ciascun benchmark.
def occupate_al_tasso(classe, codice, anno=2024):
    r = dopo_25.set_index(["territorio", "anno", "eta", "genere"])
    b, t = r.loc[(BAGHERIA, anno, classe, "F")], r.loc[(codice, anno, classe, "F")]
    return b["popolazione"] * (t["tasso_occupazione"] - b["tasso_occupazione"]) / 100

print("Scarto di Bagheria sul tasso di occupazione (punti), 2024:")
print(scarti_classe.xs(2024, level="anno").round(1).unstack("genere").to_string())
print("\nScarto femminile e maschile a 25-49, serie:")
print(scarti_classe.xs("Y25-49", level="eta").round(1).unstack("genere").to_string())
print("\nTasso di occupazione 25-49 (2024, %):")
print(tasso_classe.xs((2024, "Y25-49")).T.rename(index=NOMI).reindex(ORDINE).round(1).to_string())
print("\nQuota di casalinghe fra le donne 25-49 (2024, %):")
print(dopo_25.query("anno == 2024 and eta == 'Y25-49' and genere == 'F'")
      .set_index("nome_territorio")["quota_casalinghe"].reindex(ORDINE).round(1).to_string())
print("\nQuota di 35-49enni fra le donne 25-49 (%):")
print(quota_35_49.loc[[2018, 2024]].rename(columns=NOMI)[ORDINE].round(1).to_string())
pop_f_25_49 = dopo_25.query("territorio == @BAGHERIA and anno == 2024 and eta == 'Y25-49' and genere == 'F'")
print(f"\nDonne 25-49 a Bagheria (2024): {pop_f_25_49['popolazione'].item():,.0f}, "
      f"occupate {pop_f_25_49['occupati'].item():,.0f}. Al tasso femminile di Palermo: "
      f"{occupate_al_tasso('Y25-49', PALERMO):+.0f} occupate (sul 15-24: "
      f"{occupate_al_tasso('Y15-24', PALERMO):+.0f}); al tasso siciliano: "
      f"{occupate_al_tasso('Y25-49', SICILIA):+.0f}.")

Scarto di Bagheria sul tasso di occupazione (punti), 2024:
       vs Palermo      vs Sicilia     
genere          F    M          F    M
eta                                   
Y15-24       -1.4 -0.1       -2.2 -3.8
Y25-49       -8.4 -1.3       -8.0 -3.1
Y50-64      -12.8 -4.4      -10.9 -4.3

Scarto femminile e maschile a 25-49, serie:
       vs Palermo      vs Sicilia     
genere          F    M          F    M
anno                                  
2018         -9.8 -1.9       -9.5 -4.7
2019         -9.3 -2.0       -9.1 -4.7
2021        -10.2 -1.1       -9.3 -2.7
2022        -10.0 -0.5       -9.3 -2.4
2023         -9.7 -0.7       -9.3 -2.7
2024         -8.4 -1.3       -8.0 -3.1

Tasso di occupazione 25-49 (2024, %):
genere         F     M
territorio            
Bagheria    39.6  67.7
Palermo     48.0  69.1
Sicilia     47.6  70.9
Italia      67.1  82.1

Quota di casalinghe fra le donne 25-49 (2024, %):
nome_territorio
Bagheria    42.2
Palermo     33.8
Sicilia     33.2
Italia      18.1

**📌 Risultato chiave** — Dopo i 25 anni il deficit locale diventa femminile. A 15-24 anni
Bagheria sta sotto Palermo di 1.4 punti sulle ragazze e di 0.1 sui ragazzi, e sotto la Sicilia
più sui ragazzi (−3.8) che sulle ragazze (−2.2): lo scarto locale dei giovanissimi non è di
genere. A 25-49 anni lo è: le donne di Bagheria lavorano al 39.6% contro il 48.0% di Palermo e
il 47.6% della Sicilia (−8.4 e −8.0 punti), gli uomini al 67.7% contro 69.1% e 70.9% (−1.3 e
−3.1). Lo scarto femminile sta fra −8.0 e −10.2 punti in ogni anno dal 2018, e non è composizione
per età: le 35-49enni sono il 64.1% delle donne 25-49 a Bagheria, il 64.7% a Palermo, il 64.2%
in Sicilia. In persone, al tasso femminile di Palermo le occupate 25-49 sarebbero **695 in
più** (sul 15-24: 40). Le casalinghe sono il 42.2% delle donne 25-49, contro il 33.8% di
Palermo, il 33.2% della Sicilia e il 18.1% dell'Italia.

> Il ritardo prevedeva un divario che si chiude dopo lo studio: non si chiude, e diventa
> locale. È il sostegno più forte che la tesi della mancata conversione abbia nei dati
> comunali, e sta fuori dalla fascia target: la parte più grande del problema femminile di
> Bagheria riguarda donne sopra i 25 anni. Cautela: la classe 25-49 mescola generazioni
> diverse, e sul 50-64 lo scarto femminile è ancora più ampio (−12.8 da Palermo, −10.9 dalla
> Sicilia). La cella qui sotto verifica se la generazione giovane ne è risparmiata.

### La generazione successiva ha lo stesso problema?
La classe 25-49 mescola chi ha appena finito di studiare con la generazione delle madri, e lo
scarto femminile cresce con l'età (sul 50-64 è ancora più ampio): il deficit del 25-49
potrebbe essere tutto delle 35-49enni. La tavola comunale non ha classi più fini, ma la
serie 2018-2024 permette un test di **ricambio delle coorti**.

Ipotesi da smentire (H0): la generazione giovane, chi nel 2018 aveva meno di 35 anni, non ha
scarto locale; lo scarto del 2018 è tutto delle 35-49enni di allora. Fra 2018 e 2024 quella
generazione vecchia esce in parte dalla classe (le 44-49enni del 2018 passano ai 50-64) e al
suo posto entrano le nate fra il 1994 e il 1999. Sotto H0 lo scarto del 2024 è quello del 2018 per
il rapporto fra le quote della generazione vecchia nella classe, 41-49enni nel 2024 contro
35-49enni nel 2018 (quote dal registro: classi quinquennali nel 2018, età singole nel 2024).

Due assunzioni, entrambe dal lato prudente. Lo scarto è uniforme dentro la generazione
vecchia: se crescesse con l'età, come suggerisce il 50-64, chi è uscito ne portava di più e
sotto H0 lo scarto dovrebbe chiudersi ancora più in fretta. Nessuna convergenza generale di
Bagheria verso i benchmark: se ci fosse, sotto H0 la chiusura sarebbe ancora maggiore. L'errore è binomiale, con la convenzione del notebook. Età e generazione non si
separano con una sola classe: il test dice se la generazione giovane è risparmiata, non
quanto dello scarto viene dall'età e quanto dalla coorte.

In [64]:
# Test di ricambio delle coorti sulla classe 25-49. Le quote di popolazione sono di Bagheria:
# quelle dei benchmark sono le stesse entro due punti (controllo di composizione sopra).
def quota_generazione_vecchia(genere):
    q18 = classi5_g.loc[(2018, BAGHERIA, genere)]
    s18 = q18[BANDA[2:]].sum() / q18[BANDA].sum()                     # 35-49 su 25-49, 2018
    s24 = (sum(p2024.get((BAGHERIA, genere, a)) for a in range(41, 50))
           / sum(p2024.get((BAGHERIA, genere, a)) for a in range(25, 50)))  # 41-49 su 25-49, 2024
    return s18, s24


classi5_g = (quinquennali[quinquennali["territorio"].eq(BAGHERIA) & quinquennali["genere"].isin(["M", "F"])
                          & quinquennali["cittadinanza"].eq("TOTAL")]
             .pivot_table(index=["anno", "territorio", "genere"], columns="eta", values="valore", aggfunc="sum"))
righe_d = dopo_25.set_index(["territorio", "anno", "eta", "genere"])


def varianza_scarto(anno, genere, codice):
    """Varianza binomiale dello scarto Bagheria − benchmark sul tasso 25-49 (in punti²)."""
    v = 0.0
    for t in (BAGHERIA, codice):
        r = righe_d.loc[(t, anno, "Y25-49", genere)]
        p = r["tasso_occupazione"] / 100
        v += 1e4 * p * (1 - p) / r["popolazione"]
    return v


righe = []
for genere in ("F", "M"):
    s18, s24 = quota_generazione_vecchia(genere)
    for nome, codice in (("Palermo", PALERMO), ("Sicilia", SICILIA)):
        d18 = scarti_classe.loc[(2018, "Y25-49", genere), f"vs {nome}"]
        d24 = scarti_classe.loc[(2024, "Y25-49", genere), f"vs {nome}"]
        previsto = d18 * s24 / s18
        z = (d24 - previsto) / np.sqrt(varianza_scarto(2024, genere, codice)
                                       + (s24 / s18) ** 2 * varianza_scarto(2018, genere, codice))
        # Due generazioni, due incognite: lo scarto medio di ciascuna che riproduce 2018 e 2024.
        vecchia, giovane = np.linalg.solve([[s18, 1 - s18], [s24, 1 - s24]], [d18, d24])
        righe.append({"genere": genere, "benchmark": nome,
                      "quota_vecchia_2018": 100 * s18, "quota_vecchia_2024": 100 * s24,
                      "scarto_2018": d18, "scarto_2024": d24, "scarto_2024_se_H0": previsto,
                      "z_osservato_meno_H0": z,
                      "scarto_generazione_vecchia": vecchia, "scarto_generazione_giovane": giovane})
generazioni = pd.DataFrame(righe).round(4)
generazioni.to_csv(PROCESSED / "genere_dopo_25_generazioni.csv", index=False)
print(generazioni.set_index(["genere", "benchmark"]).round(1).to_string())

# Lo stesso passaggio a scala regionale, da un'altra fonte (RCFL, NEET per regione). Il 25-34
# si ricava per differenza fra 15-34 e 15-24 pesando con la popolazione del registro: è una
# ricostruzione, non un dato pubblicato, e la RCFL usa pesi propri.
def neet_pct(territorio, genere, eta, anno=2024):
    r = neet[neet["territorio"].eq(territorio) & neet["genere"].eq(genere)
             & neet["anno"].eq(anno) & neet["eta"].eq(eta)]
    return r["neet_pct"].item()


def neet_25_34(territorio, genere):
    pop = lambda a0, a1: sum(p2024.get((territorio, genere, a)) for a in range(a0, a1 + 1))
    return (neet_pct(territorio, genere, "Y15-34") * pop(15, 34)
            - neet_pct(territorio, genere, "Y15-24") * pop(15, 24)) / pop(25, 34)


neet_giovani = pd.DataFrame([{"territorio": t, "nome_territorio": NOMI[t], "genere": g,
                              "neet_15_24": neet_pct(t, g, "Y15-24"), "neet_25_34": neet_25_34(t, g)}
                             for t in (SICILIA, ITALIA) for g in ("F", "M")]).round(1)
neet_giovani.to_csv(PROCESSED / "genere_neet_25_34.csv", index=False)
print("\nNEET RCFL, 2024 (%): 15-24 pubblicato, 25-34 ricavato")
print(neet_giovani.pivot_table(index="nome_territorio", columns="genere",
                               values=["neet_15_24", "neet_25_34"]).to_string())

                  quota_vecchia_2018  quota_vecchia_2024  scarto_2018  scarto_2024  scarto_2024_se_H0  z_osservato_meno_H0  scarto_generazione_vecchia  scarto_generazione_giovane
genere benchmark                                                                                                                                                                  
F      Palermo                  62.5                40.0         -9.8         -8.4               -6.3                 -3.3                       -12.1                        -5.9
       Sicilia                  62.5                40.0         -9.5         -8.0               -6.1                 -3.2                       -11.9                        -5.5
M      Palermo                  62.0                39.4         -1.9         -1.3               -1.2                 -0.2                        -2.8                        -0.4
       Sicilia                  62.0                39.4         -4.7         -3.1               -3.0    

**📌 Risultato chiave** — La generazione giovane non è risparmiata. Se lo scarto femminile
del 25-49 fosse tutto delle donne che nel 2018 avevano 35-49 anni, con il ricambio delle
coorti nel 2024 sarebbe dovuto scendere a −6.3 punti da Palermo e a −6.1 dalla Sicilia; è a
−8.4 e −8.0 (z −3.3 e −3.2). Risolvendo per due generazioni, le nate dal 1984 in poi portano
uno scarto di circa **−6 punti** (−5.9 da Palermo, −5.5 dalla Sicilia), la metà di quella
precedente (−12); fra gli uomini la stessa generazione ha uno scarto di −0.4 punti. Alla
scala regionale, da un'altra fonte, il passaggio si vede per intero: in Sicilia il NEET
femminile è sotto quello maschile a 15-24 anni (17.4% contro 21.5%) e quasi il doppio a 25-34
(52.2% contro 28.9%, ricavato per differenza); in Italia 29.2% contro 16.0%.

> Tre letture. Il deficit femminile dopo i 25 anni non è solo un'eredità delle madri: la
> generazione nata dal 1984 in poi ne porta circa metà, i coetanei maschi niente. Il test non
> separa età e coorte (una sola classe, sei anni): i −6 punti possono essere un miglioramento
> fra generazioni o lo stesso scarto che cresce con l'età, e nel secondo caso le ragazze di
> oggi lo raggiungeranno. E lo scarto per generazione viene da un modello a due gradini con
> il confine fissato a 35 anni nel 2018: si cita come ordine di grandezza, non come stima.

### Il titolo di studio oltre la fascia 9-24: fin dove arriva il vantaggio delle donne
La fascia 9-24 dice quale titolo i ragazzi detengono, non quanto in alto arriveranno: a 20
anni una laurea non può ancora esserci. La tavola istruzione comunale pubblica altre tre
classi (25-49, 50-64, 65+), anche per i cinque comuni vicini. Due domande. Il vantaggio
femminile regge nelle classi adulte e sul titolo terziario? E sui titoli degli adulti
Bagheria sta sopra o sotto i comuni che la circondano?

Come nella sezione «Dopo i 25 anni», il 25-49 qui non è un proxy dei giovani: è la classe in
cui il percorso formativo è concluso. **Terziario** = `BL` + `ML_RDD`, cioè ITS, laurea di
primo e di secondo livello e dottorato (codelist in `docs/sources.md`, §14): non si chiama
«laurea» senza qualificazione.

In [65]:
# Titolo di studio per classe d'età e genere, dalla stessa tavola istruzione della fascia
# 9-24: i territori di confronto, i cinque vicini uno per uno e il vicinato come territorio
# unico (conteggi sommati e poi le quote, come in «I comuni vicini»).
CLASSI_TITOLO = ["Y9-24", "Y25-49", "Y50-64", "Y_GE65"]
TERZIARIO = ["BL", "ML_RDD"]


def quote_titolo(tavola, codice_aggregato=None):
    base = tavola[tavola["eta"].isin(CLASSI_TITOLO) & tavola["cittadinanza"].eq("TOTAL")
                  & tavola["genere"].isin(["M", "F", "T"])]
    if codice_aggregato:
        base = base.assign(territorio=codice_aggregato)
    c = base.pivot_table(index=["territorio", "anno", "eta", "genere"], columns="titolo_studio",
                         values="valore", aggfunc="sum")
    # Sulle quattro classi la partizione è esatta (lo scarto di Y_GE9 non entra qui).
    scarto = (c[TITOLI].sum(axis=1) - c["ALL"]).abs().max()
    assert scarto < 0.5, f"le categorie non ricostruiscono il totale, scarto {scarto}"
    return pd.DataFrame({
        "popolazione": c["ALL"],
        "almeno_diploma_%": 100 * c[ALMENO_DIPLOMA].sum(axis=1) / c["ALL"],
        "solo_diploma_%": 100 * c["USE_IF"] / c["ALL"],
        "terziario_%": 100 * c[TERZIARIO].sum(axis=1) / c["ALL"],
    }).reset_index()


titoli_eta = pd.concat([quote_titolo(istruzione[istruzione["territorio"].isin(CONFRONTO)]),
                        quote_titolo(istruzione_vicini),
                        quote_titolo(istruzione_vicini, CODICE_VICINATO)], ignore_index=True)
titoli_eta["nome_territorio"] = titoli_eta["territorio"].map({**NOMI, CODICE_VICINATO: NOME_VICINATO})
titoli_eta["ruolo"] = titoli_eta["territorio"].map(
    lambda t: "confronto" if t in CONFRONTO else ("vicinato" if t == CODICE_VICINATO else "vicino"))
assert titoli_eta["nome_territorio"].notna().all()

# Due percorsi, stesso numero: sul 9-24 le quote devono tornare quelle di genere_istruzione.csv
# (territori di confronto) e di genere_forbice_vicini.csv (vicinato, anno comune).
gia = pd.read_csv(PROCESSED / "genere_istruzione.csv", dtype={"territorio": str})
ctrl = titoli_eta[titoli_eta["eta"].eq("Y9-24")].merge(gia, on=["territorio", "anno", "genere"])
assert len(ctrl) == len(gia) and (ctrl["almeno_diploma_%_x"].round(1) - ctrl["almeno_diploma_%_y"]).abs().max() < 0.01
fv = pd.read_csv(PROCESSED / "genere_forbice_vicini.csv").iloc[0]
q_vic = titoli_eta.query("territorio == @CODICE_VICINATO and eta == 'Y9-24' and anno == @anno_comune").set_index("genere")
assert abs(q_vic.at["F", "almeno_diploma_%"] - fv["almeno_diploma_F"]) < 0.051
assert abs(q_vic.at["M", "almeno_diploma_%"] - fv["almeno_diploma_M"]) < 0.051

titoli_eta.round({"almeno_diploma_%": 4, "solo_diploma_%": 4, "terziario_%": 4}).to_csv(
    PROCESSED / "genere_titoli_eta.csv", index=False)

MISURE_TITOLO = ["almeno_diploma_%", "solo_diploma_%", "terziario_%"]
anno_t = titoli_eta["anno"].max()
ultimo = titoli_eta[titoli_eta["anno"].eq(anno_t)]


def per_genere(righe, indice):
    t = righe[righe["genere"].isin(["F", "M"])].pivot_table(index=indice, columns="genere",
                                                           values=MISURE_TITOLO)
    for m in MISURE_TITOLO:
        t[(m, "F-M")] = t[(m, "F")] - t[(m, "M")]
    return t.sort_index(axis=1).round(1)


print(f"Bagheria {anno_t}, per classe d'età (%, e scarto F-M in punti):")
print(per_genere(ultimo[ultimo["territorio"].eq(BAGHERIA)], "eta").reindex(CLASSI_TITOLO).to_string())

adulti = ultimo[ultimo["eta"].eq("Y25-49")]
graduatoria = (adulti[adulti["genere"].eq("T")].set_index("nome_territorio")
               [["popolazione", "almeno_diploma_%", "solo_diploma_%", "terziario_%"]]
               .sort_values("almeno_diploma_%", ascending=False).round(1))
print(f"\n25-49 anni, {anno_t}, maschi e femmine insieme, in ordine di quota con almeno il diploma:")
print(graduatoria.to_string())
print(f"\n25-49 anni, {anno_t}, per genere:")
print(per_genere(adulti, "nome_territorio").reindex(graduatoria.index).to_string())

serie_gap = per_genere(titoli_eta[titoli_eta["eta"].eq("Y25-49")
                                  & titoli_eta["territorio"].isin([BAGHERIA, CODICE_VICINATO, SICILIA])],
                       ["nome_territorio", "anno"])
print("\nScarto F-M sul 25-49, serie (punti):")
print(serie_gap[[("almeno_diploma_%", "F-M"), ("terziario_%", "F-M")]].unstack("nome_territorio").to_string())

Bagheria 2024, per classe d'età (%, e scarto F-M in punti):
       almeno_diploma_%            solo_diploma_%            terziario_%           
genere                F  F-M     M              F  F-M     M           F  F-M     M
eta                                                                                
Y9-24              33.4  4.2  29.2           30.0  3.0  27.0         3.5  1.2   2.2
Y25-49             65.2  5.6  59.6           40.5 -2.8  43.3        24.7  8.3  16.3
Y50-64             42.3  1.0  41.3           30.2  0.2  30.0        12.2  0.8  11.3
Y_GE65             23.5 -5.0  28.4           15.5 -4.0  19.5         8.0 -1.0   8.9

25-49 anni, 2024, maschi e femmine insieme, in ordine di quota con almeno il diploma:
                     popolazione  almeno_diploma_%  solo_diploma_%  terziario_%
nome_territorio                                                                
Italia                17403311.0              75.9            46.6         29.3
Sicilia                14

**📌 Risultato chiave** — Fino a 49 anni le donne di Bagheria hanno più titoli degli uomini,
e il vantaggio cresce salendo di grado. A 25-49 anni ha almeno il diploma il 65.2% delle donne
contro il 59.6% degli uomini (+5.6 punti), un titolo terziario il 24.7% contro il 16.3% (+8.3).
Fermarsi al diploma è invece più maschile (43.3% contro 40.5%), perché più donne vanno oltre.
Sul 50-64 i due generi sono quasi pari (+1.0 sul diploma, +0.8 sul terziario), sopra i 65 anni
il segno si inverte (−5.0 sul diploma). Il vantaggio adulto non è una particolarità di Bagheria:
sul 25-49 vale +6.6 e +9.0 punti in Sicilia, +6.4 e +10.8 in Italia, +3.0 e +6.8 nel vicinato
aggregato; fra il 2018 e il 2024 sale in tutti e tre (a Bagheria da +6.2 a +8.3 sul terziario).

Sul livello Bagheria sta a metà del vicinato e sotto tutti i territori di confronto: fra i
25-49enni ha almeno il diploma il 62.4% e un titolo terziario il 20.5%, contro il 60.5% e il
17.0% del vicinato aggregato, il 65.6% e il 26.6% di Palermo, il 66.5% e il 23.3% della Sicilia.
Davanti ha Santa Flavia (65.3%) e Casteldaccia (64.8%), dietro Ficarazzi (61.6%), Misilmeri
(60.4%) e Villabate (54.7%).

> Il vantaggio femminile nei titoli è generazionale (compare sotto i 50 anni) e comune a tutti
> i territori; a Bagheria è più largo che nel vicinato aggregato e un po' più stretto che in
> Sicilia. Non dice quante diplomate o laureate lavorino: l'incrocio fra titolo e condizione
> professionale non è pubblicato a livello comunale. Due cautele. Il 25-49 mescola generazioni
> diverse. Fra Bagheria, Santa Flavia e Casteldaccia ci sono 2-3 punti, e i due comuni hanno
> circa 3.500 residenti della classe: l'ordine fra i tre non è un risultato.

# **Sintesi finale**

I risultati delle sezioni precedenti, ricomposti nell'ordine in cui reggono la proposal.
Il riferimento fra virgolette è la sezione che produce la cifra: nulla qui è calcolato a
mano, tutto si rigenera rieseguendo il notebook.

---

### 1. Il paradosso istruzione / lavoro — il risultato centrale
* **Istruzione (9-24)**: 33.4% F contro 29.2% M, gap **-4.2 pp** — le ragazze studiano di più; sul 15-24, la fascia dell'occupazione, il vantaggio è +4.8 *(«Gap di genere sull'istruzione», «Su 1.000 ragazze»)*.
* **Occupazione (15-24)**: 8.2% F contro 16.5% M, gap **+8.3 pp** — lavorano la metà *(«Gap di genere sull'occupazione»)*.
* **Meccanismo**: a 15-24 la differenza è assorbita dalla permanenza nello studio (65.1% F contro 56.4% M), non dall'inattività *(«Dentro il "fuori da lavoro e studio"»)*.
* I due segni sono **opposti in tutti e quattro i territori**. A 15-24 anni però la forbice è attesa anche se il vantaggio si convertisse più tardi, perché le ragazze studiano più a lungo: la prova della mancata conversione viene dal 25-49 (§1-bis).

---

### 1-bis. Dopo i 25 anni il deficit locale diventa femminile, anche per la generazione giovane
* A 15-24 anni lo scarto di Bagheria dai benchmark non è di genere (sotto la Sicilia di 2.2 punti le ragazze, di 3.8 i ragazzi). A 25-49 lo è: donne al 39.6% contro il 48.0% di Palermo e il 47.6% della Sicilia, uomini a 1-3 punti. Al tasso femminile di Palermo le occupate 25-49 sarebbero **695 in più**, contro le 40 del 15-24; casalinghe al 42.2% delle donne 25-49 *(«Dopo i 25 anni: ritardo o mancata conversione?»)*.
* Non è solo la generazione delle madri: col ricambio delle coorti 2018-2024 lo scarto femminile si sarebbe dovuto chiudere a circa −6 punti, resta a −8 (z −3.3); le nate dal 1984 in poi portano circa −6 punti, i coetanei maschi −0.4. In Sicilia il NEET femminile passa da sotto quello maschile a 15-24 anni a quasi il doppio a 25-34 *(«La generazione successiva ha lo stesso problema?»)*.
* Il 25-49 è un test del meccanismo, non un proxy dei giovani, e sta fuori dalla fascia target: la parte più grande del problema femminile è sopra i 25 anni.

---

### 2. Il problema non è l'ampiezza del gap: è il livello
* **In punti** (8.3 pp) Bagheria sta sopra Palermo (6.9) ma sotto Sicilia (9.9) e Italia (9.7); il modello LPM lo conferma formalmente sul pooled 2022-2024 (+1.1 vs Palermo, p=0.03; -1.8/-1.9 vs Sicilia/Italia, p<0.001) *(«Il gap di Bagheria è un'anomalia locale?»)*.
* **In rapporto M/F** (2.01) Bagheria è invece la **peggiore del panel** (Palermo 1.72, Sicilia 1.95, Italia 1.56) *(«Punti percentuali o rapporto?»)*.
* Gli **effetti principali** dello stesso modello testano il livello: tasso femminile di Bagheria sotto Palermo di 1.2 pp, sotto la Sicilia di 1.9, sotto l'Italia di 8.9 (p ≤ 0.0001 sul pooled 2022-2024, p ≤ 0.009 anche sul solo 2024) *(«Il gap di Bagheria è un'anomalia locale?»)*.
* Il bersaglio della proposal è quindi il **livello dell'occupazione femminile — 8.2%, il più basso dei quattro territori** — non la chiusura di un divario "anomalo" che i dati non mostrano.

---

### 3. Le casalinghe: una stima del censimento, non una dichiarazione
* Nelle stime del censimento risulta casalinga il **13.4% delle ragazze 15-24 (387 persone)**, contro l'1.7% dei coetanei e il 4.6% delle coetanee italiane *(«Dentro gli "altri inattivi"»)*.
* Gradiente territoriale netto (Italia 4.6 < Sicilia 10.1 < Palermo 11.3 < Bagheria 13.4) e serie 2021-2024 fra 364 e 421 ragazze: **fenomeno strutturale, non episodico**. Il 2018-2019 viene da un altro metodo e non va messo in serie *(«Le casalinghe sono un conteggio o una stima?»)*.
* **Il canale non è il matrimonio precoce**: al 1.1.2025 le ragazze 15-24 già coniugate sono **41** contro 387 casalinghe — almeno l'**89% delle casalinghe non è sposata** — e la quota di già coniugate 20-24 di Bagheria (2.7%) sta *sotto* Palermo (3.2%) e Sicilia (2.9%) *(«Le casalinghe sono coniugate?»)*. Il matrimonio non è il canale (convivenze e maternità non sono osservate): il servizio è di **attivazione**, non solo di conciliazione.
* È l'unico meccanismo del thread con un target nominabile. Come KPI regge solo dentro la definizione 2021+: dal 2021 la condizione è la somma di probabilità stimate da un modello (valori non interi), non un conteggio, e la tavola comunale non ne riporta l'errore *(«Le casalinghe sono un conteggio o una stima?»)*.

---

### 4. Il gap in punti tende ad allargarsi, senza significatività
* Pendenza stimata +0.21 pp/anno a Bagheria; il p=0.008 del modello congiunto usa una varianza comune ai quattro territori, sulla sola serie di Bagheria p ≈ 0.05: direzione, non risultato. Passo indistinguibile da Palermo e Italia *(«Il gap si sta allargando?»)*.
* Nello stesso periodo il rapporto M/F **scende** (2.47 → 2.01): entrambi i tassi salgono, quello maschile di più in valore assoluto. Ogni claim deve dichiarare quale delle due scale sta usando.

---

### 5. La fuga di talenti: all'uscita dal percorso formativo, di genere solo nel triennio 2021-2024
* Bagheria trattiene meno dell'Italia in **ogni** coorte e per entrambi i generi: il drenaggio riguarda tutti *(«Ritenzione di coorte per genere»)*.
* Ma i ragazzi si perdono a 15-19 (98.5 contro 104.8), le ragazze **dopo i 25** (96.3 contro 97.6 in Sicilia, 98.8 a Palermo e 103.0 in Italia). Le ragazze restano finché studiano; il territorio le perde all'uscita dal percorso formativo — esattamente il punto in cui il vantaggio educativo dovrebbe diventare lavoro.
* La finestra 22-25 è una lettura **pooled 2021→2024**: nelle tre transizioni annuali il segno femminile compare ma oscilla (fino a 8 pp sulla stessa età, contro il ~±0.8 del solo rumore di conteggio) *(«La finestra 22-25 regge?»)*. La transizione singola è un controllo di direzione, non un titolo.
* Su cinque anni, dentro il solo censimento permanente, la perdita all'uscita dal percorso formativo si ritrova ma **non è femminile**: sulla transizione 20-24 → 25-29 i maschi perdono più delle femmine (89.2 e 91.8 contro 92.8 e 93.3), e le femmine di Bagheria stanno al livello di Palermo e della Sicilia; dove uno scarto di genere c'è, viene dai maschi trattenuti più che altrove *(«I claim reggono al 2024?»)*. Nel vicinato dei cinque comuni più vicini il cedimento femminile del triennio non compare (coorte 25-29 F a 102.4) *(«I comuni vicini»)*. Il tempismo di genere si cita come lettura del triennio 2021-2024, non come risultato.

---

### 6. Sotto il gap di genere ce n'è uno territoriale
* La quota di "altri inattivi" 15-24 è 14-20% a Bagheria, Palermo e Sicilia contro il ~9% nazionale, **per entrambi i generi** *(«Dentro il "fuori da lavoro e studio"»)*.
* E il 2011 mostra radici lunghe: occupazione femminile 15+ al 18.1% contro il 36.1% nazionale *(«Contesto storico 2011»)*.
* Un intervento di genere agisce quindi su uno svantaggio doppio: essere giovane a Bagheria, ed esservi giovane donna.

---

### 7. Traduzione in persone (per il template della proposal)
Base 2024: **2.882 ragazze 15-24, 236 occupate** *(«Il gap in persone»)*.

| Scenario | Occupate in più | Uso |
|---|---|---|
| Tasso femminile di Palermo | **+40** | obiettivo di convergenza, scritto in tasso |
| Parità con i coetanei maschi | +239 | misura del problema |
| Tasso femminile nazionale | +262 | misura del problema |

---

### 8. Il KPI ha una finestra di lettura, non solo un valore
* "+40 occupate" = **+1.40 pp**. Contro la variabilità osservata, senza interventi, nei 33 comuni siciliani di taglia simile la potenza è **50% su un anno, 59% sul biennio, 41% sul triennio**: il binomiale prometteva il 90% sul triennio perché tratta come indipendenti annualità che contano le stesse persone, e aggregare anni non toglie le divergenze persistenti fra comuni. Per le casalinghe (-2.1 pp): 82% sul biennio *(«Il KPI si può misurare?»)*.
* Conseguenza: i KPI di popolazione si leggono su finestre dichiarate prima (triennio per il tasso, biennio per le casalinghe) come **direzione** della convergenza, non come prova; l'effetto del servizio si misura sui partecipanti, dove il pilota vede solo effetti di 18-20 punti *(«Che effetto può vedere il pilota?»)*; la lettura annuale spetta a indicatori di processo (utenza per età e genere) che oggi **nessuno rileva**.

---

### 9. Dentro il gruppo dei pari: le gemelle
* Dieci comuni comparabili per struttura (matching Mahalanobis su variabili non-outcome, 2011; robustezza 8/10 e 6/10 sugli altri metodi, LOVO ≥7/10) *(«Le gemelle di Bagheria»)*: Santa Flavia, Capaci, Misilmeri, Altofonte, Trabia, Terrasini, Termini Imerese, Erice, Porto Empedocle, Sciacca.
* Sui livelli generali Bagheria sta come chi le somiglia (NEET 15-29 4/10, occupazione F 3/10, dati 2011): lo svantaggio giovanile è **di fascia territoriale**, e la proposal deve dirlo. Il profilo specifico è altrove: differenziale educativo pro-F più forte (1/10), disoccupazione F fra le peggiori (8/10 sotto), **autonomia giovanile al minimo** (F4: nessuna sotto), mobilità quasi minima (2/10).
* Su L11 Bagheria era **identica alle gemelle** nel 1991 e nel 2001; se ne stacca nel decennio 2001-2011 (18.1 contro 19.8) *(«Trend paralleli»)*: lo scarto è recente, e la diagnosi deve spiegare quel decennio.

---

### 10. Il gap delle madri: partecipazione su, assorbimento no
* 1991→2011 (15+): partecipazione femminile dall'**8° al 32° percentile** siciliano (20.6→28.7), ma occupazione femminile dal 22° al **12°** e disoccupazione femminile dal 46° al **95°** *(«Il gap delle madri»)*: le donne sono entrate nel mercato, il mercato non le ha assorbite.
* Il **sorpasso educativo è del 2011** (I1 98.9, unico territorio del panel sotto la parità): le ragazze del censimento permanente sono la prima generazione figlia del sorpasso — e trovano lo stesso muro. Il vantaggio educativo non convertito è il **regime del territorio da almeno un decennio**, non un incidente della serie 2018-2024.

---

### 11. Il "19% invisibile" è di entrambi i generi; l'etichetta no
* Fuori da lavoro, studio e ricerca (2024): **573 ragazze e 549 ragazzi** — 51% femminile *(«La ritenzione per età»)*. Di genere è la **composizione**: 387 casalinghe contro 50; 183 contro 485 in "altra condizione". Un intervento "per le invisibili" che ignorasse i ragazzi sbaglierebbe platea di metà; uno neutro che ignorasse l'etichetta mancherebbe il meccanismo.
* Il profilo per età data le rotture: ragazze **sopra la pari fino ai 23**, cedimento dalle età 24-25 (2021) in poi; ragazzi a due onde (17-19 e 23-24) con **rientri netti dopo i 26**, che le ragazze non hanno (triennio 2021-2024; su cinque anni la differenza di genere non si ripete, §5).

---

### 12. Il contesto familiare: autonomia al minimo, famiglia precoce
* Giovani che vivono da soli (F4) al **3° percentile** siciliano, ultimo anche nel gruppo gemelle; coppie giovani con figli (F7) all'81° dei 390 — ma nella mediana delle gemelle: tratto di fascia, non anomalia locale *(«Formazione familiare precoce»)*. Posizioni già così nel 1991.
* Bounds sulle casalinghe: fra il **18.8% delle 18-24enni** (nessuna minorenne) e il 25.8% delle 20-24enni (tutte maggiori di 19) *(«Chi sono le casalinghe?»)*; eccesso sull'incidenza italiana: **255 ragazze**. L'età resta non osservata: bounds, non stime.

---

### 13. Il bilancio dei giovani: la platea, il suo audit, il ricambio assente
* **La platea del target è già nata e si sta restringendo**: a saldo migratorio zero le ragazze 15-24 passano da 2.882 (2024) a 2.651 nel 2029 (-8.0%) e 2.435 nel 2034 (**-15.5%**, contro il -5.8% dei coetanei); nei benchmark il calo è simile in ampiezza ma quasi simmetrico fra i generi *(«Il bilancio dei giovani»)*. I KPI in teste si riparametrano sulla platea corrente (`genere_platea.csv`); quelli in tasso restano validi.
* **L'asimmetria ha un'origine anomala, documentata e replicata**: 117 maschi ogni 100 femmine fra i 5-14enni (2024) contro il 104-106 dei benchmark; nella norma nel 2001 e 2011, in salita da allora (z +3.5 sull'ultima finestra, identica su due tavole indipendenti, tutta nella popolazione italiana) *(«L'audit della platea»)*. Meccanismo aperto: il dato si cita **solo insieme al suo audit**.
* **Il ricambio dall'estero è debole**: stranieri all'**1.6% del 15-34** (195 persone) contro 5.1% Palermo, 6.4% Sicilia, 12.4% Italia *(«La componente straniera»)*. Lo stock 15-34 cala di 313 persone fra 2021 e 2024, ma 266 sono ricambio d'età: dentro le stesse coorti il saldo è −0.39%, come in Sicilia *(«Lo stock non è la fuga»)*. La specificità di Bagheria non è la quantità; il tempismo di genere è una lettura del triennio 2021-2024 (§5).

---

### 14. Le domande di approfondimento *(«Robustezza»)*
* **Bagheria fra i 390 comuni**: sull'occupazione femminile 15-24 è 112ª dal basso su 390 (mediana 9.7%), ma 2ª su 34 fra i comuni della sua taglia; sulle casalinghe 358ª dal basso. «Il tasso più basso» vale nel panel e fra i comuni simili, non in tutta la Sicilia.
* **Casalinghe**: dal 2021 una stima di modello, non un conteggio (celle non intere); occupati e totale restano conteggi.
* **Diplomate occupate**: al massimo il 16.1% delle ragazze 15-24 con almeno il diploma lavora (tetto dei margini), contro il 35.7% dei ragazzi; il minimo è zero.
* **NEET del bando**: 30.1% in Sicilia sul 15-34 (RCFL, 2024; 35.3% le donne), fonte regionale diversa dal censimento; sul 15-24 la RCFL dà 19.5% contro il 22.3% del proxy censuario.
* **Pilota**: con due coorti da 100 il confronto vede effetti di 18-20 punti o più, e solo sull'esito a sei mesi.
* **Costo**: fra 206.000 e 256.000 euro l'anno per la dotazione minima (cooperativa sociale o personale comunale, più il 15% di costi indiretti), fra 1.030 e 1.281 euro per posto; il percorso 4 del programma GOL paga fino a 1.198 euro per partecipante. Esperienze retribuite escluse *(«Quanto costa il pilota?»)*.

---

### Cosa entra nella proposal, e con quale KPI
L'intervento sta in `docs/policy/POLICY_PONTE_19.md`: qui ci sono evidenza, target e KPI, che sono
fatti misurabili e si rigenerano da questo notebook.

| Evidenza (sezione) | Target | KPI misurabile |
|---|---|---|
| Occupazione femminile 15-24 all'8.2%, la più bassa del panel, con rapporto M/F 2.01 *(«Punti percentuali o rapporto?», «Il gap in persone»)* | le 2.882 ragazze 15-24 residenti | tasso femminile allineato a Palermo (9.6%, cioè +40 occupate sulla platea 2024), letto come scarto da Palermo su trienni pooled |
| 387 ragazze 15-24 casalinghe, 13.4% contro il 4.6% nazionale, stabile nella serie 2021-2024 (fra 364 e 421); almeno l'89% non sposata *(«Dentro gli "altri inattivi"», «Le casalinghe sono coniugate?»)* | le ~390 casalinghe 15-24 | quota casalinghe 15-24: prima al livello di Palermo (11.3%), poi verso quello nazionale |
| Ritenzione femminile 25-29 al 96.3 contro il 103.0 nazionale *(«Ritenzione di coorte»)*; lettura del solo triennio 2021-2024: su cinque anni la perdita è di entrambi i generi e le femmine stanno al livello di Palermo *(«I claim reggono al 2024?»)* | le coorti femminili in uscita dal percorso formativo | ritenzione netta 25-29 F portata almeno al livello di Palermo (98.8) |

**Finestra di lettura** *(«Il KPI si può misurare?»)*: tasso sul **triennio**, quota casalinghe sul **biennio**, dichiarati prima; contro la variabilità dei comuni simili la potenza è 41% e 82%, quindi il tasso si legge come direzione, non come test; la ritenzione contro la variabilità osservata (±0.5 punti sulle letture annue). I KPI espressi in teste si riparametrano sulla platea corrente di `genere_platea.csv` — la platea femminile 15-24 cala dell'8% già entro il 2029 a migrazione ferma *(«Il bilancio dei giovani»)*. **Controfattuale dichiarato in anticipo** *(«Trend paralleli»)*: Palermo (pendenze non distinguibili sul pre-periodo 2018-2024, con un test a bassa potenza; senza modello binomiale la distanza da Palermo, -0.09 pp/anno, è più piccola di quella del 67% dei comuni simili), con la mediana delle gemelle come ancora dei target di lungo periodo.

### Cosa resta nell'analisi e non entra nella proposal
* **Decomposizione del gap per titolo di studio**: non calcolabile a livello comunale *(«Verifica di fattibilità»)*.
* **Tasso di disoccupazione femminile**: denominatore di 430-680 persone, oscillazioni in larga parte rumore *(«Dentro il "fuori da lavoro e studio"»)*.
* **Magnitudine del gap di istruzione**: ~1.5 dei 4.2 punti sono composizione per età; il segno regge, il numero va citato con la cautela *(«Verifica di composizione per età»)*.
* **Primato del vantaggio educativo**: vero sul 9-24 di fig05; sulle fasce allineate all'età da diploma non regge (15-24: +4.8 con la Sicilia a +4.7; 18-24: +5.7 contro 7.1 Sicilia e 6.2 Italia). Nei claim usare il distacco dal vicinato e la mancata conversione, non il primato assoluto *(«Su 1.000 ragazze»)*.
* **Indicatori 2011**: sfondo storico su 15+, mai in serie con il 2018-2024 *(«Contesto storico 2011»)*.
* **Correlazioni sui 390 comuni** (nuvola, mobilità): ecologiche e al 2011 — orientano le ipotesi, non dimostrano meccanismi *(«La nuvola dei 390»)*.
* **F5-F7**: denominatore = totale famiglie, quindi sensibili alla struttura per età; si citano percentili e gemelle, non i livelli *(«Formazione familiare precoce»)*.
* **Sex ratio 5-14**: anomalia replicata su due tavole ma senza meccanismo identificato; i check decisivi (nati per sesso del comune, stessa serie sulle gemelle) richiedono fonti nuove, non incluse in questa consegna *(«L'audit della platea»)*.
* **Stato civile (DCIS_POPRES1)**: al 1.1.2025 coincide alla singola unità con SETA_1 al 31.12.2024 (stesso conteggio censuario); osserva il matrimonio formale, non le convivenze né la maternità *(«Le casalinghe sono coniugate?»)*.
* **Transizioni annuali di ritenzione**: controllo di direzione della finestra 22-25, non misura autonoma *(«La finestra 22-25 regge?»)*.
* **Trend paralleli**: non rifiutati su sei punti ≠ dimostrati; il pre-periodo del confronto con le gemelle è ora misurato fino al 2024 *(«Il ponte fra i due censimenti»)*, ma il DiD resta incalcolabile finché un intervento non esiste *(«Trend paralleli»)*.

# **Tabelle esportate**
Le figure R di questo thread leggono da qui. Nessun ricalcolo in R.

In [66]:
pd.DataFrame(
    [(p.name, sum(1 for _ in p.open()) - 1) for p in sorted(PROCESSED.glob("genere_*.csv"))],
    columns=["file", "righe"])

,file,righe
0,genere_base_persone.csv,1
1,genere_casalinghe.csv,72
2,genere_casalinghe_bounds.csv,3
3,genere_coerenza_fonti.csv,16
4,genere_composizione_stato.csv,288
...,...,...
61,genere_stato_civile.csv,168
62,genere_stock_coorti.csv,4
63,genere_stranieri.csv,32
64,genere_tetto_platea.csv,4
